# GELAR RASA 2026 ? EDA, preprocessing, train-only pseudo-labeling, GPU training, submission

Notebook satu-run ini memakai **hanya gambar train dan test dari dataset GelarRasa**. Assignment, crop, fitur, validasi, model, dan CSV dibentuk di notebook. Tidak membaca bundle NPZ, folder processed lama, arsip label, atau SampleSubmission eksternal. Cache NPZ yang dipakai dalam proses dibuat di run ini dari foto train mentah dan divalidasi dengan fingerprint sumber.

Assignment train memakai 37 seed yang sudah ditinjau peserta (maksimal lima per kelas) dan pseudo-label konsensus cross-fit train-only. Test hanya untuk transformasi tetap dan inferensi.

Run All menghasilkan EDA, manifest, keputusan assignment per ID, processed train/test, grouped OOF, checkpoint, probabilitas, dan `DSC2026_Submission.csv` di `/kaggle/working/gelarrasa_run/`. Jumlah sampel supervised fit ditentukan oleh consensus train-only. Metrik OOF mengukur agreement terhadap target algoritmis, bukan accuracy resmi.

Notebook memilih GPU jika tersedia; training CNN memerlukan GPU Kaggle aktif dan koneksi Internet untuk bobot pretrained non-VLM yang dipatok.

In [ ]:
# OpenCV 5 removed its Caffe importer, which the pinned MINC branch needs.
# Install the compatible 4.x wheel before any cell imports cv2.
import importlib.metadata as _opencv_metadata, subprocess as _opencv_subprocess, sys as _opencv_sys
_opencv_packages=('opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless')
_installed_opencv=[]
for _package in _opencv_packages:
    try: _installed_opencv.append((_package,_opencv_metadata.version(_package)))
    except _opencv_metadata.PackageNotFoundError: pass
if any(_version.split('.',1)[0].isdigit() and int(_version.split('.',1)[0])>=5 for _,_version in _installed_opencv):
    _headless=any('headless' in _package for _package,_ in _installed_opencv)
    _contrib=any('contrib' in _package for _package,_ in _installed_opencv)
    _wheel=('opencv-contrib-python-headless' if _headless and _contrib else
            'opencv-contrib-python' if _contrib else
            'opencv-python-headless' if _headless else 'opencv-python')
    _opencv_subprocess.check_call([_opencv_sys.executable,'-m','pip','install','--quiet','--no-deps','--force-reinstall',f'{_wheel}==4.12.0.88'])

import os,sys,json,time,random,hashlib,platform,zipfile,shutil,gc,base64,zlib,tempfile
from pathlib import Path
from datetime import datetime,timezone
import numpy as np,pandas as pd
from PIL import Image,ImageOps,ImageFilter
import matplotlib.pyplot as plt
from IPython.display import display
import torch,torchvision
import importlib.metadata as package_metadata
SEED=2026
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
def seed_everything(value):
    random.seed(value); np.random.seed(value); torch.manual_seed(value)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(value)
seed_everything(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True,warn_only=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IS_KAGGLE=Path('/kaggle/working').is_dir()
INPUT_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working/gelarrasa_run') if IS_KAGGLE else Path(os.environ.get('GELARRASA_WORK_ROOT',str(Path(tempfile.gettempdir())/'gelarrasa_run')))
for d in ['eda','processed','labels','validation','checkpoints','predictions','logs','figures']:
    (WORK_ROOT/d).mkdir(parents=True,exist_ok=True)
GPU_INFO=[{'name':torch.cuda.get_device_name(i),'memory_GB':round(torch.cuda.get_device_properties(i).total_memory/1024**3,2)} for i in range(torch.cuda.device_count())]
ENVIRONMENT={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'torchvision':torchvision.__version__,'pillow':package_metadata.version('Pillow'),'numpy':np.__version__,'pandas':pd.__version__,'cuda':torch.version.cuda,'gpu':GPU_INFO,'seed':SEED}
print('Pipeline: train_only_cluster_gpu_v1 | mode:', 'GPU' if DEVICE.type=='cuda' else 'CPU preprocessing')
print(json.dumps(ENVIRONMENT,indent=2))
(WORK_ROOT/'logs/environment.json').write_text(json.dumps(ENVIRONMENT,indent=2),encoding='utf-8')

## 1. Deteksi dan integritas input

Jalur Kaggle pada log sebelumnya didukung langsung. Fallback mencari pasangan train/test dan sample CSV yang dapat berada pada tingkat folder berbeda. Keunikan ID diperiksa per split; nomor train dan test boleh sama. Nilai kategori contoh pada sample CSV diabaikan.

In [ ]:
# Temukan dataset dari folder gambar kompetisi saja; skema submission diturunkan dari nama file test.
IMAGE_EXTS={'.jpg','.jpeg','.png','.webp','.bmp'}
def is_dataset_root(p): return (p/'train').is_dir() and (p/'test').is_dir()

def find_dataset_root():
    candidates=[]
    known=[Path('/kaggle/input/datasets/arielzzzzzz/gelarrasa/Dataset DSC GelarRasa 2026'),
           Path(r'D:/lomba/NSC UB/GELAR-RASA/data/Dataset DSC GelarRasa 2026'),
           Path.cwd()/'GELAR-RASA'/'data'/'Dataset DSC GelarRasa 2026',
           Path.cwd()/'data'/'Dataset DSC GelarRasa 2026',Path.cwd()/'Dataset DSC GelarRasa 2026']
    for p in known:
        if is_dataset_root(p): return p.resolve()
        if p not in candidates:candidates.append(p)
    roots=[Path('/kaggle/input')] if IS_KAGGLE else [Path('/mnt/data'),Path.cwd(),Path.cwd().parent]
    for root in roots:
        if not root.exists():continue
        try:
            for base,dirs,_ in os.walk(root,followlinks=True):
                dirs.sort();p=Path(base)
                if p.name=='Dataset DSC GelarRasa 2026' and p not in candidates:
                    candidates.append(p)
        except (PermissionError,OSError):pass
    for p in candidates:
        if is_dataset_root(p):return p.resolve()
    visible=[]
    for root in roots:
        if root.exists():
            try:visible.append({'root':str(root),'children':[x.name for x in root.iterdir()][:30]})
            except OSError:pass
    raise FileNotFoundError('Dataset DSC GelarRasa 2026/train dan test tidak ditemukan: '+json.dumps(visible,ensure_ascii=False))

DATA_ROOT=find_dataset_root();TRAIN_ROOT=DATA_ROOT/'train';TEST_ROOT=DATA_ROOT/'test'
def image_files(folder):
    direct=[x for x in folder.iterdir() if x.is_file() and x.suffix.lower() in IMAGE_EXTS]
    files=direct or [x for x in folder.rglob('*') if x.is_file() and x.suffix.lower() in IMAGE_EXTS]
    return sorted(files,key=lambda x:(x.stem,x.as_posix()))
def norm_id(x):return str(x).strip().split('.')[0].zfill(4)
train_paths=image_files(TRAIN_ROOT);test_paths=image_files(TEST_ROOT)
train_ids=[norm_id(x.stem) for x in train_paths];test_ids=[norm_id(x.stem) for x in test_paths]
assert len(train_paths)==1920,f'Train harus 1920 gambar, ditemukan {len(train_paths)} di {TRAIN_ROOT}'
assert len(test_paths)==480,f'Test harus 480 gambar, ditemukan {len(test_paths)} di {TEST_ROOT}'
assert len(set(train_ids))==len(train_ids) and len(set(test_ids))==len(test_ids)
sample=pd.DataFrame({'image_id':[p.stem for p in test_paths],'waste_category':['']*len(test_paths)})
sample['image_id_norm']=sample.image_id.map(norm_id)
assert sample.image_id.is_unique and set(sample.image_id_norm)==set(test_ids)
CLASSES=['cardboard','electronic','foodwaste','glass','metal','paper','plastic','textile']
print('DATA_ROOT:',DATA_ROOT,'| TRAIN_ROOT:',TRAIN_ROOT,'| TEST_ROOT:',TEST_ROOT)
print('train:',len(train_paths),'| test:',len(test_paths),'| classes:',len(CLASSES),'| submission order derived from test filenames')
raw_train_paths=list(train_paths);raw_test_paths=list(test_paths);raw_train_ids=list(train_ids);raw_test_ids=list(test_ids)


## 2. EDA sumber dan kualitas citra

Inventaris semua sumber sebelum transformasi: hash file/piksel, dimensi, rasio, ukuran byte dan indikator edge variance. Identitas gambar berasal dari pasangan split dan ID, bukan angka ID lintas split. Kesamaan piksel train-test merupakan audit deskriptif.

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
def profile(p,split):
    r={'split':split,'image_id':norm_id(p.stem),'file_name':p.name,'file_sha256':sha256_file(p),'bytes':p.stat().st_size}
    with Image.open(p) as src:
        im=ImageOps.exif_transpose(src); rgb=im.convert('RGB'); gray=np.asarray(rgb.convert('L'),dtype=np.float32)
        r.update(width=rgb.width,height=rgb.height,mode=im.mode,aspect_ratio=rgb.width/rgb.height,
                 pixel_sha256=hashlib.sha256(rgb.tobytes()).hexdigest(),
                 blur_edge_variance=float(np.asarray(rgb.convert('L').filter(ImageFilter.FIND_EDGES),dtype=np.float32).var()))
    return r
manifest=pd.DataFrame([profile(p,'train') for p in train_paths]+[profile(p,'test') for p in test_paths])
manifest.to_csv(WORK_ROOT/'eda/image_manifest.csv',index=False)
summary=manifest.groupby('split').agg(n=('image_id','size'),median_width=('width','median'),median_height=('height','median'),median_bytes=('bytes','median'),median_aspect=('aspect_ratio','median'),median_edge_variance=('blur_edge_variance','median')).reset_index()
summary.to_csv(WORK_ROOT/'eda/image_summary.csv',index=False)
a=set(manifest.query("split=='train'").pixel_sha256); b=set(manifest.query("split=='test'").pixel_sha256)
(WORK_ROOT/'eda/exact_duplicate_summary.json').write_text(json.dumps({'train_test_exact_pixel_overlap':len(a&b),'use':'diagnostic only'},indent=2),encoding='utf-8')
print(summary.to_string(index=False))
fig,axs=plt.subplots(1,3,figsize=(15,4))
for split,g in manifest.groupby('split'):
    axs[0].hist(g.width,bins=30,alpha=.5,label=split); axs[1].hist(g.height,bins=30,alpha=.5,label=split); axs[2].hist(g.aspect_ratio,bins=30,alpha=.5,label=split)
for ax,title in zip(axs,['Width','Height','Aspect ratio']): ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/image_dimensions.png',dpi=150); plt.show()
def contact_sheet(paths,title,out,n=24):
    selected=paths[:n]; cols=6; rows=(len(selected)+cols-1)//cols
    fig,axs=plt.subplots(rows,cols,figsize=(15,rows*2.7)); axs=np.array(axs).reshape(-1)
    for ax,p in zip(axs,selected):
        with Image.open(p) as im: ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(norm_id(p.stem)); ax.axis('off')
    for ax in axs[len(selected):]: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(out,dpi=130); plt.show()
contact_sheet(train_paths,'Train samples',WORK_ROOT/'figures/train_contact_sheet.png')
contact_sheet(test_paths,'Test preview',WORK_ROOT/'figures/test_contact_sheet.png')

## 3. Bentuk sampel dan fitur train dari gambar mentah

Sel ini menerapkan kebijakan crop dan pengecualian yang tertulis di notebook. Citra RGB yang sudah diorientasi dipadatkan ke kanvas 224?224, lalu encoder EfficientNetV2-S menghitung fitur visual. Sebanyak 37 seed train yang sudah ditinjau peserta menjadi label manusia eksplisit; jumlahnya tidak melebihi lima per kelas. Foto test diproses untuk inferensi tanpa label.

In [ ]:
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import SVC
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from scipy.fft import dctn
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from collections import Counter

HUMAN_SEEDS = {
    'plastic': ['0056','0085','0237','1051','1115'],
    'metal': ['0072','1180','1605'],
    'electronic': ['0086','0956','1132','1434','1863'],
    'paper': ['0141','0256','0396','1114','1637'],
    'foodwaste': ['0200','0265','0636','1039'],
    'glass': ['0251','0889','1399','1685','1855'],
    'cardboard': ['0346','0536','0955','1215','1284'],
    'textile': ['0403','0504','1154','1332','1451'],
}
assert sum(map(len,HUMAN_SEEDS.values()))==37 and max(map(len,HUMAN_SEEDS.values()))<=5
SEED_LABELS={sample_id:category for category,ids in HUMAN_SEEDS.items() for sample_id in ids}
assert len(SEED_LABELS)==37
POLICY={
    'drop': ['0134','0298','0545','1043'],
    'reserve': [],
    'hold': ['0058','0079','0147','0160','0183','0189','0225','0310','0321','0400','0498','0559','0964','0978','1018','1142','1157','1281','1436','1515','1531','1657','1662','1848','1885','1904','1913'],
    'residual_force_include': ['0018','0604','0376'],
    'residual_hold': ['0010','0079','0120','0131','0265','0321','0437','0597','0672','0774','0857','0930','0963','1027','1028','1142','1176','1246','1351','1417','1446','1465','1522','1541','1557','1637','1657','1662','1694','1885','1890','1892'],
    'crops': [
        ('0415','calculator',(0,115,235,522)),
        ('0437','laptop',(0,44,373,429)),
        ('0437','apple',(370,292,466,427)),
        ('0951','printed_page',(143,145,445,398)),
        ('0951','calculator',(456,142,640,503)),
        ('1007','red_bag',(267,63,472,232)),
        ('1351','lowerleft_box',(0,349,118,484)),
    ],
    'output_size':224,'max_content_side':202,'padding_rgb':[127,127,127],
}
PROCESS_CROPS=True  # Crop dibentuk ulang di notebook dari piksel dataset train mentah.
ACTIVE_CROP_SPECS=POLICY['crops'] if PROCESS_CROPS else []
RESIDUAL_FORCE_INCLUDE=set(POLICY['residual_force_include'])
RESIDUAL_HOLD=set(POLICY['residual_hold'])
TRAIN_HOLD=set(POLICY['hold'])|RESIDUAL_HOLD
assert len(POLICY['hold'])==27 and len(RESIDUAL_FORCE_INCLUDE)==3 and len(RESIDUAL_HOLD)==32
assert not RESIDUAL_FORCE_INCLUDE & TRAIN_HOLD
assert not (set(POLICY['drop'])|set(POLICY['reserve'])) & (TRAIN_HOLD|RESIDUAL_FORCE_INCLUDE)
assert not (set(POLICY['drop'])|set(POLICY['reserve'])) & set(SEED_LABELS)
assert len(POLICY['crops'])==7 and PROCESS_CROPS
def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def rgb_source(path):
    with Image.open(path) as source:
        im=ImageOps.exif_transpose(source)
        if 'A' in im.getbands() or 'transparency' in im.info:
            rgba=im.convert('RGBA')
            im=Image.alpha_composite(Image.new('RGBA',rgba.size,(255,255,255,255)),rgba)
        return im.convert('RGB').copy()
def processed_canvas(im):
    side=POLICY['max_content_side']; size=POLICY['output_size']
    content=ImageOps.contain(im,(side,side),method=Image.Resampling.LANCZOS)
    out=Image.new('RGB',(size,size),tuple(POLICY['padding_rgb']))
    out.paste(content,((size-content.width)//2,(size-content.height)//2))
    return out
raw_sources=[]
for split,paths in [('train',raw_train_paths),('test',raw_test_paths)]:
    for path in paths: raw_sources.append({'split':split,'file_name':path.name,'sha256':file_sha256(path)})
assert len(raw_sources)==2400 and len({(r['split'],r['file_name']) for r in raw_sources})==2400
PROCESSED_ROOT=WORK_ROOT/'processed'
STAGING=PROCESSED_ROOT/'staging'; STAGING.mkdir(parents=True,exist_ok=True)
crop_by_parent={}
for parent,region,bbox in ACTIVE_CROP_SPECS:crop_by_parent.setdefault(parent,[]).append((region,bbox))
sample_rows=[]
for split,paths in [('train',raw_train_paths),('test',raw_test_paths)]:
    for path in paths:
        parent=path.stem
        if split=='train' and parent in set(POLICY['drop']+POLICY['reserve']):continue
        im=rgb_source(path)
        regions=[(None,(0,0,im.width,im.height))]
        if split=='train' and parent in crop_by_parent:
            regions.extend(crop_by_parent[parent])
        for region,bbox in regions:
            x0,y0,x1,y1=bbox
            assert 0<=x0<x1<=im.width and 0<=y0<y1<=im.height,(parent,region,bbox,im.size)
            sample_id=parent+'__'+region if region else parent
            stage=STAGING/split/f'{sample_id}.png';stage.parent.mkdir(parents=True,exist_ok=True)
            processed_canvas(im.crop(bbox)).save(stage,format='PNG')
            sample_rows.append({'split':split,'sample_id':sample_id,'parent_id':parent,'source_file':path.name,
                'source_path':str(path),'bbox':list(bbox),'is_crop':region is not None,'region_name':region or '',
                'training_eligible':split=='train' and (parent not in TRAIN_HOLD or parent in RESIDUAL_FORCE_INCLUDE),'staging_path':str(stage)})
candidates=pd.DataFrame(sample_rows)
assert candidates.groupby('split').sample_id.apply(lambda ids:ids.is_unique).all()
_train_source_ids=set(raw_train_ids)
_excluded_present=_train_source_ids & set(POLICY['drop']+POLICY['reserve'])
_crop_parents={parent for parent,_,_ in ACTIVE_CROP_SPECS}
assert _crop_parents<=_train_source_ids and not (_crop_parents&_excluded_present)
_expected_train_rows=len(raw_train_paths)-len(_excluded_present)+len(ACTIVE_CROP_SPECS)
_train_candidate_count=len(candidates.query("split == 'train'"))
_test_candidate_count=len(candidates.query("split == 'test'"))
assert _train_candidate_count==_expected_train_rows, f"train candidates {_train_candidate_count} != expected {_expected_train_rows}"
assert _test_candidate_count==len(raw_test_paths)
assert int(candidates.is_crop.sum())==len(ACTIVE_CROP_SPECS)

class StageDataset(Dataset):
    def __init__(self,paths):
        self.paths=list(paths)
        self.tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as im:return self.tf(im.convert('RGB'))
try: feature_encoder=models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1).eval().to(DEVICE)
except Exception as exc:raise RuntimeError('Bobot EfficientNetV2-S tidak tersedia. Aktifkan Internet Kaggle.') from exc
feature_encoder.classifier=torch.nn.Identity()
def extract_stage_features(paths):
    loader=DataLoader(StageDataset(paths),batch_size=24,shuffle=False,num_workers=2 if os.name!='nt' else 0,pin_memory=DEVICE.type=='cuda')
    chunks=[]
    with torch.inference_mode(),torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
        for batch in loader:chunks.append(feature_encoder(batch.to(DEVICE,non_blocking=True)).float().cpu().numpy())
    return normalize(np.concatenate(chunks).astype(np.float32))
stage_features=extract_stage_features(candidates.staging_path)
assert stage_features.shape[0]==len(candidates) and np.isfinite(stage_features).all()
def color_descriptor(path):
    import cv2
    rgb=np.asarray(Image.open(path).convert('RGB'),dtype=np.uint8)
    hsv=cv2.cvtColor(rgb,cv2.COLOR_RGB2HSV)
    hist=[]
    for channel,bins,high in [(0,18,180),(1,12,256),(2,12,256)]:
        count=np.histogram(hsv[:,:,channel],bins=bins,range=(0,high))[0].astype(np.float32)
        hist.extend(count/max(1,count.sum()))
    pix=rgb.astype(np.float32)/255
    return np.asarray(hist+list(pix.mean((0,1)))+list(pix.std((0,1))),dtype=np.float32)
stage_color=np.stack([color_descriptor(path) for path in candidates.staging_path])
del feature_encoder;gc.collect();torch.cuda.empty_cache()
print('Train-only input rows:',int(candidates.split.eq('train').sum()),'| test inference rows:',int(candidates.split.eq('test').sum()))


## 4. Bangun ulang fitur untuk assignment aktif dari foto train mentah

Sel ini mengganti cache embedding DINOv2/DINOv3 yang dipakai oleh refinement aktif. Ia memproses seluruh foto train sumber dan view flip dari 256 foto train yang dipilih deterministik dari piksel train unik. Test tidak dibaca di sel ini. Hasilnya adalah fitur terhitung notebook, bukan label atau assignment yang disalin.


In [ ]:
# Rebuild the two image-only feature banks used by the active train-only refinement chain.
# Inputs are raw organizer TRAIN images only; no assignment NPZ, archive labels, or test images.
import gc, timm, cv2
from huggingface_hub import hf_hub_download
from timm.data import resolve_model_data_config
from torchvision.transforms.functional import to_tensor, normalize

ASSIGN_FEATURE_DIR=WORK_ROOT/'features'/'assignment_rebuild'
ASSIGN_FEATURE_DIR.mkdir(parents=True,exist_ok=True)
ASSIGN_SEED=20261105
DINO2_REPO='timm/vit_small_patch14_dinov2.lvd142m'
DINO2_REV='4610ca143709d58a633b6397a74412c2c3842454'
DINO3S_REPO='timm/vit_small_patch16_dinov3.lvd1689m'
DINO3S_REV='3bf4720a82ec2066db88137180ff1f83a675cef0'

def _sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()

def _raw_rgb(path):
    with Image.open(path) as src:
        src.load();im=ImageOps.exif_transpose(src)
        if 'A' in im.getbands() or 'transparency' in im.info:
            rgba=im.convert('RGBA')
            im=Image.alpha_composite(Image.new('RGBA',rgba.size,(255,255,255,255)),rgba)
        return im.convert('RGB').copy()

def _letterbox(im,size):
    scale=size/max(im.size);dims=(max(1,min(size,round(im.width*scale))),max(1,min(size,round(im.height*scale))))
    out=Image.new('RGB',(size,size),(124,116,104));out.paste(im.resize(dims,Image.Resampling.LANCZOS),((size-dims[0])//2,(size-dims[1])//2))
    return out

def _dino3_small_input(path,mirror=False):
    im=_raw_rgb(path);size=256;scale=size/max(im.size)
    dims=(max(1,min(size,round(im.width*scale))),max(1,min(size,round(im.height*scale))))
    rgb=np.asarray(im.resize(dims,Image.Resampling.BICUBIC)).copy()
    canvas=np.full((size,size,3),(124,116,104),dtype=np.uint8);valid=np.zeros((size,size),dtype=np.float32)
    x,y=(size-dims[0])//2,(size-dims[1])//2;canvas[y:y+dims[1],x:x+dims[0]]=rgb;valid[y:y+dims[1],x:x+dims[0]]=1
    border=np.concatenate([rgb[0],rgb[-1],rgb[:,0],rgb[:,-1]],axis=0).astype(np.float32)
    center=np.median(border,axis=0);disp=np.linalg.norm(border-center,axis=1);fg=valid.copy()
    if float(np.quantile(disp,.75))<30:
        mask=(np.linalg.norm(rgb.astype(np.float32)-center,axis=2)>max(25,float(np.quantile(disp,.95))+10)).astype(np.uint8)
        mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((3,3),dtype=np.uint8));fraction=float(mask.mean())
        if .03<=fraction<=.90:
            fg=np.zeros_like(valid);fg[y:y+dims[1],x:x+dims[0]]=mask;fg+=.05*valid
    if mirror:
        canvas=np.ascontiguousarray(canvas[:,::-1]);valid=valid[:,::-1].copy();fg=fg[:,::-1].copy()
    valid=cv2.resize(valid,(16,16),interpolation=cv2.INTER_AREA).reshape(-1)
    fg=cv2.resize(fg,(16,16),interpolation=cv2.INTER_AREA).reshape(-1)
    return Image.fromarray(canvas),valid,fg

train_paths=sorted(raw_train_paths,key=lambda p:int(p.stem))
train_keys=np.array(['train:'+str(int(p.stem)).zfill(4) for p in train_paths],dtype=str)
train_source_sha256=np.array([_sha256_file(p) for p in train_paths],dtype=str)
train_input_fingerprint=hashlib.sha256(''.join(train_keys+'|'+train_source_sha256).encode()).hexdigest()
assert len(train_paths)==1920 and len(set(train_keys))==1920
profile_train=manifest.loc[manifest.split.eq('train')].drop_duplicates('pixel_sha256')
flip_probe=profile_train.sample(n=256,random_state=ASSIGN_SEED).sort_values('image_id')
flip_keys=np.array(['train:'+str(int(v)).zfill(4) for v in flip_probe.image_id],dtype=str)
flip_path_by_key={key:path for key,path in zip(train_keys,train_paths)}
assert len(flip_keys)==256 and set(flip_keys)<=set(train_keys)

def _extract_global_dino2(paths,keys,mirrors=None,batch_size=32):
    weight=hf_hub_download(DINO2_REPO,'model.safetensors',revision=DINO2_REV)
    model=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0,dynamic_img_size=True,
                            pretrained_cfg_overlay={'file':str(weight)}).eval().to(DEVICE)
    mean=(.485,.456,.406);std=(.229,.224,.225);features=[];patch_features=[];mirrors=mirrors or [False]*len(paths)
    with torch.inference_mode():
        for start in range(0,len(paths),batch_size):
            images=[];masks=[]
            for path,mirror in zip(paths[start:start+batch_size],mirrors[start:start+batch_size]):
                original=_raw_rgb(path);im=_letterbox(original,224)
                if mirror:im=ImageOps.mirror(im)
                images.append(normalize(to_tensor(im),mean,std))
                scale=224/max(original.size);nw=max(1,min(224,round(original.width*scale)));nh=max(1,min(224,round(original.height*scale)))
                left,top=(224-nw)//2,(224-nh)//2;centers=np.arange(16)*14+7
                mask=((centers[:,None]>=top)&(centers[:,None]<top+nh)&(centers[None,:]>=left)&(centers[None,:]<left+nw))
                if mirror:mask=np.fliplr(mask)
                masks.append(mask.reshape(-1).astype(np.float32))
            x=torch.stack(images).to(DEVICE,non_blocking=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=DEVICE.type=='cuda'):
                tokens=model.forward_features(x)
            patches=tokens[:,model.num_prefix_tokens:].float();assert patches.shape[1:]==(256,384)
            valid=torch.from_numpy(np.stack(masks)).to(DEVICE);valid=valid/valid.sum(1,keepdim=True).clamp_min(1e-8)
            mean_patch=(patches*valid[:,:,None]).sum(1)
            std_patch=((patches-mean_patch[:,None,:]).square()*valid[:,:,None]).sum(1).clamp_min(0).sqrt()
            cls=model.forward_head(tokens).float()
            global_block=torch.nn.functional.normalize(cls,dim=1).cpu().numpy().astype(np.float32)
            patch_block=torch.cat([torch.nn.functional.normalize(v,dim=1) for v in [cls,mean_patch,std_patch]],dim=1).cpu().numpy().astype(np.float32)
            features.append(global_block);patch_features.append(patch_block)
            if start==0 or start+batch_size>=len(paths):print('DINOv2 train global+patch features',min(start+batch_size,len(paths)),'/',len(paths),flush=True)
    del model;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return np.concatenate(features),np.concatenate(patch_features),_sha256_file(weight)

def _extract_dense_dino3s(items,batch_size=8):
    weight=hf_hub_download(DINO3S_REPO,'model.safetensors',revision=DINO3S_REV)
    model=timm.create_model('vit_small_patch16_dinov3.lvd1689m',pretrained=True,num_classes=0,
                            pretrained_cfg_overlay={'file':str(weight)}).eval().to(DEVICE)
    cfg=resolve_model_data_config(model);prefix=model.num_prefix_tokens;mean=cfg['mean'];std=cfg['std'];features=[]
    with torch.inference_mode():
        for start in range(0,len(items),batch_size):
            imgs=[];valids=[];foregrounds=[]
            for key,mirror in items[start:start+batch_size]:
                canvas,valid,fg=_dino3_small_input(flip_path_by_key[key],mirror)
                imgs.append(normalize(to_tensor(canvas),mean,std));valids.append(valid);foregrounds.append(fg)
            x=torch.stack(imgs).to(DEVICE,non_blocking=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=DEVICE.type=='cuda'):
                tokens=model.forward_features(x)
            patches=tokens[:,prefix:].float();assert patches.shape[1]==256
            valid=torch.from_numpy(np.stack(valids)).to(DEVICE);fg=torch.from_numpy(np.stack(foregrounds)).to(DEVICE)
            valid=valid/valid.sum(1,keepdim=True).clamp_min(1e-8);fg=fg/fg.sum(1,keepdim=True).clamp_min(1e-8)
            mean_valid=(patches*valid[:,:,None]).sum(1);mean_fg=(patches*fg[:,:,None]).sum(1)
            std_fg=((patches-mean_fg[:,None,:]).square()*fg[:,:,None]).sum(1).clamp_min(0).sqrt()
            block=torch.cat([torch.nn.functional.normalize(v,dim=1) for v in [tokens[:,0].float(),mean_valid,mean_fg,std_fg]],1)
            features.append(block.cpu().numpy().astype(np.float32))
            if start==0 or start+batch_size>=len(items):print('DINOv3 Small train views',min(start+batch_size,len(items)),'/',len(items),flush=True)
    del model;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return np.concatenate(features),_sha256_file(weight),{'mean':mean,'std':std,'prefix_tokens':prefix}

# Generated caches are keyed by raw IDs and pinned model revisions; only train images are processed.
d2_path=ASSIGN_FEATURE_DIR/'dinov2_small_train_features.npz'
if d2_path.exists():
    z=np.load(d2_path,allow_pickle=False);assert np.array_equal(z['keys'],train_keys) and np.array_equal(z['source_sha256'],train_source_sha256) and str(z['revision'].item())==DINO2_REV
    dino2_train_features=z['features'];dino2_patch_features=z['patch_features']
else:
    dino2_train_features,dino2_patch_features,d2_sha=_extract_global_dino2(train_paths,train_keys)
    np.savez_compressed(d2_path,keys=train_keys,source_sha256=train_source_sha256,features=dino2_train_features,patch_features=dino2_patch_features,revision=np.array(DINO2_REV),weight_sha256=np.array(d2_sha),input_fingerprint=np.array(train_input_fingerprint))
flip_paths=[flip_path_by_key[k] for k in flip_keys]
d2_flip_path=ASSIGN_FEATURE_DIR/'dinov2_small_train_hflip_probe.npz'
if d2_flip_path.exists():
    z=np.load(d2_flip_path,allow_pickle=False);assert np.array_equal(z['keys'],flip_keys) and str(z['revision'].item())==DINO2_REV and str(z['input_fingerprint'].item())==train_input_fingerprint
    dino2_flip_features=z['features'];dino2_patch_flip_features=z['patch_features']
else:
    dino2_flip_features,dino2_patch_flip_features,d2_flip_sha=_extract_global_dino2(flip_paths,flip_keys,[True]*len(flip_paths),batch_size=16)
    np.savez_compressed(d2_flip_path,keys=flip_keys,features=dino2_flip_features,patch_features=dino2_patch_flip_features,revision=np.array(DINO2_REV),weight_sha256=np.array(d2_flip_sha),input_fingerprint=np.array(train_input_fingerprint))

d3_items=[(k,False) for k in train_keys]+[(k,True) for k in flip_keys]
d3_path=ASSIGN_FEATURE_DIR/'dinov3_small_train_descriptors.npz'
if d3_path.exists():
    z=np.load(d3_path,allow_pickle=False);assert np.array_equal(z['keys'],train_keys) and np.array_equal(z['source_sha256'],train_source_sha256) and np.array_equal(z['flip_keys'],flip_keys) and str(z['revision'].item())==DINO3S_REV and str(z['input_fingerprint'].item())==train_input_fingerprint
    dino3_train_features=z['features'];dino3_flip_features=z['flip_features']
else:
    all_d3,d3_sha,d3_cfg=_extract_dense_dino3s(d3_items)
    dino3_train_features=all_d3[:len(train_keys)];dino3_flip_features=all_d3[len(train_keys):]
    np.savez_compressed(d3_path,keys=train_keys,source_sha256=train_source_sha256,features=dino3_train_features,flip_keys=flip_keys,flip_features=dino3_flip_features,
                        revision=np.array(DINO3S_REV),weight_sha256=np.array(d3_sha),input_fingerprint=np.array(train_input_fingerprint),mean=np.asarray(d3_cfg['mean']),std=np.asarray(d3_cfg['std']))
assert dino2_train_features.shape==(1920,384) and dino2_flip_features.shape==(256,384)
assert dino2_patch_features.shape==(1920,1152) and dino2_patch_flip_features.shape==(256,1152)
assert dino3_train_features.shape==(1920,1536) and dino3_flip_features.shape==(256,1536)
assert all(np.isfinite(x).all() for x in [dino2_train_features,dino2_flip_features,dino2_patch_features,dino2_patch_flip_features,dino3_train_features,dino3_flip_features])
ASSIGNMENT_FEATURE_PROVENANCE={'source':'raw organizer train images','test_rows_processed':0,'train_rows':len(train_keys),'hflip_train_probe_rows':len(flip_keys),
 'dino2_revision':DINO2_REV,'dino3_small_revision':DINO3S_REV,'feature_files':[str(d2_path),str(d2_flip_path),str(d3_path)],
 'raw_train_source_fingerprint':train_input_fingerprint,'source_hashes_in_feature_files':True,
 'assignment_created':False,'classifier_training_started':False}
(ASSIGN_FEATURE_DIR/'feature_provenance.json').write_text(json.dumps(ASSIGNMENT_FEATURE_PROVENANCE,indent=2),encoding='utf-8')
print(json.dumps(ASSIGNMENT_FEATURE_PROVENANCE,indent=2))

## 4.1 Baseline cluster anonim train-only

Head konsistensi tetangga dibangun ulang dari fitur DINOv2 yang dihitung di notebook. Grup duplikat dan pHash dibentuk dari foto train saja, lalu dipakai untuk memisahkan fit/holdout. Klaster tetap anonim; tahap ini belum menjadi delapan label material dan tidak membuka preprocessing/crop/classifier.


In [ ]:
# Rebuild the unlabeled coarse routing head from raw TRAIN images only.
# The resulting cluster IDs are anonymous and are not material labels.
import itertools
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize as sklearn_normalize
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits

COARSE_DIR=WORK_ROOT/'labels'/'coarse_train_only'
COARSE_DIR.mkdir(parents=True,exist_ok=True)
raw_by_id={norm_id(p.stem):p for p in raw_train_paths}
ordered_train_ids=np.array(sorted(raw_by_id,key=lambda x:int(x)),dtype=str)
train_profile=manifest[(manifest.split=='train')&manifest.image_id.isin(ordered_train_ids)].drop_duplicates('image_id').set_index('image_id').loc[ordered_train_ids].reset_index()
assert len(train_profile)==1920 and train_profile.image_id.is_unique
feature_pos={sid.removeprefix('train:'):i for i,sid in enumerate(train_keys)}
assert set(ordered_train_ids)<=set(feature_pos)
X_train_coarse=np.asarray([dino2_train_features[feature_pos[sid]] for sid in ordered_train_ids],dtype=np.float32)
assert X_train_coarse.shape==(1920,384) and np.isfinite(X_train_coarse).all()
X_train_coarse=sklearn_normalize(X_train_coarse).astype(np.float32)
train_pixel_hash=train_profile.pixel_sha256.astype(str).to_numpy()

# Reproduce 64-bit pHash and connected candidate groups without reading test.
def _phash64(path,view='original'):
    im=rgb_source(path)
    if view=='hflip':im=ImageOps.mirror(im)
    elif view=='vflip':im=ImageOps.flip(im)
    gray=np.asarray(im.convert('L').resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
    low=dctn(gray,type=2,norm='ortho')[:8,:8].reshape(-1)
    threshold=np.median(low[1:])
    return np.packbits(low>threshold,bitorder='big')
phash=np.stack([_phash64(raw_by_id[sid]) for sid in ordered_train_ids])
hash_h=np.stack([_phash64(raw_by_id[sid],'hflip') for sid in ordered_train_ids])
hash_v=np.stack([_phash64(raw_by_id[sid],'vflip') for sid in ordered_train_ids])
bitcount=np.asarray([int(i).bit_count() for i in range(256)],dtype=np.uint8)
def _hamming_to(left,right):
    out=np.empty((len(left),len(right)),dtype=np.uint8)
    for start in range(0,len(left),128):
        out[start:start+128]=bitcount[np.bitwise_xor(left[start:start+128,None,:],right[None,:,:])].sum(2)
    return out
D=np.minimum(_hamming_to(phash,phash),np.minimum(_hamming_to(phash,hash_h),_hamming_to(phash,hash_v)))
D=np.minimum(D,D.T);np.fill_diagonal(D,0)
parent=list(range(len(ordered_train_ids)))
def _find(i):
    while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
    return i
def _union(i,j):
    a,b=_find(i),_find(j)
    if a!=b:parent[max(a,b)]=min(a,b)
# Exact-source and exact-pixel duplicates are always joined.
for col in ['file_sha256','pixel_sha256']:
    buckets={}
    for i,value in enumerate(train_profile[col].astype(str)):buckets.setdefault(value,[]).append(i)
    for indices in buckets.values():
        for j in indices[1:]:_union(indices[0],j)
for i,j in zip(*np.where(np.triu(D<=4,k=1))):_union(int(i),int(j))
coarse_groups=np.asarray([_find(i) for i in range(len(ordered_train_ids))],dtype=np.int64)
unique_mask=~pd.Series(train_pixel_hash).duplicated(keep='first').to_numpy()
unique_idx=np.flatnonzero(unique_mask);unique_groups=coarse_groups[unique_idx]
assert len(unique_idx)>=1900 and len(np.unique(unique_groups))>=8
splitter=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=20261105)
fit_local,held_local=next(splitter.split(X_train_coarse[unique_idx],groups=unique_groups))
fit_idx=unique_idx[fit_local];held_idx=unique_idx[held_local]
assert not set(coarse_groups[fit_idx])&set(coarse_groups[held_idx])

# SCAN-inspired consistency-minus-marginal-entropy head; no semantic labels.
def _neighbor_graph(query,pool,query_hash,pool_hash,k=10):
    similarity=query@pool.T
    similarity[query_hash[:,None]==pool_hash[None,:]]=-np.inf
    if np.any(np.isfinite(similarity).sum(1)<k):raise RuntimeError('Too few nonduplicate train neighbors.')
    return torch.as_tensor(np.argsort(-similarity,axis=1,kind='stable')[:,:k].copy(),dtype=torch.long,device=DEVICE)
def _coarse_components(prob,neighbor_index,train_prob=None):
    neighbor_prob=prob[neighbor_index] if train_prob is None else train_prob[neighbor_index]
    dot=(prob[:,None,:]*neighbor_prob).sum(-1).clamp_min(1e-8)
    consistency=-dot.log().mean();avg=prob.mean(0).clamp_min(1e-8)
    entropy=-(avg*avg.log()).sum();return consistency,entropy
def _init_coarse_head(x_np,seed):
    km=KMeans(n_clusters=8,n_init=10,max_iter=300,random_state=seed).fit(x_np)
    head=torch.nn.Linear(x_np.shape[1],8).to(DEVICE);centers=torch.as_tensor(km.cluster_centers_,device=DEVICE,dtype=torch.float32)
    with torch.no_grad():head.weight.copy_(20*centers);head.bias.copy_(-10*centers.square().sum(1))
    return head
def _coarse_metrics(head,x,train_x,neighbor_index):
    with torch.no_grad():
        p=head(x).softmax(-1);q=head(train_x).softmax(-1);cons,entropy=_coarse_components(p,neighbor_index,q)
        labels=p.argmax(-1);sizes=torch.bincount(labels,minlength=8)
        return {'consistency':float(cons),'marginal_entropy':float(entropy),'selection_score':float(cons-2*entropy),
                'occupied':int((sizes>0).sum()),'min_cluster':int(sizes.min()),'max_cluster':int(sizes.max())}
def _fit_coarse(x_np,hashes,seed,entropy_weight,epochs,val_np=None,val_hash=None):
    torch.manual_seed(seed);x=torch.as_tensor(x_np,device=DEVICE);nn=_neighbor_graph(x_np,x_np,hashes,hashes)
    head=_init_coarse_head(x_np,seed);optimizer=torch.optim.Adam(head.parameters(),lr=.03);best=None
    if val_np is not None:
        vx=torch.as_tensor(val_np,device=DEVICE);vn=_neighbor_graph(val_np,x_np,val_hash,hashes)
    for epoch in range(epochs+1):
        if epoch:
            head.train();prob=head(x).softmax(-1);cons,entropy=_coarse_components(prob,nn)
            loss=cons-entropy_weight*entropy;optimizer.zero_grad();loss.backward();optimizer.step()
        if epoch%10==0 or epoch==epochs:
            train_metrics=_coarse_metrics(head,x,x,nn)
            if val_np is not None:
                val_metrics=_coarse_metrics(head,vx,x,vn)
                if train_metrics['occupied']==8 and val_metrics['occupied']==8 and (best is None or val_metrics['selection_score']<best['score']):
                    best={'score':val_metrics['selection_score'],'epoch':epoch,'state':{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},'metrics':val_metrics}
    return head,best

Xfit=X_train_coarse[fit_idx];Xheld=X_train_coarse[held_idx]
hfit=train_pixel_hash[fit_idx];hheld=train_pixel_hash[held_idx]
coarse_trials=[];best_trial=None
with threadpool_limits(limits=4):
    for entropy_weight,trial_seed in itertools.product([.5,2.0],[11,29,47]):
        _,candidate=_fit_coarse(Xfit,hfit,trial_seed,entropy_weight,120,Xheld,hheld)
        row={'seed':trial_seed,'entropy_weight':entropy_weight,'eligible':candidate is not None}
        if candidate is not None:
            row.update(epoch=candidate['epoch'],**candidate['metrics'])
            if best_trial is None or candidate['score']<best_trial['score']:
                best_trial={'seed':trial_seed,'entropy_weight':entropy_weight,'epoch':candidate['epoch'],'score':candidate['score'],'state':candidate['state'],'metrics':candidate['metrics']}
        coarse_trials.append(row);print(json.dumps(row),flush=True)
if best_trial is None:raise RuntimeError('No noncollapsed train-only coarse head passed its holdout checks.')
coarse_head,_=_fit_coarse(X_train_coarse[unique_idx],train_pixel_hash[unique_idx],best_trial['seed'],best_trial['entropy_weight'],best_trial['epoch'])
coarse_head.eval()
with torch.inference_mode():coarse_prob=coarse_head(torch.as_tensor(X_train_coarse,device=DEVICE)).softmax(-1).float().cpu().numpy()
coarse_pred=coarse_prob.argmax(1)
coarse_table=pd.DataFrame({'image_id':ordered_train_ids,'source_sha256':train_profile.file_sha256.astype(str),
    'pixel_sha256':train_pixel_hash,'source_group':coarse_groups,'coarse_cluster':coarse_pred,
    'cluster_confidence':coarse_prob.max(1),'fit_partition':np.where(np.isin(np.arange(len(ordered_train_ids)),fit_idx),'fit_train',np.where(np.isin(np.arange(len(ordered_train_ids)),held_idx),'heldout_train','duplicate_train'))})
coarse_table.to_csv(COARSE_DIR/'stage4_train_coarse_clusters.csv',index=False)
np.savez_compressed(COARSE_DIR/'stage4_train_coarse_probabilities.npz',image_ids=ordered_train_ids,probabilities=coarse_prob,source_sha256=coarse_table.source_sha256.to_numpy(str))
coarse_fit_ids=ordered_train_ids[fit_idx].tolist()
coarse_heldout_ids=ordered_train_ids[held_idx].tolist()
coarse_fit_group_ids=coarse_groups[fit_idx].astype(str).tolist()
coarse_heldout_group_ids=coarse_groups[held_idx].astype(str).tolist()
assert set(coarse_fit_group_ids).isdisjoint(coarse_heldout_group_ids)
torch.save({'state_dict':{k:v.detach().cpu() for k,v in coarse_head.state_dict().items()},'input_dimension':384,
    'selected_seed':best_trial['seed'],'entropy_weight':best_trial['entropy_weight'],'epochs':best_trial['epoch'],
    'fit_ids':coarse_fit_ids,'heldout_ids':coarse_heldout_ids,
    'fit_group_ids':coarse_fit_group_ids,'heldout_group_ids':coarse_heldout_group_ids,
    'training_unique_ids':ordered_train_ids[unique_idx].tolist(),'test_used_in_fit':False},COARSE_DIR/'stage4_coarse_head.pt')
coarse_report={'method':'train_only_dinov2_neighbor_consistency_anonymous_8cluster_v1','raw_train_rows':1920,
    'unique_train_pixels_fit':len(unique_idx),'fit_rows':len(fit_idx),'heldout_rows':len(held_idx),'source_groups':int(len(np.unique(coarse_groups))),
    'test_rows_in_fit':0,'semantic_labels_used':False,'selected':{k:v for k,v in best_trial.items() if k!='state'},
    'trial_metrics':coarse_trials,'cluster_counts':{int(k):int(v) for k,v in zip(*np.unique(coarse_pred,return_counts=True))},
    'assignment_is_semantic':False,'source_sha256':train_source_sha256.tolist()}
(COARSE_DIR/'stage4_train_coarse_report.json').write_text(json.dumps(coarse_report,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame(coarse_trials));display(coarse_table.coarse_cluster.value_counts().sort_index().rename('train_images').to_frame())
print('Anonymous train-only coarse routing head saved:',COARSE_DIR)


## 4.2 Fitur CNN semantik train-only untuk routing

Ekstrak embedding dan prior kelas dari tiga backbone non-VLM yang dipakai oleh cabang Stage12/16: ResNet50 daur ulang, ConvNeXt Tiny ImageNet-22k dengan ontology WordNet, dan EfficientNetV2-S daur ulang. Bobot/revisi di-pin dan hash diverifikasi. Semua gambar, termasuk view flip, berasal dari train; notebook tidak membaca test pada tahap assignment. Ini hanya fitur/prior; Stage12/16 tetap harus dilatih ulang dari fitur tersebut.


In [ ]:
# Recompute pinned non-VLM semantic feature banks from raw TRAIN only.
# No archive labels, assignment NPZ, test images, or remote inference are used.
import gc, json, hashlib, time
from huggingface_hub import hf_hub_download
from torchvision import transforms
from torchvision.models import resnet50, efficientnet_v2_s
from safetensors.torch import load_file as load_safetensors
import timm
from timm.data import ImageNetInfo, create_transform, resolve_model_data_config

SEMANTIC_FEATURE_DIR=WORK_ROOT/'features'/'semantic_rebuild'
SEMANTIC_FEATURE_DIR.mkdir(parents=True,exist_ok=True)
FEATURE_MODEL_SPECS={
 'resnet_waste':{'repo':'kendrickfff/my_resnet50_garbage_classificationv1.2','revision':'c689678ec70cf53baa0a081644d29901e59f4a4a','file':'resnet50_garbage_classificationv1.2.pth','sha256':'0bcac0ef095d6bc6872e2d5b0c18c31953264f2781a7e48ac89e1adc83739cb4'},
 'convnext_ontology':{'repo':'timm/convnext_tiny.fb_in22k','revision':'342f2588bb2ea79d793d0712c9be16ec6ae0149a','file':'model.safetensors','sha256':'2b40f8d53b566a8c08c0c75f93ff5ac6fa33bd213afb27ee40766af7ffe5943b'},
 'efficientnet_waste':{'repo':'attilaultzindur/garbage_classifier_effnetv2s_ft','revision':'35f28e0f432e4d874d88bb214da6cb96553f10fd','file':'model.safetensors','sha256':'f0d1ba4d1966d0cf2eb8143e5c3b64f50d3e2c5484c67036dfcf52936cbb319c'}
}
SEMANTIC_SOURCE_CLASSES={
 'resnet_waste':['battery','biological','brown-glass','cardboard','clothes','green-glass','metal','paper','plastic','shoes','trash','white-glass'],
 'efficientnet_waste':['battery','biological','cardboard','clothes','glass','metal','paper','plastic','shoes','trash']}
SEMANTIC_ROOTS={
 'glass':['glass.n.01','glass.n.02','glassware.n.01'],'paper':['paper.n.01','newspaper.n.01','paper.n.04','document.n.01','book.n.01'],
 'electronic':['electronic_equipment.n.01','electrical_device.n.01','battery.n.02','computer.n.01','computer_hardware.n.01'],
 'metal':['metallic_element.n.01','alloy.n.01','metalware.n.01','can.n.01'],'plastic':['plastic.n.01','plastic_bag.n.01','credit_card.n.01'],
 'cardboard':['cardboard.n.01','carton.n.02'],'textile':['fabric.n.01','garment.n.01','leather.n.01'],
 'foodwaste':['food.n.01','food.n.02','garbage.n.01']}
SEMANTIC_CLASSES=['glass','paper','electronic','metal','plastic','cardboard','textile','foodwaste']
sem_raw_by_id={norm_id(p.stem):p for p in raw_train_paths}
sem_ids=np.array(sorted(sem_raw_by_id,key=lambda x:int(x)),dtype=str)
sem_flip_ids=np.asarray([str(k).removeprefix('train:') for k in flip_keys],dtype=str)
assert len(sem_ids)==1920 and len(sem_flip_ids)==256 and set(sem_flip_ids)<=set(sem_ids)
sem_source_hash=np.asarray([sha256_file(sem_raw_by_id[k]) for k in sem_ids],dtype=str)
sem_flip_hash=np.asarray([sha256_file(sem_raw_by_id[k]) for k in sem_flip_ids],dtype=str)
sem_content_fingerprint=hashlib.sha256(('|'.join(sem_ids)+'\n'+'|'.join(sem_source_hash)).encode()).hexdigest()
def _assignment_view(key,mirror=False):
    im=rgb_source(sem_raw_by_id[key]);scale=224/max(im.size)
    dims=(max(1,min(224,round(im.width*scale))),max(1,min(224,round(im.height*scale))))
    canvas=Image.new('RGB',(224,224),(124,116,104));resized=im.resize(dims,Image.Resampling.LANCZOS)
    canvas.paste(resized,((224-dims[0])//2,(224-dims[1])//2))
    return ImageOps.mirror(canvas) if mirror else canvas
def _sha_path(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
def _extract_semantic_views(model_name,model,transform,forward_fn,batch_size=16):
    items=[(key,False) for key in sem_ids]+[(key,True) for key in sem_flip_ids]
    embeddings=[];aux=[];model.eval()
    with torch.inference_mode():
        for start in range(0,len(items),batch_size):
            batch_items=items[start:start+batch_size]
            x=torch.stack([transform(_assignment_view(key,mirror)) for key,mirror in batch_items]).to(DEVICE,non_blocking=True)
            feat,extra=forward_fn(model,x)
            embeddings.append(torch.nn.functional.normalize(feat.float(),dim=1).cpu().numpy().astype('float32'))
            aux.append(None if extra is None else extra.float().cpu().numpy().astype('float32'))
            if start==0 or start+batch_size>=len(items):print(model_name,'train views',min(start+batch_size,len(items)),'/',len(items),flush=True)
    emb=np.concatenate(embeddings);extra=None if aux[0] is None else np.concatenate(aux)
    return emb[:len(sem_ids)],emb[len(sem_ids):],extra[:len(sem_ids)] if extra is not None else None,extra[len(sem_ids):] if extra is not None else None

def _cache_semantic(name,weight_sha,extract):
    cache=SEMANTIC_FEATURE_DIR/f'{name}_train_features.npz'
    identity=hashlib.sha256(json.dumps({'model':name,'weight_sha256':weight_sha,'raw_train_fingerprint':sem_content_fingerprint,
        'train_ids_sha256':hashlib.sha256('|'.join(sem_ids).encode()).hexdigest(),'flip_ids':sem_flip_ids.tolist(),
        'transform_recipe':{'resnet_waste':'raw224-letterbox124116104+resize256-centercrop224','convnext_ontology':'raw224-letterbox124116104+timm-eval-transform','efficientnet_waste':'raw224-letterbox124116104+warp224'}} ,sort_keys=True).encode()).hexdigest()
    if cache.exists():
        z=np.load(cache,allow_pickle=False)
        if str(z['identity'].item())==identity and np.array_equal(z['ids'],sem_ids) and np.array_equal(z['flip_ids'],sem_flip_ids):
            print('Using verified notebook-generated feature cache:',name);cached={k:z[k] for k in z.files}
            if 'aux' not in cached and 'probabilities' in cached:cached['aux']=cached['probabilities']
            if 'flip_aux' not in cached and 'flip_probabilities' in cached:cached['flip_aux']=cached['flip_probabilities']
            return cached
    result=extract()
    if not isinstance(result,dict):
        features,flip_features,aux,flip_aux=result
        result={'features':features,'flip_features':flip_features,'aux':aux,'flip_aux':flip_aux}
    payload={'ids':sem_ids,'flip_ids':sem_flip_ids,'source_sha256':sem_source_hash,'flip_source_sha256':sem_flip_hash,'identity':np.array(identity)}
    payload.update(result);np.savez_compressed(cache,**payload);return payload

def _weight(name):
    spec=FEATURE_MODEL_SPECS[name]
    path=hf_hub_download(spec['repo'],spec['file'],revision=spec['revision'])
    actual=_sha_path(path)
    if actual!=spec['sha256']:raise RuntimeError(f'{name} checkpoint SHA256 mismatch: {actual}')
    return Path(path),actual

# Stage7-style recycling ResNet50 features and native class probabilities.
resnet_path,resnet_sha=_weight('resnet_waste')
resnet_classes=SEMANTIC_SOURCE_CLASSES['resnet_waste']
resnet_state=torch.load(resnet_path,map_location='cpu',weights_only=True)
resnet_model=resnet50(weights=None);resnet_model.fc=torch.nn.Linear(2048,len(resnet_classes));resnet_model.load_state_dict(resnet_state)
resnet_fc_w=resnet_model.fc.weight.detach().clone().to(DEVICE);resnet_fc_b=resnet_model.fc.bias.detach().clone().to(DEVICE);resnet_model.fc=torch.nn.Identity();resnet_model=resnet_model.eval().to(DEVICE)
resnet_tf=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
def _resnet_forward(model,x):
    f=model(x);return f,torch.nn.functional.linear(f,resnet_fc_w,resnet_fc_b).softmax(-1)
resnet_waste=_cache_semantic('resnet_waste',resnet_sha,lambda:_extract_semantic_views('ResNet50-waste',resnet_model,resnet_tf,_resnet_forward))
resnet_model=None;gc.collect();
if DEVICE.type=='cuda':torch.cuda.empty_cache()

# Stage11-style ConvNeXt Tiny ImageNet-22k embeddings and WordNet-mapped priors.
conv_path,conv_sha=_weight('convnext_ontology')
conv_model=timm.create_model('convnext_tiny.fb_in22k',pretrained=True,pretrained_cfg_overlay={'file':str(conv_path)}).eval().to(DEVICE)
conv_cfg=resolve_model_data_config(conv_model);conv_tf=create_transform(**conv_cfg,is_training=False)
try:
    import nltk
except ModuleNotFoundError:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','nltk==3.9.2'])
    import nltk
from nltk.corpus import wordnet as wn
try:wn.synsets('glass')
except LookupError:
    import nltk
    if not nltk.download('wordnet',download_dir=str(WORK_ROOT/'wordnet'),quiet=True):raise RuntimeError('WordNet taxonomy download failed; enable Kaggle Internet.')
    nltk.data.path.insert(0,str(WORK_ROOT/'wordnet'));from nltk.corpus import wordnet as wn
root_synsets={wn.synset(syn):SEMANTIC_CLASSES.index(category) for category,names in SEMANTIC_ROOTS.items() for syn in names}
imagenet_names=ImageNetInfo('imagenet-22k').label_names();assert len(imagenet_names)==21841
ontology_map=np.full(len(imagenet_names),-1,dtype=np.int32)
for j,name in enumerate(imagenet_names):
    synset=wn.synset_from_pos_and_offset(name[0],int(name[1:]))
    if synset is None:continue
    distances=dict(synset.hypernym_distances());matches=[(distances[root],category) for root,category in root_synsets.items() if root in distances]
    if matches:ontology_map[j]=min(matches)[1]
assert set(ontology_map[ontology_map>=0])==set(range(8))
ontology_projection=torch.as_tensor((ontology_map[:,None]==np.arange(8)[None,:]).astype('float32'),device=DEVICE)
def _convnext_forward(model,x):
    spatial=model.forward_features(x);f=model.forward_head(spatial,pre_logits=True);logits=model.get_classifier()(f)
    prob=logits.softmax(-1);evidence=prob@ontology_projection;coverage=evidence.sum(1);conditional=evidence/coverage[:,None].clamp_min(1e-8)
    top5=prob.topk(5,dim=1).indices.float()
    return f,torch.cat([evidence,coverage[:,None],conditional,top5],dim=1)
conv_arrays=_cache_semantic('convnext_ontology',conv_sha,lambda:_extract_semantic_views('ConvNeXt-ontology',conv_model,conv_tf,_convnext_forward,batch_size=8))
conv_model=None;gc.collect();
if DEVICE.type=='cuda':torch.cuda.empty_cache()
# Split packed auxiliary output back into evidence, coverage, conditional and top-5 indices.
def _unpack_ontology(aux):
    evidence=aux[:,:len(SEMANTIC_CLASSES)];coverage=aux[:,len(SEMANTIC_CLASSES)];start=len(SEMANTIC_CLASSES)+1
    conditional=aux[:,start:start+len(SEMANTIC_CLASSES)];top5=aux[:,start+len(SEMANTIC_CLASSES):].astype(np.int32)
    return {'features':None,'evidence':evidence,'coverage':coverage,'conditional':conditional,'top5_indices':top5}
conv_aux=_unpack_ontology(conv_arrays['aux']);conv_flip_aux=_unpack_ontology(conv_arrays['flip_aux'])
convnext_ontology={'ids':conv_arrays['ids'],'flip_ids':conv_arrays['flip_ids'],'features':conv_arrays['features'],
    'flip_features':conv_arrays['flip_features'],'evidence':conv_aux['evidence'],'coverage':conv_aux['coverage'],
    'conditional':conv_aux['conditional'],'top5_indices':conv_aux['top5_indices'],
    'flip_evidence':conv_flip_aux['evidence'],'flip_coverage':conv_flip_aux['coverage'],'flip_conditional':conv_flip_aux['conditional'],
    'flip_top5_indices':conv_flip_aux['top5_indices']}

# Stage14-style EfficientNetV2-S recycling features and native class probabilities.
second_path,second_sha=_weight('efficientnet_waste')
second_config_path=hf_hub_download(FEATURE_MODEL_SPECS['efficientnet_waste']['repo'],'config.json',revision=FEATURE_MODEL_SPECS['efficientnet_waste']['revision'])
second_classes=json.loads(Path(second_config_path).read_text(encoding='utf-8'))['label_names']
if second_classes!=SEMANTIC_SOURCE_CLASSES['efficientnet_waste']:raise RuntimeError('Unexpected EfficientNet source class order.')
second_model=efficientnet_v2_s(weights=None)
second_model.classifier=torch.nn.Sequential(torch.nn.Linear(1280,256),torch.nn.ReLU(inplace=True),torch.nn.Dropout(.5),torch.nn.Linear(256,len(second_classes)))
second_model.load_state_dict(load_safetensors(str(second_path),device='cpu'))
second_head=second_model.classifier.to(DEVICE);second_model.classifier=torch.nn.Identity();second_model=second_model.eval().to(DEVICE)
second_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
def _second_forward(model,x):
    f=model(x);return f,second_head(f).softmax(-1)
second_waste=_cache_semantic('efficientnet_waste',second_sha,lambda:_extract_semantic_views('EfficientNetV2-S-waste',second_model,second_tf,_second_forward))
second_model=None;second_head=None;gc.collect();
if DEVICE.type=='cuda':torch.cuda.empty_cache()

def _split_feature_bank(bank,aux_key='aux',flip_aux_key='flip_aux'):
    return {'features':bank['features'],'flip_features':bank['flip_features'],
        'probabilities':bank[aux_key],'flip_probabilities':bank[flip_aux_key],
        'ids':bank['ids'],'flip_ids':bank['flip_ids']}
resnet_waste=_split_feature_bank(resnet_waste)
second_waste=_split_feature_bank(second_waste)
assert resnet_waste['features'].shape==(1920,2048) and second_waste['features'].shape==(1920,1280)
assert convnext_ontology['features'].shape==(1920,768) and convnext_ontology['conditional'].shape==(1920,8)
assert all(np.isfinite(v).all() for v in [resnet_waste['features'],resnet_waste['probabilities'],second_waste['features'],second_waste['probabilities'],convnext_ontology['features'],convnext_ontology['conditional']])
# Rebuild the pinned eight-class image-only teacher feature bank from train images.
# The upstream checkpoint is a data-only pickle archive; parse tensor records without executing reducers/callables.
import pickletools, zipfile
from dataclasses import dataclass, field
from huggingface_hub import hf_hub_download

TASK8_REPOSITORY='C0mpl1cated3/trash_classifier_v2'
TASK8_REVISION='ac54e811e3a5d505b57bf6dc1495ed5cd695c036'
TASK8_SOURCE_SHA256='cbefd51e99c7f325fa1f08b44ed185160895be5bd7f49c7cfd046a329fee5b19'
TASK8_CLASS_ORDER=['cardboard','e-waste','glass','metal','organic','paper','plastic','textile']
TASK8_TO_TASK={'cardboard':'cardboard','e-waste':'electronic','glass':'glass','metal':'metal','organic':'foodwaste','paper':'paper','plastic':'plastic','textile':'textile'}
@dataclass(frozen=True)
class _S18Symbol: name:str
@dataclass(eq=False)
class _S18Node:
    factory:object;args:object=();state:object=None;sequence:list=field(default_factory=list);mapping:dict=field(default_factory=dict)
@dataclass
class _S18Storage: kind:str;key:str;count:int
@dataclass
class _S18Tensor: storage:object;offset:int;shape:tuple;stride:tuple

def _s18_fields(node):
    if isinstance(node,dict):return node
    if not isinstance(node,_S18Node):return {}
    if isinstance(node.state,dict):return node.state
    if isinstance(node.state,tuple):return {k:v for part in node.state if isinstance(part,dict) for k,v in part.items()}
    return node.mapping

def _s18_parse_pickle(data):
    stack=[];memo={};marker=object()
    def marked():
        index=len(stack)-1
        while index>=0 and stack[index] is not marker:index-=1
        if index<0:raise ValueError('Malformed serialized tensor archive: MARK not found.')
        values=stack[index+1:];del stack[index:];return values
    def put(obj,key,value):(obj if isinstance(obj,dict) else obj.mapping).__setitem__(key,value)
    def extend(obj,values):(obj if isinstance(obj,list) else obj.sequence).extend(values)
    def reduce_data(fn,args):
        name=fn.name if isinstance(fn,_S18Symbol) else ''
        if name=='collections.OrderedDict':return dict(args[0]) if args else {}
        if name in ['torch._utils._rebuild_tensor_v2','torch._utils._rebuild_tensor']:return _S18Tensor(args[0],int(args[1]),tuple(args[2]),tuple(args[3]))
        if name.startswith('torch._utils._rebuild_parameter'):return args[0]
        if name=='torch._tensor._rebuild_from_type_v2':return reduce_data(args[0],args[2])
        return _S18Node(fn,args)
    for op,arg,pos in pickletools.genops(data):
        name=op.name
        if name in ['PROTO','FRAME']:continue
        if name=='MARK':stack.append(marker)
        elif name=='STOP':return stack[-1]
        elif name=='POP':stack.pop()
        elif name=='POP_MARK':marked()
        elif name=='DUP':stack.append(stack[-1])
        elif name=='NONE':stack.append(None)
        elif name=='NEWTRUE':stack.append(True)
        elif name=='NEWFALSE':stack.append(False)
        elif name in ['INT','BININT','BININT1','BININT2','LONG','LONG1','LONG4','FLOAT','BINFLOAT','STRING','BINSTRING','SHORT_BINSTRING','UNICODE','BINUNICODE','SHORT_BINUNICODE','BINUNICODE8','BINBYTES','SHORT_BINBYTES','BINBYTES8','BYTEARRAY8']:stack.append(arg)
        elif name=='EMPTY_TUPLE':stack.append(())
        elif name=='EMPTY_LIST':stack.append([])
        elif name=='EMPTY_DICT':stack.append({})
        elif name=='EMPTY_SET':stack.append(set())
        elif name=='TUPLE':stack.append(tuple(marked()))
        elif name in ['TUPLE1','TUPLE2','TUPLE3']:
            n=int(name[-1]);values=tuple(stack[-n:]);del stack[-n:];stack.append(values)
        elif name=='LIST':stack.append(marked())
        elif name=='DICT':
            values=marked();stack.append(dict(zip(values[::2],values[1::2])))
        elif name=='APPEND':value=stack.pop();extend(stack[-1],[value])
        elif name=='APPENDS':values=marked();extend(stack[-1],values)
        elif name=='SETITEM':value=stack.pop();key=stack.pop();put(stack[-1],key,value)
        elif name=='SETITEMS':
            values=marked()
            for key,value in zip(values[::2],values[1::2]):put(stack[-1],key,value)
        elif name=='GLOBAL':stack.append(_S18Symbol(arg.replace(' ','.')))
        elif name=='STACK_GLOBAL':attribute=stack.pop();module=stack.pop();stack.append(_S18Symbol(module+'.'+attribute))
        elif name=='REDUCE':args=stack.pop();fn=stack.pop();stack.append(reduce_data(fn,args))
        elif name in ['NEWOBJ','NEWOBJ_EX']:
            if name=='NEWOBJ_EX':stack.pop()
            args=stack.pop();cls=stack.pop();stack.append(_S18Node(cls,args))
        elif name=='BUILD':
            state=stack.pop();obj=stack[-1]
            if isinstance(obj,_S18Node):obj.state=state
        elif name in ['BINPUT','LONG_BINPUT','PUT']:memo[int(arg)]=stack[-1]
        elif name=='MEMOIZE':memo[len(memo)]=stack[-1]
        elif name in ['BINGET','LONG_BINGET','GET']:stack.append(memo[int(arg)])
        elif name=='BINPERSID':
            ident=stack.pop();assert ident[0]=='storage' and isinstance(ident[1],_S18Symbol)
            stack.append(_S18Storage(ident[1].name,str(ident[2]),int(ident[4])))
        else:raise ValueError(f'Unsupported checkpoint metadata opcode {name} at {pos}; no serialized callable was run.')
    raise ValueError('Checkpoint metadata has no STOP opcode.')

def _s18_extract_task8_state(checkpoint_path):
    dtype_map={'torch.FloatStorage':'<f4','torch.DoubleStorage':'<f8','torch.HalfStorage':'<f2','torch.LongStorage':'<i8','torch.IntStorage':'<i4','torch.ByteStorage':'u1','torch.BoolStorage':'?'}
    state={}
    with zipfile.ZipFile(checkpoint_path) as archive:
        assert sum(item.file_size for item in archive.infolist())<1024**3
        pickle_name=next(name for name in archive.namelist() if name.endswith('/data.pkl'));prefix=pickle_name.rsplit('/',1)[0]+'/'
        root=_s18_parse_pickle(archive.read(pickle_name));model=_s18_fields(root).get('model');assert model is not None
        specs={}
        def collect(node,path=''):
            data=_s18_fields(node)
            for section in ['_parameters','_buffers']:
                for key,value in data.get(section,{}).items():
                    if value is not None:assert isinstance(value,_S18Tensor);specs[path+key]=value
            for key,child in data.get('_modules',{}).items():
                if child is not None:collect(child,path+key+'.')
        collect(model);assert len(specs)>200
        stores={}
        for name,spec in specs.items():
            storage=spec.storage;assert storage.kind in dtype_map
            if storage.key not in stores:
                values=np.frombuffer(archive.read(prefix+'data/'+storage.key),dtype=np.dtype(dtype_map[storage.kind]))
                assert len(values)==storage.count;stores[storage.key]=values
            values=stores[storage.key]
            assert all(dim>=0 for dim in spec.shape) and all(stride>=0 for stride in spec.stride)
            largest=spec.offset+sum((dim-1)*stride for dim,stride in zip(spec.shape,spec.stride))
            assert 0<=spec.offset<=largest<len(values)
            view=np.lib.stride_tricks.as_strided(values[spec.offset:],shape=spec.shape,strides=tuple(stride*values.dtype.itemsize for stride in spec.stride)).copy()
            state[name]=torch.from_numpy(view)
    return state

task8_checkpoint=hf_hub_download(TASK8_REPOSITORY,'waste_classifier_v2.pkl',revision=TASK8_REVISION)
assert _sha_path(task8_checkpoint)==TASK8_SOURCE_SHA256,'Pinned eight-class checkpoint hash mismatch.'
task8_identity=hashlib.sha256((sem_content_fingerprint+TASK8_SOURCE_SHA256+'|'+','.join(sem_flip_ids)).encode()).hexdigest()
task8_cache=SEMANTIC_FEATURE_DIR/'task8_teacher_train_features.npz'
if task8_cache.exists():
    _z=np.load(task8_cache,allow_pickle=False)
    if str(_z['identity'].item())==task8_identity and np.array_equal(_z['ids'],sem_ids) and np.array_equal(_z['flip_ids'],sem_flip_ids):
        task8_teacher_features=_z['features'];task8_teacher_flip_features=_z['flip_features']
        task8_teacher_probabilities=_z['probabilities'];task8_teacher_flip_probabilities=_z['flip_probabilities']
    else:task8_cache.unlink()
if not task8_cache.exists():
    class _S18ConcatPool(torch.nn.Module):
        def forward(self,x):return torch.cat([torch.nn.functional.adaptive_max_pool2d(x,1),torch.nn.functional.adaptive_avg_pool2d(x,1)],dim=1)
    _body=torch.nn.Sequential(*list(torchvision.models.resnet50(weights=None).children())[:-2])
    _head=torch.nn.Sequential(_S18ConcatPool(),torch.nn.Flatten(),torch.nn.BatchNorm1d(4096),torch.nn.Dropout(.25),torch.nn.Linear(4096,512,bias=False),torch.nn.ReLU(inplace=True),torch.nn.BatchNorm1d(512),torch.nn.Dropout(.5),torch.nn.Linear(512,8,bias=False))
    _task8_model=torch.nn.Sequential(_body,_head)
    _task8_model.load_state_dict(_s18_extract_task8_state(task8_checkpoint),strict=True);_task8_model.eval().to(DEVICE)
    _task8_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    _task8_items=[(key,False) for key in sem_ids]+[(key,True) for key in sem_flip_ids]
    _task8_feats=[];_task8_probs=[]
    with torch.inference_mode():
        for start in range(0,len(_task8_items),8):
            rows=_task8_items[start:start+8];images=[]
            for key,mirror in rows:
                image=ImageOps.fit(rgb_source(sem_raw_by_id[key]),(224,224),method=Image.Resampling.BILINEAR,centering=(.5,.5))
                if mirror:image=ImageOps.mirror(image)
                images.append(_task8_tf(image))
            raw=_task8_model[1][0](_task8_model[0](torch.stack(images).to(DEVICE))).flatten(1)
            logits=_task8_model[1][2:](raw);source_prob=logits.softmax(-1).cpu().numpy()
            task_prob=np.zeros((len(rows),8),dtype=np.float32)
            for j,source_name in enumerate(TASK8_CLASS_ORDER):task_prob[:,SEMANTIC_CLASSES.index(TASK8_TO_TASK[source_name])]+=source_prob[:,j]
            _task8_feats.append(torch.nn.functional.normalize(raw,dim=1).cpu().numpy().astype(np.float32));_task8_probs.append(task_prob)
            if start==0 or (start//8+1)%24==0:print('Eight-class teacher train features',min(start+8,len(_task8_items)),'/',len(_task8_items),flush=True)
    _task8_all_features=np.vstack(_task8_feats);_task8_all_prob=np.vstack(_task8_probs)
    task8_teacher_features=_task8_all_features[:len(sem_ids)];task8_teacher_flip_features=_task8_all_features[len(sem_ids):]
    task8_teacher_probabilities=_task8_all_prob[:len(sem_ids)];task8_teacher_flip_probabilities=_task8_all_prob[len(sem_ids):]
    np.savez_compressed(task8_cache,ids=sem_ids,flip_ids=sem_flip_ids,source_sha256=sem_source_hash,
        features=task8_teacher_features,flip_features=task8_teacher_flip_features,probabilities=task8_teacher_probabilities,
        flip_probabilities=task8_teacher_flip_probabilities,identity=np.array(task8_identity),checkpoint_sha256=np.array(TASK8_SOURCE_SHA256))
    del _task8_model,_body,_head,_task8_feats,_task8_probs,_task8_all_features,_task8_all_prob
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
assert task8_teacher_features.shape==(len(sem_ids),4096) and task8_teacher_flip_features.shape==(len(sem_flip_ids),4096)
assert np.isfinite(task8_teacher_features).all() and np.isfinite(task8_teacher_flip_features).all()

# Build auditable weak source-consensus anchors from raw-train CNN evidence.
# This reproduces the source-vote construction pattern; it is not a final assignment.
ASSIGNMENT_CLASS_ORDER=list(SEMANTIC_CLASSES)
_SOURCE_TO_TASK={
 'battery':'electronic','biological':'foodwaste','brown-glass':'glass','green-glass':'glass','white-glass':'glass',
 'glass':'glass','cardboard':'cardboard','clothes':'textile','shoes':'textile','metal':'metal','paper':'paper','plastic':'plastic'}
def _map_source_probabilities(probabilities,source_classes):
    evidence=np.zeros((len(probabilities),len(ASSIGNMENT_CLASS_ORDER)),dtype=np.float32)
    for j,name in enumerate(source_classes):
        target=_SOURCE_TO_TASK.get(name)
        if target is not None:evidence[:,ASSIGNMENT_CLASS_ORDER.index(target)]+=probabilities[:,j]
    coverage=evidence.sum(1)
    conditional=evidence/np.maximum(coverage[:,None],1e-8)
    return conditional,coverage
resnet_task_prior,resnet_task_coverage=_map_source_probabilities(resnet_waste['probabilities'],SEMANTIC_SOURCE_CLASSES['resnet_waste'])
efficientnet_task_prior,efficientnet_task_coverage=_map_source_probabilities(second_waste['probabilities'],SEMANTIC_SOURCE_CLASSES['efficientnet_waste'])
source_priors=np.stack([resnet_task_prior,efficientnet_task_prior,convnext_ontology['conditional']]).astype(np.float32)
source_coverages=np.stack([resnet_task_coverage,efficientnet_task_coverage,convnext_ontology['coverage']]).astype(np.float32)
source_thresholds=np.asarray([.6,.6,.2],dtype=np.float32)[:,None]
source_reliable=(source_priors.max(2)>=.8)&(source_coverages>=source_thresholds)
source_votes=np.zeros((len(sem_ids),len(ASSIGNMENT_CLASS_ORDER)),dtype=np.uint8)
for j in range(source_priors.shape[0]):
    rows=np.flatnonzero(source_reliable[j])
    source_votes[rows,source_priors[j,rows].argmax(1)]+=1
ranked_votes=np.sort(source_votes,axis=1)
source_anchor_mask=(ranked_votes[:,-1]>=2)&(ranked_votes[:,-1]>ranked_votes[:,-2])
source_anchor_labels=np.full(len(sem_ids),-1,dtype=np.int16)
source_anchor_labels[source_anchor_mask]=source_votes[source_anchor_mask].argmax(1)
assert source_anchor_mask.any() and np.all(source_anchor_labels[~source_anchor_mask]==-1)
SOURCE_ANCHOR_DIR=WORK_ROOT/'labels'/'source_consensus_train_only'
SOURCE_ANCHOR_DIR.mkdir(parents=True,exist_ok=True)
np.savez_compressed(SOURCE_ANCHOR_DIR/'source_consensus_anchors.npz',image_ids=sem_ids,source_sha256=sem_source_hash,
    anchor_mask=source_anchor_mask,anchor_label_index=source_anchor_labels,source_votes=source_votes,
    source_reliable=source_reliable,source_coverages=source_coverages,source_probabilities=source_priors,
    class_order=np.asarray(ASSIGNMENT_CLASS_ORDER),source_names=np.asarray(['resnet_waste','efficientnet_waste','convnext_ontology']))
source_anchor_report={'method':'three_family_non_vlm_train_only_consensus','train_images':len(sem_ids),'test_images_processed':0,
    'source_families':['resnet_waste','efficientnet_waste','convnext_ontology'],'minimum_agreeing_sources':2,
    'reliability_rule':'per-source maximum conditional probability >= 0.8 and coverage >= [0.6,0.6,0.2]',
    'anchor_rows':int(source_anchor_mask.sum()),'unanchored_rows':int((~source_anchor_mask).sum()),
    'anchor_class_counts':{c:int(np.sum(source_anchor_labels[source_anchor_mask]==i)) for i,c in enumerate(ASSIGNMENT_CLASS_ORDER)},
    'test_used_for_grouping_fit_selection_or_anchor_generation':False,'official_ground_truth':False,
    'raw_train_fingerprint':sem_content_fingerprint}
(SOURCE_ANCHOR_DIR/'source_consensus_report.json').write_text(json.dumps(source_anchor_report,indent=2,ensure_ascii=False),encoding='utf-8')
SEMANTIC_FEATURE_PROVENANCE={'train_images':len(sem_ids),'flip_train_views':len(sem_flip_ids),'test_images_processed':0,
    'raw_train_fingerprint':sem_content_fingerprint,'models':{k:{'repo':v['repo'],'revision':v['revision'],'weight_sha256':FEATURE_MODEL_SPECS[k]['sha256']} for k,v in FEATURE_MODEL_SPECS.items()},
    'feature_shapes':{'resnet_waste':list(resnet_waste['features'].shape),'convnext_ontology':list(convnext_ontology['features'].shape),'efficientnet_waste':list(second_waste['features'].shape)},
    'source_class_orders':SEMANTIC_SOURCE_CLASSES,'ontology_classes':SEMANTIC_CLASSES,'source_consensus_anchor_rows':int(source_anchor_mask.sum()),
    'test_used_for_fit_or_feature_selection':False,'assignment_heads_retrained':False,'active_assignment_parity_verified':False}
(SEMANTIC_FEATURE_DIR/'semantic_feature_provenance.json').write_text(json.dumps(SEMANTIC_FEATURE_PROVENANCE,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps({'feature_provenance':SEMANTIC_FEATURE_PROVENANCE,'source_anchor_report':source_anchor_report},indent=2,ensure_ascii=False))


### 4b. Rebuild generic ImageNet object evidence from raw train

ResNet50 ImageNet object logits are recalculated from train originals and the fixed train flip probe. The eight task classes receive only the published object-to-category evidence mapping; this is a weak prior, not a final assignment. Test images are not read in this step.


In [ ]:
# Stage6 object-evidence branch, recomputed from raw TRAIN only.
import gc, hashlib, json, time
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import to_tensor, normalize as tv_normalize

OBJECT_FEATURE_DIR=WORK_ROOT/'features'/'generic_object_evidence'
OBJECT_FEATURE_DIR.mkdir(parents=True,exist_ok=True)
OBJECT_FEATURE_CACHE=OBJECT_FEATURE_DIR/'resnet50_imagenet_v2_train_only.npz'
OBJECT_WEIGHTS=ResNet50_Weights.IMAGENET1K_V2
OBJECT_CLASS_ORDER=list(SEMANTIC_CLASSES)
OBJECT_TERMS={
 'glass':['beer bottle','beaker','goblet','wine bottle','beer glass'],
 'paper':['comic book','book jacket','crossword puzzle','envelope','menu','paper towel','toilet tissue'],
 'electronic':['computer keyboard','cellular telephone','desktop computer','digital clock','electric fan','electric guitar','espresso maker','hair dryer','iPod','iron','laptop','microphone','microwave','modem','monitor','mouse','oscilloscope','photocopier','printer','projector','radio','reflex camera','refrigerator','remote control','sewing machine','space heater','stove','switch','tape player','television','toaster','vacuum','waffle iron','washer','CD player','cassette player','dial telephone','loudspeaker','power drill'],
 'metal':['ashcan','barrel','bottlecap','can opener','chain','cocktail shaker','coffeepot','corkscrew','frying pan','milk can','nail','screw','thimble','wok','padlock','safety pin','screwdriver','wrench','hatchet','hammer','steel drum','soup bowl'],
 'plastic':['water bottle','pop bottle','pill bottle','plastic bag','bucket','soap dispenser','water jug','syringe','rubber eraser','ping-pong ball','tennis ball'],
 'cardboard':['carton'],
 'textile':['abaya','apron','bath towel','bikini','bow tie','brassiere','cardigan','Christmas stocking','diaper','dishrag','doormat','feather boa','fur coat','gown','handkerchief','jersey','kimono','knee pad','lab coat','maillot','mitten','pajama','poncho','quilt','running shoe','sandal','sarong','ski mask','sleeping bag','sock','stole','suit','sweatshirt','swimming trunks','trench coat','velvet','vestment','wool','backpack','purse','pillow','teddy'],
 'foodwaste':['guacamole','consomme','hot pot','trifle','ice cream','ice lolly','French loaf','bagel','pretzel','cheeseburger','hotdog','mashed potato','head cabbage','broccoli','cauliflower','zucchini','spaghetti squash','acorn squash','butternut squash','cucumber','artichoke','bell pepper','cardoon','mushroom','Granny Smith','strawberry','orange','lemon','fig','pineapple','banana','jackfruit','custard apple','pomegranate','hay','carbonara','chocolate sauce','dough','meat loaf','pizza','potpie','burrito','red wine','espresso','eggnog','crab','American lobster','spiny lobster','crayfish','Dungeness crab','rock crab','fiddler crab','sea cucumber']}
OBJECT_ITEMS=[(k,False) for k in sem_ids]+[(k,True) for k in sem_flip_ids]
object_identity=hashlib.sha256(json.dumps({'source_fingerprint':sem_content_fingerprint,'weights':OBJECT_WEIGHTS.url,
 'weights_enum':'IMAGENET1K_V2','view':'stage1 raw RGB letterbox224 pad124116104','flip_ids':sem_flip_ids.tolist(),
 'feature':'ResNet50 1000 class logits + max object-category evidence'},sort_keys=True).encode()).hexdigest()
object_logits=None
if OBJECT_FEATURE_CACHE.exists():
    q=np.load(OBJECT_FEATURE_CACHE,allow_pickle=False)
    if str(q['identity'].item())==object_identity and np.array_equal(q['image_ids'],sem_ids) and np.array_equal(q['flip_ids'],sem_flip_ids):
        object_logits=q['logits'].astype(np.float32);object_flip_logits=q['flip_logits'].astype(np.float32)
        print('Loaded verified notebook-generated ImageNet object evidence cache.')
if object_logits is None:
    object_model=resnet50(weights=OBJECT_WEIGHTS).eval().to(DEVICE)
    object_mean=(.485,.456,.406);object_std=(.229,.224,.225)
    def _object_input(key,mirror):
        canvas=_assignment_view(key,mirror)
        return tv_normalize(to_tensor(canvas),object_mean,object_std)
    object_outputs=[];object_flip_outputs=[];object_started=time.time()
    with torch.inference_mode():
        for first in range(0,len(OBJECT_ITEMS),32):
            batch_items=OBJECT_ITEMS[first:first+32]
            batch=torch.stack([_object_input(k,m) for k,m in batch_items]).to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
                logits=object_model(batch)
            values=logits.float().cpu().numpy().astype(np.float32)
            for item,value in zip(batch_items,values):
                (object_outputs if not item[1] else object_flip_outputs).append(value)
            if first==0 or first+len(batch_items)>=len(OBJECT_ITEMS):
                print('ResNet50 ImageNet object logits',min(first+len(batch_items),len(OBJECT_ITEMS)),'/',len(OBJECT_ITEMS),flush=True)
    object_logits=np.stack(object_outputs);object_flip_logits=np.stack(object_flip_outputs)
    assert object_logits.shape==(len(sem_ids),1000) and object_flip_logits.shape==(len(sem_flip_ids),1000)
    np.savez_compressed(OBJECT_FEATURE_CACHE,image_ids=sem_ids,flip_ids=sem_flip_ids,source_sha256=sem_source_hash,
        flip_source_sha256=sem_flip_hash,logits=object_logits,flip_logits=object_flip_logits,
        class_names=np.asarray(OBJECT_WEIGHTS.meta['categories']),identity=np.array(object_identity),weights_url=np.array(OBJECT_WEIGHTS.url))
    del object_model;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
object_names=list(OBJECT_WEIGHTS.meta['categories']);object_lookup={name:i for i,name in enumerate(object_names)}
object_prob=torch.softmax(torch.from_numpy(object_logits),dim=1).numpy()
object_evidence=np.zeros((len(sem_ids),len(OBJECT_CLASS_ORDER)),dtype=np.float32);missing_object_terms={}
for ci,category in enumerate(OBJECT_CLASS_ORDER):
    available=[object_lookup[t] for t in OBJECT_TERMS[category] if t in object_lookup]
    missing_object_terms[category]=[t for t in OBJECT_TERMS[category] if t not in object_lookup]
    assert available,category
    object_evidence[:,ci]=object_prob[:,available].max(1)
object_coverage=np.clip(object_evidence.max(1)/np.maximum(object_prob.max(1),1e-8),0,1)
OBJECT_EVIDENCE_PROVENANCE={'method':'torchvision_resnet50_imagenet_v2_object_evidence','train_images':len(sem_ids),
 'flip_train_views':len(sem_flip_ids),'test_images_processed':0,'source_fingerprint':sem_content_fingerprint,
 'weights_url':OBJECT_WEIGHTS.url,'weights_enum':'IMAGENET1K_V2','cache_identity':object_identity,
 'class_order':OBJECT_CLASS_ORDER,'missing_object_terms':missing_object_terms,
 'test_used_for_fit_grouping_or_selection':False,'external_training_images_loaded':False}
(OBJECT_FEATURE_DIR/'provenance.json').write_text(json.dumps(OBJECT_EVIDENCE_PROVENANCE,indent=2,ensure_ascii=False),encoding='utf-8')
print('Raw-train ImageNet object evidence ready:',object_evidence.shape,'| avg coverage',round(float(object_coverage.mean()),4))


### 4c. Rebuild MINC material evidence and calibrated generic prior

This cell downloads only the pinned pretrained MINC GoogLeNet model files (no images), evaluates two fixed views of raw train images, and combines material/object evidence using the historical ontology rule. Results feed the train-only assignment-head replay; they are not treated as verified labels.


In [ ]:
# Historical generic object/material prior, rebuilt from raw TRAIN images only.
import concurrent.futures, hashlib, json, tarfile, time, urllib.request
import cv2

MINC_DIR=WORK_ROOT/'pretrained'/'minc15'
MINC_MODEL_DIR=MINC_DIR/'model'
MINC_MODEL_DIR.mkdir(parents=True,exist_ok=True)
MINC_URL='https://opensurfaces.cs.cornell.edu/static/minc/minc-model.tar.gz'
MINC_EXPECTED={'categories.txt':'ef8d90b59508e8b5c79f07ef4b6811017a228d8960bbbaeee02a203bb796be90',
 'deploy-googlenet.prototxt':'5174393647c3073624e540abcf09ed1d1c54ac4e74d90063a61789cfee6ccbcc',
 'minc-googlenet.caffemodel':'471f1a429f4fbd8b4771abaaf0fe0eb9aeaed3257321abec3e2cb537bffefba9'}
def _file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
def _minc_model_ready():
    return all((MINC_MODEL_DIR/name).exists() and _file_sha(MINC_MODEL_DIR/name)==digest for name,digest in MINC_EXPECTED.items())
if not _minc_model_ready():
    ranges=MINC_DIR/'ranges';ranges.mkdir(exist_ok=True);chunk_size=8*1024*1024
    def _fetch_minc_range(index):
        target=ranges/f'{index:04d}.part'
        if target.exists() and target.stat().st_size==chunk_size:return index
        start=index*chunk_size;end=start+chunk_size-1
        request=urllib.request.Request(MINC_URL,headers={'Range':f'bytes={start}-{end}'})
        for attempt in range(3):
            try:
                with urllib.request.urlopen(request,timeout=180) as response:
                    if response.status!=206 or not response.headers.get('Content-Range','').startswith(f'bytes {start}-{end}/'):
                        raise RuntimeError('MINC server did not honor the requested byte range.')
                    data=response.read()
                if len(data)!=chunk_size:raise RuntimeError('Unexpected MINC range length.')
                target.write_bytes(data);return index
            except Exception:
                if attempt==2:raise
    extracted=False
    # The model follows the unused AlexNet payload in the published tarball.
    for limit_mb in [320,400,512,640,712]:
        needed=limit_mb//8
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
            list(pool.map(_fetch_minc_range,range(needed)))
        prefix=MINC_DIR/'model-prefix.tar.gz'
        with prefix.open('wb') as dst:
            for j in range(needed):
                with (ranges/f'{j:04d}.part').open('rb') as src:
                    for block in iter(lambda:src.read(1<<20),b''):dst.write(block)
        selected=[]
        try:
            with tarfile.open(prefix,mode='r|gz') as archive:
                for member in archive:
                    name=Path(member.name).name
                    if member.isfile() and name in MINC_EXPECTED:
                        target=MINC_MODEL_DIR/name
                        source=archive.extractfile(member)
                        if source is None:raise RuntimeError('Could not read requested MINC model member.')
                        with source,target.open('wb') as dst:
                            for block in iter(lambda:source.read(1<<20),b''):dst.write(block)
                        selected.append(name)
                        if name=='minc-googlenet.caffemodel':break
        except (tarfile.ReadError,EOFError):
            pass
        if len(selected)==3 and _minc_model_ready():
            extracted=True;break
    if not extracted and not _minc_model_ready():
        raise RuntimeError('Could not recover and checksum the three MINC model files. Enable Kaggle Internet and rerun.')
if not hasattr(cv2.dnn,'readNetFromCaffe'):
    raise RuntimeError(f'OpenCV {cv2.__version__} lacks Caffe DNN support required for the pinned MINC material model.')
MINC_CATEGORIES=(MINC_MODEL_DIR/'categories.txt').read_text(encoding='utf-8').splitlines()
assert len(MINC_CATEGORIES)==23
minc_net=cv2.dnn.readNetFromCaffe(str(MINC_MODEL_DIR/'deploy-googlenet.prototxt'),str(MINC_MODEL_DIR/'minc-googlenet.caffemodel'))
minc_net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV);minc_net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU);cv2.setNumThreads(4)
def _minc_views(key,mirror=False):
    im=_assignment_view(key,False)
    full=im.resize((224,224),Image.Resampling.LANCZOS)
    side=max(1,round(min(im.size)*.7));left=(im.width-side)//2;top=(im.height-side)//2
    center=im.crop((left,top,left+side,top+side)).resize((224,224),Image.Resampling.LANCZOS)
    if mirror:full=ImageOps.mirror(full);center=ImageOps.mirror(center)
    return [cv2.cvtColor(np.asarray(v),cv2.COLOR_RGB2BGR) for v in [full,center]]
MINC_ITEMS=[(key,False) for key in sem_ids]+[(key,True) for key in sem_flip_ids]
minc_identity=hashlib.sha256(json.dumps({'raw_train_fingerprint':sem_content_fingerprint,'weights':MINC_EXPECTED,
 'view_recipe':'full+center_square_0.7_of_short_side;224;mean_bgr104117124','flip_ids':sem_flip_ids.tolist()},sort_keys=True).encode()).hexdigest()
MINC_CACHE=WORK_ROOT/'features'/'generic_object_evidence'/'minc_material_probabilities_train_only.npz'
minc_original=minc_flip=None
if MINC_CACHE.exists():
    z=np.load(MINC_CACHE,allow_pickle=False)
    if str(z['identity'].item())==minc_identity and np.array_equal(z['image_ids'],sem_ids) and np.array_equal(z['flip_ids'],sem_flip_ids):
        minc_original=z['probabilities'].astype(np.float32);minc_flip=z['flip_probabilities'].astype(np.float32)
        print('Loaded verified notebook-generated MINC feature cache.')
if minc_original is None:
    output_original=[];output_flip=[];started=time.time()
    for first in range(0,len(MINC_ITEMS),8):
        items=MINC_ITEMS[first:first+8]
        batch=[view for key,mirror in items for view in _minc_views(key,mirror)]
        blob=cv2.dnn.blobFromImages(batch,scalefactor=1,size=(224,224),mean=(104,117,124),swapRB=False,crop=False)
        minc_net.setInput(blob);pred=minc_net.forward().reshape(len(items),2,len(MINC_CATEGORIES))
        assert np.isfinite(pred).all() and np.allclose(pred.sum(2),1,atol=1e-4)
        means=pred.mean(1)
        for (key,mirror),row in zip(items,means):
            (output_flip if mirror else output_original).append(row.astype(np.float32))
        if first==0 or first+len(items)>=len(MINC_ITEMS):
            print('MINC raw-train material probabilities',min(first+len(items),len(MINC_ITEMS)),'/',len(MINC_ITEMS),'elapsed',round(time.time()-started),'s',flush=True)
    minc_original=np.stack(output_original);minc_flip=np.stack(output_flip)
    assert minc_original.shape==(len(sem_ids),23) and minc_flip.shape==(len(sem_flip_ids),23)
    np.savez_compressed(MINC_CACHE,image_ids=sem_ids,flip_ids=sem_flip_ids,source_sha256=sem_source_hash,
       flip_source_sha256=sem_flip_hash,probabilities=minc_original,flip_probabilities=minc_flip,
       categories=np.asarray(MINC_CATEGORIES),identity=np.array(minc_identity),model_sha256=np.array(MINC_EXPECTED['minc-googlenet.caffemodel']))
# Apply the historical calibrated object/material evidence fusion, with no archive labels.
def _calibrated_generic_prior(object_logits,material_probabilities):
    prob=torch.softmax(torch.from_numpy(object_logits),dim=1).numpy();lookup={name:i for i,name in enumerate(OBJECT_WEIGHTS.meta['categories'])}
    obj=np.zeros((len(prob),8),dtype=np.float32)
    for ci,category in enumerate(OBJECT_CLASS_ORDER):
        obj[:,ci]=prob[:,[lookup[t] for t in OBJECT_TERMS[category] if t in lookup]].max(1)
    op=obj/np.maximum(obj.sum(1,keepdims=True),1e-8)
    coverage=np.clip(obj.max(1)/np.maximum(prob.max(1),1e-8),0,1)
    material=np.zeros_like(op);midx={name:i for i,name in enumerate(MINC_CATEGORIES)}
    material_map={'glass':['glass'],'paper':['paper'],'metal':['metal'],'plastic':['plastic'],'textile':['fabric','leather'],'foodwaste':['food']}
    for category,sources in material_map.items():material[:,OBJECT_CLASS_ORDER.index(category)]=material_probabilities[:,[midx[n] for n in sources]].sum(1)
    mapped=material.sum(1);mp=material/np.maximum(mapped[:,None],1e-8)
    object_weight=.75*coverage/(.75*coverage+.25*mapped+1e-8)
    score=object_weight[:,None]*op+(1-object_weight[:,None])*mp
    for category in ['electronic','cardboard']:
        ci=OBJECT_CLASS_ORDER.index(category);other=op.copy();other[:,ci]=0
        strong=(op[:,ci]*coverage>=.35)&((op[:,ci]-other.max(1))>=.15)
        score[strong]=.9*op[strong]+.1*mp[strong]
    score=np.maximum(score,1e-8);score/=score.sum(1,keepdims=True)
    return score.astype(np.float32),obj,coverage
GENERIC_PRIOR,GENERIC_OBJECT_EVIDENCE,GENERIC_OBJECT_COVERAGE=_calibrated_generic_prior(object_logits,minc_original)
GENERIC_PRIOR_FLIP,GENERIC_OBJECT_EVIDENCE_FLIP,GENERIC_OBJECT_COVERAGE_FLIP=_calibrated_generic_prior(object_flip_logits,minc_flip)
GENERIC_PRIOR_DIR=WORK_ROOT/'features'/'generic_object_evidence'
np.savez_compressed(GENERIC_PRIOR_DIR/'calibrated_generic_priors_train_only.npz',image_ids=sem_ids,flip_ids=sem_flip_ids,
    source_sha256=sem_source_hash,flip_source_sha256=sem_flip_hash,prior_probabilities=GENERIC_PRIOR,flip_prior_probabilities=GENERIC_PRIOR_FLIP,
    object_evidence=GENERIC_OBJECT_EVIDENCE,object_coverage=GENERIC_OBJECT_COVERAGE,minc_material_probabilities=minc_original,
    class_order=np.asarray(OBJECT_CLASS_ORDER),minc_categories=np.asarray(MINC_CATEGORIES),identity=np.array(minc_identity))
GENERIC_PRIOR_PROVENANCE={'method':'ResNet50-ImageNetV2-object-evidence_plus_MINC-GoogLeNet-material-evidence-calibrated-fusion',
 'train_images':len(sem_ids),'flip_train_views':len(sem_flip_ids),'test_images_processed':0,'raw_train_fingerprint':sem_content_fingerprint,
 'object_weights_url':OBJECT_WEIGHTS.url,'object_logits_shape':list(object_logits.shape),'minc_model_sha256':MINC_EXPECTED,
 'minc_categories':MINC_CATEGORIES,'minc_view_aggregation':'mean of full-frame and center-square probability','class_order':OBJECT_CLASS_ORDER,
 'test_used_for_fit_or_selection':False,'archive_assignment_labels_read':False,'official_ground_truth':False}
(GENERIC_PRIOR_DIR/'calibrated_generic_prior_provenance.json').write_text(json.dumps(GENERIC_PRIOR_PROVENANCE,indent=2,ensure_ascii=False),encoding='utf-8')
print('Generic raw-train prior ready:',GENERIC_PRIOR.shape,'| test rows 0 | distribution',dict(zip(OBJECT_CLASS_ORDER,np.bincount(GENERIC_PRIOR.argmax(1),minlength=8).tolist())))


### 4d. Fit the generic semantic assignment head on train only

A small head is trained against the generated soft object/material prior. The graph, scaler-free DINOv2 representation, held-out groups, flip consistency, and neighbor diagnostics use train IDs only. This creates the generic source prior required by later semantic routing stages.


In [ ]:
from sklearn.preprocessing import normalize as sklearn_normalize
# Stage6 calibrated material/object assignment head, refit on raw-train-derived evidence.
import copy, time
import torch.nn.functional as F
from threadpoolctl import threadpool_limits

GENERIC_HEAD_DIR=WORK_ROOT/'labels'/'generic_assignment_head'
GENERIC_HEAD_DIR.mkdir(parents=True,exist_ok=True)
_replay_id_to_pos={sid:i for i,sid in enumerate(sem_ids)}
_fit_ids=np.asarray(coarse_fit_ids,dtype=str);_held_ids=np.asarray(coarse_heldout_ids,dtype=str)
assert len(_fit_ids)>0 and len(_held_ids)>0 and set(_fit_ids).isdisjoint(_held_ids)
_fit_idx=np.asarray([_replay_id_to_pos[k] for k in _fit_ids],dtype=np.int64)
_held_idx=np.asarray([_replay_id_to_pos[k] for k in _held_ids],dtype=np.int64)
assert set(coarse_fit_group_ids).isdisjoint(coarse_heldout_group_ids)
assert len(_fit_idx)==len(coarse_fit_group_ids) and len(_held_idx)==len(coarse_heldout_group_ids)
_dino_by_id={norm_id(k.split(':',1)[1]):v for k,v in zip(train_keys,dino2_train_features)}
_x_all=sklearn_normalize(np.stack([_dino_by_id[k] for k in sem_ids]).astype(np.float32)).astype(np.float32)
_flip_pos_by_id={norm_id(k.split(':',1)[1]):i for i,k in enumerate(flip_keys)}
_flip_fit_local=np.asarray([j for j,k in enumerate(_fit_ids) if k in _flip_pos_by_id],dtype=np.int64)
_flip_val_local=np.asarray([j for j,k in enumerate(_held_ids) if k in _flip_pos_by_id],dtype=np.int64)
_flip_fit_bank=np.asarray([_flip_pos_by_id[_fit_ids[j]] for j in _flip_fit_local],dtype=np.int64)
_flip_val_bank=np.asarray([_flip_pos_by_id[_held_ids[j]] for j in _flip_val_local],dtype=np.int64)
assert len(_flip_fit_local)>0 and len(_flip_val_local)>0
_x_flip_fit=torch.as_tensor(dino2_flip_features[_flip_fit_bank],device=DEVICE,dtype=torch.float32)
_x_flip_val=torch.as_tensor(dino2_flip_features[_flip_val_bank],device=DEVICE,dtype=torch.float32)
_prior=GENERIC_PRIOR.astype(np.float32)
assert _prior.shape==(len(sem_ids),8) and np.isfinite(_prior).all()
with threadpool_limits(limits=4):
    _fit_sim=_x_all[_fit_idx]@_x_all[_fit_idx].T
    np.fill_diagonal(_fit_sim,-np.inf)
    _nn=np.argsort(-_fit_sim,axis=1,kind='stable')[:,:10]
    _neighbor_logits=_fit_sim[np.arange(len(_fit_idx))[:,None],_nn]/.07
    _neighbor_logits-=_neighbor_logits.max(1,keepdims=True)
    _edge=np.exp(_neighbor_logits);_edge/=_edge.sum(1,keepdims=True)
    _val_sim=_x_all[_held_idx]@_x_all[_fit_idx].T
    _near_val=np.argsort(-_val_sim,axis=1,kind='stable')[:,:10]
_prior_fit=_prior[_fit_idx];_anchor=_prior_fit.max(1)>=.75
_anchor&=( _prior_fit.argmax(1)[:,None]==_prior_fit.argmax(1)[_nn]).mean(1)>=.7
_targets_by_alpha={}
for _alpha in [.4,.7,.9]:
    _target=_prior_fit.copy()
    for _ in range(60):
        _target=(1-_alpha)*_prior_fit+_alpha*(_target[_nn]*_edge[:,:,None]).sum(1)
        _target[_anchor]=_prior_fit[_anchor]
        _target/=np.maximum(_target.sum(1,keepdims=True),1e-8)
    _targets_by_alpha[_alpha]=_target.astype(np.float32)

def _make_assignment_head(kind,dimension=384):
    return (torch.nn.Linear(dimension,8) if kind=='linear' else
            torch.nn.Sequential(torch.nn.Linear(dimension,128),torch.nn.GELU(),torch.nn.Linear(128,8)))

def _soft_ce(logits,target):
    return -(target*F.log_softmax(logits,dim=1)).sum(1).mean()

def _js_divergence(a,b):
    mean=(a+b)*.5;eps=1e-8
    return .5*((a*(a.clamp_min(eps).log()-mean.clamp_min(eps).log())).sum(1).mean()+
               (b*(b.clamp_min(eps).log()-mean.clamp_min(eps).log())).sum(1).mean())

_x_fit=torch.as_tensor(_x_all[_fit_idx],device=DEVICE,dtype=torch.float32)
_x_val=torch.as_tensor(_x_all[_held_idx],device=DEVICE,dtype=torch.float32)
_flip_fit_local_t=torch.as_tensor(_flip_fit_local,device=DEVICE,dtype=torch.long)
_flip_val_local_t=torch.as_tensor(_flip_val_local,device=DEVICE,dtype=torch.long)
_history=[];_best=None;_started=time.time()
for _alpha,_soft_target in _targets_by_alpha.items():
    _soft=torch.as_tensor(_soft_target,device=DEVICE,dtype=torch.float32)
    for _kind in ['linear','mlp']:
        torch.manual_seed(20261106)
        _head=_make_assignment_head(_kind).to(DEVICE)
        _optimizer=torch.optim.Adam(_head.parameters(),lr=.01,weight_decay=1e-4)
        for _epoch in range(1,161):
            _head.train();_logits=_head(_x_fit);_prob=_logits.softmax(1)
            _loss=_soft_ce(_logits,_soft)
            _alt=_head(_x_flip_fit).softmax(1);_orig=_prob[_flip_fit_local_t]
            _loss=_loss+.2*_js_divergence(_orig,_alt)
            if not torch.isfinite(_loss):raise RuntimeError('Non-finite generic assignment-head loss.')
            _optimizer.zero_grad(set_to_none=True);_loss.backward();_optimizer.step()
            if _epoch%10==0:
                _head.eval()
                with torch.inference_mode():
                    _fit_pred=_head(_x_fit).argmax(1).cpu().numpy()
                    _val_pred=_head(_x_val).argmax(1).cpu().numpy()
                    _val_flip_pred=_head(_x_flip_val).argmax(1).cpu().numpy()
                _paired_val=_val_pred[_flip_val_local]
                _flip_stability=float(np.mean(_paired_val==_val_flip_pred))
                _teacher_agreement=float(np.mean(_val_pred==_prior[_held_idx].argmax(1)))
                _neighbor_agreement=float(np.mean(_val_pred[:,None]==_fit_pred[_near_val]))
                _counts=np.bincount(_val_pred,minlength=8)
                _score=.35*_flip_stability+.35*_teacher_agreement+.30*_neighbor_agreement
                _row={'alpha':float(_alpha),'head':_kind,'epoch':_epoch,'flip_stability':_flip_stability,
                      'teacher_agreement':_teacher_agreement,'neighbor_agreement':_neighbor_agreement,'score':_score,
                      'occupied_classes':int((_counts>0).sum()),'minimum_class_count':int(_counts.min()),
                      'fit_rows':len(_fit_idx),'heldout_rows':len(_held_idx)}
                _history.append(_row)
                if _row['occupied_classes']==8 and _row['minimum_class_count']>=5 and (_best is None or _score>_best['score']+1e-7):
                    _best={**_row,'state':copy.deepcopy(_head.state_dict()),'alpha':_alpha,'kind':_kind}
        print('Stage6 assignment candidate',_kind,'alpha',_alpha,'final epoch',_epoch,flush=True)
if _best is None:
    pd.DataFrame(_history).to_csv(GENERIC_HEAD_DIR/'development_metrics.csv',index=False)
    raise RuntimeError('No Stage6 head passed train-only held-out occupancy checks; downstream semantic routing remains closed.')
_head=_make_assignment_head(_best['kind']).to(DEVICE);_head.load_state_dict(_best['state']);_head.eval()
with torch.inference_mode():
    generic_head_prob=_head(torch.as_tensor(_x_all,device=DEVICE,dtype=torch.float32)).softmax(1).float().cpu().numpy()
    generic_head_flip_prob=_head(torch.as_tensor(dino2_flip_features,device=DEVICE,dtype=torch.float32)).softmax(1).float().cpu().numpy()
assert generic_head_prob.shape==(len(sem_ids),8) and generic_head_flip_prob.shape==(len(sem_flip_ids),8)
GENERIC_HEAD_DIR.mkdir(parents=True,exist_ok=True)
torch.save({'state_dict':{k:v.detach().cpu() for k,v in _head.state_dict().items()},'kind':_best['kind'],
    'feature_dimension':384,'class_order':OBJECT_CLASS_ORDER,'alpha':_best['alpha'],'selected_epoch':_best['epoch'],
    'fit_ids':_fit_ids.tolist(),'heldout_ids':_held_ids.tolist(),'test_used_for_fit':False},GENERIC_HEAD_DIR/'stage6_generic_head.pt')
np.savez_compressed(GENERIC_HEAD_DIR/'stage6_generic_probabilities.npz',image_ids=sem_ids,flip_ids=sem_flip_ids,
    source_sha256=sem_source_hash,flip_source_sha256=sem_flip_hash,probabilities=generic_head_prob,
    flip_probabilities=generic_head_flip_prob,class_order=np.asarray(OBJECT_CLASS_ORDER))
pd.DataFrame(_history).to_csv(GENERIC_HEAD_DIR/'development_metrics.csv',index=False)
GENERIC_HEAD_REPORT={'method':'train_only_DINOv2_graph_soft_targets_from_calibrated_ImageNet_object_plus_MINC_material_prior',
 'train_rows':len(sem_ids),'test_rows_used':0,'fit_rows':len(_fit_idx),'heldout_rows':len(_held_idx),
 'fit_group_count':len(set(coarse_fit_group_ids)),'heldout_group_count':len(set(coarse_heldout_group_ids)),
 'selected':{k:v for k,v in _best.items() if k!='state'},'source_prior':'not an official label; computed from raw train pixels and pretrained non-VLM model weights',
 'source_fingerprint':sem_content_fingerprint,'class_order':OBJECT_CLASS_ORDER,'archive_assignment_labels_read':False,
 'test_features_or_targets_read':False,'active_assignment_parity_verified':False,'seconds':time.time()-_started}
(GENERIC_HEAD_DIR/'stage6_generic_head_report.json').write_text(json.dumps(GENERIC_HEAD_REPORT,indent=2,ensure_ascii=False),encoding='utf-8')
GENERIC_PRIOR=generic_head_prob;GENERIC_PRIOR_FLIP=generic_head_flip_prob
print('Train-only Stage6 generic assignment head ready:',GENERIC_HEAD_REPORT['selected'])


## 4. Hybrid CNN + HOG + warna untuk kandidat label train

Sesuai materi hybrid (slide 15): bandingkan CNN-only, HOG-only, warna-only, HOG+warna, dan fusi CNN+HOG+warna dengan RBF-SVM. Standardisasi dan normalisasi L2 dilakukan per blok di dalam setiap fold; bobot blok dan parameter SVM dipilih hanya di inner folds. Outer folds menahan source-like group seed; seluruh fitur dihitung dari foto train mentah.

Hanya 37 seed peserta menjadi target manusia (maksimum lima per kelas). Model membuat kandidat pseudo-label untuk foto train. OOF seed, konfigurasi, margin SVM yang tidak terkalibrasi, serta kandidat per ID disimpan terpisah. Hasil hybrid ini dicatat sebagai eksperimen; pipeline final memakai target Stage18 yang dibentuk secara train-only.

In [ ]:
# Hybrid train-label candidate from raw TRAIN images only; no test features or targets.
import gc, math, time
from collections import defaultdict
from scipy.fft import dctn
from skimage.feature import hog
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, make_scorer, confusion_matrix, precision_recall_fscore_support
from IPython.display import display
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torchvision import models
from torchvision.transforms.functional import to_tensor, normalize as tv_normalize

hybrid_wall_started=time.perf_counter()
HYBRID_DIR=WORK_ROOT/'features'/'hybrid_labeling'
for _artifact_dir in ['features','validation','labels','logs']:
    (WORK_ROOT/_artifact_dir).mkdir(parents=True,exist_ok=True)
HYBRID_DIR.mkdir(parents=True,exist_ok=True)
HYBRID_CLASSES=list(CLASSES)
seed_items=sorted([(sid,cat) for cat,ids in HUMAN_SEEDS.items() for sid in ids],key=lambda x:x[0])
seed_ids=np.array([sid for sid,_ in seed_items],dtype=str)
y_seed=np.array([HYBRID_CLASSES.index(cat) for _,cat in seed_items],dtype=np.int64)
assert len(seed_ids)==37 and len(set(seed_ids))==37
assert all(sum(cat==c for _,cat in seed_items)<=5 for c in HYBRID_CLASSES)
path_by_id={norm_id(p.stem):p for p in raw_train_paths}
assert set(seed_ids)<=set(path_by_id) and len(path_by_id)==1920
train_order=np.array(sorted(path_by_id),dtype=str)
ordered_paths=[path_by_id[sid] for sid in train_order]
source_hashes=np.array([sha256_file(p) for p in ordered_paths],dtype=str)
input_fingerprint=hashlib.sha256(('|'.join(train_order)+'\n'+'|'.join(source_hashes)).encode()).hexdigest()

# EfficientNet-B0 is used only as a frozen, image-only feature extractor.
CNN_WEIGHTS=models.EfficientNet_B0_Weights.IMAGENET1K_V1
cnn_weight_path=Path(torch.hub.get_dir())/'checkpoints'/CNN_WEIGHTS.url.rsplit('/',1)[-1]
try:
    cnn=models.efficientnet_b0(weights=CNN_WEIGHTS).to(DEVICE).eval()
except Exception as e:
    raise RuntimeError('EfficientNet-B0 pretrained weights are required for the hybrid train-label experiment.') from e
cnn_weight_sha=sha256_file(cnn_weight_path) if cnn_weight_path.exists() else None
cnn_mean=(.485,.456,.406);cnn_std=(.229,.224,.225)

def _hybrid_canvas(path):
    return processed_canvas(rgb_source(path))

def _cnn_tensor(path):
    return tv_normalize(to_tensor(_hybrid_canvas(path)),cnn_mean,cnn_std)

class _RawTrainBatch(torch.utils.data.Dataset):
    def __init__(self,ids): self.ids=list(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self,i): return _cnn_tensor(path_by_id[self.ids[i]])

feature_cache=HYBRID_DIR/'train_cnn_hog_color.npz'
feature_prepare_started=time.perf_counter()
feature_identity=hashlib.sha256(json.dumps({'raw_train':input_fingerprint,'weights':str(CNN_WEIGHTS),'weight_sha256':cnn_weight_sha,
 'canvas':'RGB letterbox224 content202 pad127','hog':'grayscale64 orientations9 cell8 block2','rgb_hist':'256 bins/channel; normalized counts'},sort_keys=True).encode()).hexdigest()
if feature_cache.exists():
    with np.load(feature_cache,allow_pickle=False) as z:
        assert str(z['identity'].item())==feature_identity and np.array_equal(z['image_ids'],train_order)
        assert np.array_equal(z['source_sha256'],source_hashes), 'Hybrid feature cache source hashes do not match raw train.'
        cnn_features=z['cnn'].astype('float32');hog_features=z['hog'].astype('float32');color_features=z['color'].astype('float32')
    print('Loaded verified train-only hybrid feature cache.')
else:
    loader=DataLoader(_RawTrainBatch(train_order),batch_size=48,shuffle=False,num_workers=2 if os.name!='nt' else 0,pin_memory=DEVICE.type=='cuda')
    cnn_parts=[]
    with torch.inference_mode(),torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
        for batch in loader:
            batch=batch.to(DEVICE,non_blocking=True)
            emb=cnn.avgpool(cnn.features(batch)).flatten(1)
            cnn_parts.append(emb.float().cpu().numpy())
    cnn_features=np.concatenate(cnn_parts).astype('float32')
    del loader,cnn_parts;gc.collect();torch.cuda.empty_cache()
    hog_rows=[];color_rows=[]
    for sid in train_order:
        path=path_by_id[sid];canvas=_hybrid_canvas(path)
        gray=np.asarray(canvas.convert('L').resize((64,64),Image.Resampling.BICUBIC),dtype=np.float32)/255.
        hog_rows.append(hog(gray,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2),block_norm='L2-Hys',feature_vector=True).astype('float32'))
        # Histogram uses original RGB pixels so neutral letterbox padding cannot dominate color.
        rgb=np.asarray(rgb_source(path),dtype=np.uint8)
        color_rows.append(np.concatenate([np.histogram(rgb[:,:,ch],bins=256,range=(0,256))[0].astype('float32')/rgb.shape[0]/rgb.shape[1] for ch in range(3)]))
    hog_features=np.stack(hog_rows).astype('float32');color_features=np.stack(color_rows).astype('float32')
    assert cnn_features.shape==(1920,1280) and hog_features.shape==(1920,1764) and color_features.shape==(1920,768)
    np.savez_compressed(feature_cache,image_ids=train_order,cnn=cnn_features,hog=hog_features,color=color_features,identity=np.array(feature_identity),source_sha256=source_hashes)
    del hog_rows,color_rows;gc.collect()

feature_prepare_seconds=time.perf_counter()-feature_prepare_started
feature_index={sid:i for i,sid in enumerate(train_order)}
seed_pos=np.array([feature_index[sid] for sid in seed_ids],dtype=int)
X_all={'cnn_only':cnn_features,'hog_only':hog_features,'color_only':color_features,
       'hog_color':np.concatenate([hog_features,color_features],axis=1),
       'hybrid_cnn_hog_color':np.concatenate([cnn_features,hog_features,color_features],axis=1)}
BLOCK_DIMS={'cnn_only':(cnn_features.shape[1],),'hog_only':(hog_features.shape[1],),'color_only':(color_features.shape[1],),
            'hog_color':(hog_features.shape[1],color_features.shape[1]),
            'hybrid_cnn_hog_color':(cnn_features.shape[1],hog_features.shape[1],color_features.shape[1])}

# Group exact/near-identical reviewed seed images together for outer CV.
seed_profile=manifest[(manifest.split=='train')&manifest.image_id.isin(seed_ids)].set_index('image_id')
assert set(seed_profile.index)==set(seed_ids)
parent=list(range(len(seed_ids)))
def find(a):
    while parent[a]!=a: parent[a]=parent[parent[a]];a=parent[a]
    return a
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[rb]=ra
for attr in ['file_sha256','pixel_sha256']:
    buckets=defaultdict(list)
    for i,sid in enumerate(seed_ids): buckets[str(seed_profile.loc[sid,attr])].append(i)
    for ids in buckets.values():
        for j in ids[1:]:union(ids[0],j)
seed_gray=[]
for sid in seed_ids:
    with Image.open(path_by_id[sid]) as src: im=ImageOps.exif_transpose(src).convert('L').resize((32,32),Image.Resampling.LANCZOS)
    low=dctn(np.asarray(im,dtype=np.float32),type=2,norm='ortho')[:8,:8];vals=low.reshape(-1)[1:];bits=np.r_[False,(low.reshape(-1)[1:]>np.median(vals))]
    seed_gray.append(bits)
seed_hash=np.stack(seed_gray)
for i in range(len(seed_ids)):
    for j in range(i):
        if np.count_nonzero(seed_hash[i]!=seed_hash[j])<=4:union(i,j)
seed_groups=np.array([find(i) for i in range(len(seed_ids))])
assert len(np.unique(seed_groups))>=3

OUTER_SPLITS=len(np.unique(seed_groups))

def _min_class_group_count(y,groups):
    y=np.asarray(y);groups=np.asarray(groups)
    return min(len(np.unique(groups[y==cls])) for cls in np.unique(y))

def _valid_group_splitter(y,groups,n_splits,seed_start,max_tries=500):
    # Require all classes on both sides; lower fold count is handled by the caller.
    y=np.asarray(y);groups=np.asarray(groups)
    if n_splits<2 or n_splits>_min_class_group_count(y,groups):
        raise ValueError('Grouped fold count exceeds the number of independent groups in a class.')
    for state in range(seed_start,seed_start+max_tries):
        splitter=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=state)
        folds=list(splitter.split(np.zeros((len(y),1)),y,groups))
        if all(len(np.unique(y[tr]))==len(HYBRID_CLASSES) for tr,va in folds):
            return splitter,state
    raise RuntimeError(f'Could not build {n_splits} grouped folds with all classes represented in every fit partition.')

outer_cv=LeaveOneGroupOut()
outer_split_seed=None
outer_fold_list=list(outer_cv.split(np.zeros((len(seed_ids),1)),y_seed,seed_groups))
assert all(len(np.unique(y_seed[fit]))==len(HYBRID_CLASSES) for fit,_ in outer_fold_list)
PARAM_GRID={'svc__C':[.1,1.,10.],'svc__gamma':['scale',.001,.01,.1]}
BLOCK_WEIGHT_GRID={'cnn_only':[(1.,)],'hog_only':[(1.,)],'color_only':[(1.,)],
 'hog_color':[(.5,.5),(.67,.33),(.33,.67)],
 'hybrid_cnn_hog_color':[(1/3,1/3,1/3),(.5,.25,.25),(.25,.5,.25),(.25,.25,.5)]}
def _inner_macro_f1_present(estimator,X,y_true):
    y_true=np.asarray(y_true); y_pred=estimator.predict(X)
    return f1_score(y_true,y_pred,labels=np.unique(y_true),average='macro',zero_division=0)
INNER_SCORER=_inner_macro_f1_present
from sklearn.base import BaseEstimator,TransformerMixin
class BlockBalancedScaler(BaseEstimator,TransformerMixin):
    """Fold-fitted standardization followed by per-image L2 normalization per feature block."""
    def __init__(self,block_dims,block_weights=None,eps=1e-12):
        self.block_dims=block_dims;self.block_weights=block_weights;self.eps=eps
    def fit(self,X,y=None):
        self.offsets_=np.cumsum((0,)+tuple(self.block_dims));self.means_=[];self.scales_=[]
        for a,b in zip(self.offsets_[:-1],self.offsets_[1:]):
            z=np.asarray(X[:,a:b],dtype=np.float64);mu=z.mean(axis=0);sd=z.std(axis=0);sd[sd<self.eps]=1.
            self.means_.append(mu);self.scales_.append(sd)
        weights=np.asarray(self.block_weights if self.block_weights is not None else np.ones(len(self.block_dims))/len(self.block_dims),dtype=np.float64)
        if len(weights)!=len(self.block_dims) or np.any(weights<0) or not np.isfinite(weights).all() or weights.sum()<=0:
            raise ValueError('Invalid per-block weights.')
        self.weights_=weights/weights.sum();return self
    def transform(self,X):
        X=np.asarray(X,dtype=np.float64);parts=[]
        for i,(a,b) in enumerate(zip(self.offsets_[:-1],self.offsets_[1:])):
            z=(X[:,a:b]-self.means_[i])/self.scales_[i];norm=np.linalg.norm(z,axis=1,keepdims=True)
            z=z/np.maximum(norm,self.eps);parts.append(z*np.sqrt(self.weights_[i]))
        return np.concatenate(parts,axis=1).astype('float32')
def _pipeline(name,weights=None):
    return Pipeline([('scale',BlockBalancedScaler(BLOCK_DIMS[name],weights)),('svc',SVC(kernel='rbf',class_weight='balanced',decision_function_shape='ovr'))])
def _grid_for(name):
    return {'svc__C':PARAM_GRID['svc__C'],'svc__gamma':PARAM_GRID['svc__gamma'],'scale__block_weights':BLOCK_WEIGHT_GRID[name]}

oof_pred={name:np.full(len(seed_ids),-1,dtype=np.int64) for name in X_all}
oof_fit_started=time.perf_counter()
oof_fold=np.full(len(seed_ids),-1,dtype=np.int64);outer_scores=[]
for fold,(fit_local,val_local) in enumerate(outer_fold_list):
    assert not set(seed_groups[fit_local])&set(seed_groups[val_local])
    assert set(np.unique(y_seed[fit_local]))==set(range(len(HYBRID_CLASSES))),f'Outer fold {fold}: a class is absent from fit seeds.'
    inner_n=min(3,_min_class_group_count(y_seed[fit_local],seed_groups[fit_local]))
    inner_cv,inner_split_seed=_valid_group_splitter(y_seed[fit_local],seed_groups[fit_local],inner_n,SEED+fold+1)
    for name,X in X_all.items():
        search=GridSearchCV(_pipeline(name),_grid_for(name),scoring=INNER_SCORER,cv=inner_cv,n_jobs=1,refit=True,error_score='raise')
        search.fit(X[seed_pos[fit_local]],y_seed[fit_local],groups=seed_groups[fit_local])
        oof_pred[name][val_local]=search.predict(X[seed_pos[val_local]])
        oof_fold[val_local]=fold
        outer_scores.append({'fold':fold,'method':name,'inner_best_params':json.dumps(search.best_params_,sort_keys=True),
          'inner_best_macro_f1':float(search.best_score_),'validation_n':len(val_local),'inner_split_seed':inner_split_seed})
assert (oof_fold>=0).all() and all((p>=0).all() for p in oof_pred.values())
oof_fit_seconds=time.perf_counter()-oof_fit_started

summary=[]
oof_classwise=[]
oof_confusions=[]
for name,pred in oof_pred.items():
    summary.append({'method':name,'seed_oof_accuracy':float(accuracy_score(y_seed,pred)),
      'seed_oof_balanced_accuracy':float(balanced_accuracy_score(y_seed,pred)),
      'seed_oof_macro_f1':float(f1_score(y_seed,pred,labels=np.arange(len(HYBRID_CLASSES)),average='macro',zero_division=0)),
      'seed_n':len(y_seed),'outer_group_folds':OUTER_SPLITS,'outer_split_seed':outer_split_seed,'metric_scope':'nested grouped OOF on 37 human seeds; not independent after method development'})
    prec,rec,f1c,support=precision_recall_fscore_support(y_seed,pred,labels=np.arange(len(HYBRID_CLASSES)),zero_division=0)
    for k,cls in enumerate(HYBRID_CLASSES):
        oof_classwise.append({'method':name,'class':cls,'precision':float(prec[k]),'recall':float(rec[k]),'f1':float(f1c[k]),'support':int(support[k])})
    cm=confusion_matrix(y_seed,pred,labels=np.arange(len(HYBRID_CLASSES)))
    for true_i,true_cls in enumerate(HYBRID_CLASSES):
        for pred_i,pred_cls in enumerate(HYBRID_CLASSES):
            oof_confusions.append({'method':name,'true_class':true_cls,'predicted_class':pred_cls,'count':int(cm[true_i,pred_i])})
pd.DataFrame(summary).to_csv(WORK_ROOT/'validation/hybrid_label_seed_oof_summary.csv',index=False)
pd.DataFrame(oof_classwise).to_csv(WORK_ROOT/'validation/hybrid_label_seed_oof_classwise.csv',index=False)
pd.DataFrame(oof_confusions).to_csv(WORK_ROOT/'validation/hybrid_label_seed_oof_confusion.csv',index=False)
pd.DataFrame(outer_scores).to_csv(WORK_ROOT/'validation/hybrid_label_inner_selection.csv',index=False)
oof_table=pd.DataFrame({'image_id':seed_ids,'human_label':[HYBRID_CLASSES[i] for i in y_seed],'source_group':seed_groups,'outer_fold':oof_fold})
for name,pred in oof_pred.items():oof_table[name+'_oof']=[HYBRID_CLASSES[i] for i in pred]
oof_table.to_csv(WORK_ROOT/'validation/hybrid_label_seed_oof.csv',index=False)

# Refit each frozen recipe on all 37 human seeds; write full-train label candidates only.
candidate_fit_started=time.perf_counter()
candidate_table=pd.DataFrame({'image_id':train_order,'source_sha256':source_hashes,'is_human_seed':[sid in set(seed_ids) for sid in train_order]})
for name,X in X_all.items():
    final_n=min(3,_min_class_group_count(y_seed,seed_groups))
    final_cv,final_split_seed=_valid_group_splitter(y_seed,seed_groups,final_n,SEED+101)
    final_search=GridSearchCV(_pipeline(name),_grid_for(name),scoring=INNER_SCORER,cv=final_cv,n_jobs=1,refit=True,error_score='raise')
    final_search.fit(X[seed_pos],y_seed,groups=seed_groups)
    pred=final_search.predict(X);dec=final_search.decision_function(X)
    if dec.ndim==1: dec=np.column_stack([-dec,dec])
    ordered=np.sort(dec,axis=1);margin=ordered[:,-1]-ordered[:,-2]
    candidate_table[name+'_candidate']=[HYBRID_CLASSES[i] for i in pred]
    candidate_table[name+'_svm_margin_uncalibrated']=margin.astype('float32')
    candidate_table[name+'_selected_params']=json.dumps(final_search.best_params_,sort_keys=True)
    candidate_table[name+'_split_seed']=final_split_seed
# Keep participant-confirmed seed labels as the fixed seed assignments in the hybrid output.
human_map={sid:cat for sid,cat in seed_items}
candidate_table['hybrid_assigned_candidate']=[human_map.get(sid,candidate_table.loc[i,'hybrid_cnn_hog_color_candidate']) for i,sid in enumerate(train_order)]
candidate_table['assignment_source']=['participant_seed' if sid in human_map else 'hybrid_cnn_hog_color_svm_pseudolabel' for sid in train_order]
candidate_table.to_csv(WORK_ROOT/'labels/hybrid_train_label_candidates.csv',index=False)
candidate_fit_seconds=time.perf_counter()-candidate_fit_started
hybrid_total_seconds=time.perf_counter()-hybrid_wall_started
provenance={'method':'frozen_EfficientNetB0_embedding_plus_HOG_plus_RGB_histogram_RBF_SVM',
 'input':'raw organizer train images only','train_images':len(train_order),'test_images_processed':0,
 'human_seeds':len(seed_ids),'seed_class_counts':pd.Series([c for _,c in seed_items]).value_counts().reindex(HYBRID_CLASSES,fill_value=0).to_dict(),
 'cnn_weights':str(CNN_WEIGHTS),'cnn_weight_sha256':cnn_weight_sha,'feature_cache_identity':feature_identity,
 'feature_dimensions':{'cnn_embedding':int(cnn_features.shape[1]),'hog':int(hog_features.shape[1]),'rgb_histogram':int(color_features.shape[1])},
 'feature_ablations':list(X_all),'scaler':'per-block StandardScaler fit within each grouped fold, then per-image L2 normalization and inner-fold block-weight selection; refit on all 37 reviewed train seeds for candidate labels',
 'svm':'RBF; inner grouped search C=[0.1,1,10], gamma=[scale,0.001,0.01,0.1]',
 'source_like_groups':'union of identical file SHA, identical decoded RGB SHA, and seed pHash distance <=4',
 'seed_oof_summary':summary,'label_candidate_rows':len(candidate_table),
 'runtime_seconds':{'feature_prepare_including_verified_cache':feature_prepare_seconds,'nested_grouped_oof':oof_fit_seconds,'full_seed_refit_and_candidate_inference':candidate_fit_seconds,'cell_total':hybrid_total_seconds},
 'evaluation_artifacts':['hybrid_label_seed_oof_classwise.csv','hybrid_label_seed_oof_confusion.csv'],
 'candidate_used_as_active_assignment':False,'active_assignment_parity_verified':False,'test_used_for_fit':False}
(WORK_ROOT/'logs/hybrid_label_feature_provenance.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame(summary).sort_values('seed_oof_macro_f1',ascending=False))
display(pd.DataFrame(oof_classwise))
print('Hybrid runtime (s):',json.dumps(provenance['runtime_seconds'],sort_keys=True))
print('Hybrid label candidates:',WORK_ROOT/'labels/hybrid_train_label_candidates.csv')
print('These are candidates only; no processed class folders were written.')

### 4.5 Rebuild semantic anchors and a train-only assignment head

This cell reconstructs the first semantic-anchor stage from raw-train-derived CNN priors and the anonymous Stage4 clusters. Cluster-to-class matching is fit on Stage4 fit IDs only; a local head is trained on the resulting clean fit anchors and selected on the held-out train groups. It writes a candidate baseline for the later replay stages, not the active label folders. Test images and archived assignment files are not read.


In [ ]:
from sklearn.preprocessing import normalize as sklearn_normalize
# Rebuild Stage8-style semantic anchor mapping and student from raw-train-derived banks only.
import copy, time
import torch.nn.functional as F
from threadpoolctl import threadpool_limits
from scipy.optimize import linear_sum_assignment

STAGE8_DIR=WORK_ROOT/'labels'/'stage8_semantic_anchor'
STAGE8_DIR.mkdir(parents=True,exist_ok=True)
_stage8_order=list(SEMANTIC_CLASSES)
_stage8_n=len(sem_ids)
assert _stage8_n==1920 and GENERIC_PRIOR.shape==(_stage8_n,8)
assert resnet_task_prior.shape==(_stage8_n,8) and efficientnet_task_prior.shape==(_stage8_n,8)
# Align the anonymous Stage4 outputs and DINO descriptors to the semantic bank by numeric train ID.
_stage8_coarse_pos={str(sid):i for i,sid in enumerate(ordered_train_ids)}
_stage8_coarse=np.stack([coarse_prob[_stage8_coarse_pos[sid]] for sid in sem_ids]).astype(np.float32)
_stage8_id_to_pos={sid:i for i,sid in enumerate(sem_ids)}
_stage8_fit_ids=np.asarray(coarse_fit_ids,dtype=str);_stage8_held_ids=np.asarray(coarse_heldout_ids,dtype=str)
_stage8_fit=np.asarray([_stage8_id_to_pos[sid] for sid in _stage8_fit_ids],dtype=np.int64)
_stage8_held=np.asarray([_stage8_id_to_pos[sid] for sid in _stage8_held_ids],dtype=np.int64)
assert set(_stage8_fit_ids).isdisjoint(_stage8_held_ids)
assert set(coarse_fit_group_ids).isdisjoint(coarse_heldout_group_ids)
assert len(_stage8_fit)==len(coarse_fit_group_ids) and len(_stage8_held)==len(coarse_heldout_group_ids)

# Fit one semantic name per anonymous group from fit-only source likelihoods.
_stage8_priors=np.stack([GENERIC_PRIOR,resnet_task_prior]).astype(np.float32)
_stage8_scores=np.zeros((8,8),dtype=np.float64)
for _g in range(8):
    _members=_stage8_fit[_stage8_coarse[_stage8_fit].argmax(1)==_g]
    assert len(_members)>0, f'Empty Stage4 fit cluster {_g}'
    _stage8_scores[_g]=np.max(np.stack([q[_members].mean(0) for q in _stage8_priors]),axis=0)
_stage8_left,_stage8_right=linear_sum_assignment(-_stage8_scores)
_stage8_mapping=dict(zip(_stage8_left.tolist(),_stage8_right.tolist()))
_stage8_mapped=np.asarray([_stage8_mapping[int(g)] for g in _stage8_coarse.argmax(1)],dtype=np.int64)
_stage8_support=np.max(np.stack([q[np.arange(_stage8_n),_stage8_mapped] for q in _stage8_priors]),axis=0)
_stage8_agrees=np.any(np.stack([q.argmax(1)==_stage8_mapped for q in _stage8_priors]),axis=0)
_stage8_clean=(_stage8_support>=.65)&_stage8_agrees&(_stage8_coarse.max(1)>=.8)
_stage8_labels=_stage8_mapped.copy()
_stage8_strong=(GENERIC_PRIOR.argmax(1)==resnet_task_prior.argmax(1))&(GENERIC_PRIOR.max(1)>=.75)&(resnet_task_prior.max(1)>=.9)
_stage8_source_names=SEMANTIC_SOURCE_CLASSES['resnet_waste']
_stage8_battery=np.asarray(resnet_waste['probabilities'])[:,_stage8_source_names.index('battery')]
_stage8_strong&=(_stage8_labels!=_stage8_order.index('electronic'))|(_stage8_battery>=.8)
_stage8_labels[_stage8_strong]=resnet_task_prior.argmax(1)[_stage8_strong]
_stage8_clean|=_stage8_strong
_stage8_seed_local=np.flatnonzero(_stage8_clean[_stage8_fit])
_stage8_val_mask=_stage8_clean[_stage8_held]
_stage8_fit_counts=np.bincount(_stage8_labels[_stage8_fit[_stage8_seed_local]],minlength=8)
_stage8_val_counts=np.bincount(_stage8_labels[_stage8_held][_stage8_val_mask],minlength=8)
assert (_stage8_fit_counts>=3).all(), _stage8_fit_counts
assert (_stage8_val_counts>0).all(), _stage8_val_counts

_stage8_dino_by_id={norm_id(k.split(':',1)[1]):v for k,v in zip(train_keys,dino2_train_features)}
_stage8_dino=sklearn_normalize(np.stack([_stage8_dino_by_id[k] for k in sem_ids]).astype(np.float32)).astype(np.float32)
_stage8_waste=sklearn_normalize(np.asarray(resnet_waste['features'],dtype=np.float32)).astype(np.float32)
_stage8_clean_id=lambda k:norm_id(str(k).split(':')[-1])
_stage8_flip_pos={_stage8_clean_id(k):i for i,k in enumerate(sem_flip_ids)}
_stage8_dino_flip_by_id={_stage8_clean_id(k):v for k,v in zip(flip_keys,dino2_flip_features)}
_stage8_flip_ids=np.asarray([_stage8_clean_id(k) for k in sem_flip_ids],dtype=str)
assert set(_stage8_flip_ids)==set(_stage8_dino_flip_by_id)
_stage8_waste_flip=sklearn_normalize(np.asarray(resnet_waste['flip_features'],dtype=np.float32)).astype(np.float32)
_stage8_dino_flip=sklearn_normalize(np.stack([_stage8_dino_flip_by_id[_stage8_clean_id(k)] for k in _stage8_flip_ids]).astype(np.float32)).astype(np.float32)
def _stage8_block(branch,flip=False):
    waste=_stage8_waste_flip if flip else _stage8_waste
    dino=_stage8_dino_flip if flip else _stage8_dino
    if branch=='waste': return waste
    if branch=='dino': return dino
    if branch=='hybrid': return (np.concatenate([waste,dino],axis=1)/np.float32(np.sqrt(2))).astype(np.float32)
    raise ValueError(branch)

_stage8_flip_fit_local=np.asarray([j for j,k in enumerate(_stage8_fit_ids) if _stage8_clean_id(k) in _stage8_flip_pos],dtype=np.int64)
_stage8_flip_val_local=np.asarray([j for j,k in enumerate(_stage8_held_ids) if _stage8_clean_id(k) in _stage8_flip_pos],dtype=np.int64)
_stage8_flip_fit_bank=np.asarray([_stage8_flip_pos[_stage8_clean_id(_stage8_fit_ids[j])] for j in _stage8_flip_fit_local],dtype=np.int64)
_stage8_flip_val_bank=np.asarray([_stage8_flip_pos[_stage8_clean_id(_stage8_held_ids[j])] for j in _stage8_flip_val_local],dtype=np.int64)
assert len(_stage8_flip_fit_local)>0 and len(_stage8_flip_val_local)>0
with threadpool_limits(limits=4):
    _stage8_geometry=_stage8_block('hybrid')
    _stage8_sim=_stage8_geometry[_stage8_fit]@_stage8_geometry[_stage8_fit].T
    np.fill_diagonal(_stage8_sim,-np.inf)
    _stage8_nn=np.argsort(-_stage8_sim,axis=1,kind='stable')[:,:10]
    _stage8_nv=np.argsort(-(_stage8_geometry[_stage8_held]@_stage8_geometry[_stage8_fit].T),axis=1,kind='stable')[:,:10]

def _stage8_make_head(kind,dimension):
    return torch.nn.Linear(dimension,8) if kind=='linear' else torch.nn.Sequential(torch.nn.Linear(dimension,128),torch.nn.GELU(),torch.nn.Linear(128,8))
def _stage8_js(a,b):
    m=(a+b)*.5;e=1e-8
    return .5*((a*(a.clamp_min(e).log()-m.clamp_min(e).log())).sum(1).mean()+(b*(b.clamp_min(e).log()-m.clamp_min(e).log())).sum(1).mean())
def _stage8_evaluate(head,branch):
    x=_stage8_block(branch);xf=_stage8_block(branch,True)
    head.eval()
    with torch.inference_mode():
        tr=head(torch.as_tensor(x[_stage8_fit],device=DEVICE)).argmax(1).cpu().numpy()
        val=head(torch.as_tensor(x[_stage8_held],device=DEVICE)).argmax(1).cpu().numpy()
        val_flip=head(torch.as_tensor(xf[_stage8_flip_val_bank],device=DEVICE)).argmax(1).cpu().numpy()
    _macro=float(np.mean([(val[_stage8_val_mask&(_stage8_labels[_stage8_held]==c)]==c).mean() for c in range(8)]))
    _flip=float((val[_stage8_flip_val_local]==val_flip).mean())
    _near=float((val[:,None]==tr[_stage8_nv]).mean())
    _counts=np.bincount(val,minlength=8)
    return {'score':.5*_macro+.25*_flip+.25*_near,'macro_anchor_agreement':_macro,'flip_agreement':_flip,
            'neighbor_agreement':_near,'occupied_classes':int((_counts>0).sum()),'minimum_class_count':int(_counts.min())}

_stage8_started=time.time();_stage8_hist=[];_stage8_best=None
_stage8_y=torch.as_tensor(_stage8_labels[_stage8_fit[_stage8_seed_local]],device=DEVICE,dtype=torch.long)
_stage8_w=torch.as_tensor((_stage8_seed_local.size/(8*_stage8_fit_counts)).astype(np.float32),device=DEVICE)
for _branch in ['waste','dino','hybrid']:
    _x=_stage8_block(_branch);_xf=_stage8_block(_branch,True)
    _xt=torch.as_tensor(_x[_stage8_fit],device=DEVICE,dtype=torch.float32)
    _xflipfit=torch.as_tensor(_xf[_stage8_flip_fit_bank],device=DEVICE,dtype=torch.float32)
    _xflipfit_local=torch.as_tensor(_stage8_flip_fit_local,device=DEVICE,dtype=torch.long)
    for _kind in ['linear','mlp']:
        torch.manual_seed(20261106)
        _head=_stage8_make_head(_kind,_x.shape[1]).to(DEVICE)
        _opt=torch.optim.Adam(_head.parameters(),lr=.003,weight_decay=1e-4)
        for _epoch in range(1,181):
            _head.train();_logits=_head(_xt);_prob=_logits.softmax(-1)
            _ce=F.cross_entropy(_logits[_stage8_seed_local],_stage8_y,weight=_stage8_w)
            _graph=-(_prob[:,None,:]*_prob[_stage8_nn]).sum(-1).clamp_min(1e-8).log().mean()
            _alt=_head(_xflipfit).softmax(-1);_orig=_prob[_xflipfit_local]
            _loss=_ce+.03*_graph+.2*_stage8_js(_orig,_alt)
            if not torch.isfinite(_loss):raise RuntimeError('Non-finite Stage8 loss.')
            _opt.zero_grad(set_to_none=True);_loss.backward();_opt.step()
            if _epoch%10==0:
                _metric=_stage8_evaluate(_head,_branch)
                _row={'branch':_branch,'head':_kind,'epoch':_epoch,**_metric};_stage8_hist.append(_row)
                if _metric['occupied_classes']==8 and _metric['minimum_class_count']>=5 and (_stage8_best is None or _metric['score']>_stage8_best['score']+1e-7):
                    _stage8_best={**_row,'state':copy.deepcopy(_head.state_dict())}
        print('Stage8 fit-only head',_branch,_kind,'complete',flush=True)
if _stage8_best is None:
    pd.DataFrame(_stage8_hist).to_csv(STAGE8_DIR/'development_metrics.csv',index=False)
    raise RuntimeError('Stage8 has no candidate satisfying held-out class-occupancy guard; active label replay stays closed.')
_stage8_state=_stage8_best.pop('state')
_stage8_final=_stage8_make_head(_stage8_best['head'],_stage8_block(_stage8_best['branch']).shape[1]).to(DEVICE)
_stage8_final.load_state_dict(_stage8_state);_stage8_final.eval()
with torch.inference_mode():
    _stage8_prob=_stage8_final(torch.as_tensor(_stage8_block(_stage8_best['branch']),device=DEVICE,dtype=torch.float32)).softmax(-1).float().cpu().numpy()
_stage8_rank=np.sort(_stage8_prob,axis=1)
_stage8_report={'method':'raw_train_fit_only_anonymous_group_semantic_anchor_student','train_rows':_stage8_n,'fit_rows':len(_stage8_fit),'heldout_rows':len(_stage8_held),
    'fit_anchor_counts':dict(zip(_stage8_order,_stage8_fit_counts.tolist())),'heldout_anchor_counts':dict(zip(_stage8_order,_stage8_val_counts.tolist())),
    'mapping':{str(k):_stage8_order[v] for k,v in _stage8_mapping.items()},'selected':_stage8_best,'class_order':_stage8_order,
    'test_images_processed':0,'test_used_for_fit':False,'archive_assignments_read':False,'active_assignment_parity_verified':False,
    'seconds':time.time()-_stage8_started,'source_fingerprint':sem_content_fingerprint}
torch.save({'state_dict':{k:v.detach().cpu() for k,v in _stage8_final.state_dict().items()},'branch':_stage8_best['branch'],'head':_stage8_best['head'],
    'class_order':_stage8_order,'fit_ids':_stage8_fit_ids.tolist(),'heldout_ids':_stage8_held_ids.tolist(),'fit_group_ids':coarse_fit_group_ids,'heldout_group_ids':coarse_heldout_group_ids,
    'selection':{k:v for k,v in _stage8_best.items() if k!='state'}},STAGE8_DIR/'stage8_semantic_head.pt')
np.savez_compressed(STAGE8_DIR/'stage8_semantic_probabilities.npz',image_ids=sem_ids,source_sha256=sem_source_hash,
    probabilities=_stage8_prob,flip_ids=sem_flip_ids,flip_source_sha256=sem_flip_hash,class_order=np.asarray(_stage8_order))
pd.DataFrame(_stage8_hist).to_csv(STAGE8_DIR/'development_metrics.csv',index=False)
(STAGE8_DIR/'stage8_semantic_report.json').write_text(json.dumps(_stage8_report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Stage8 raw-only semantic candidate:',json.dumps(_stage8_report['selected'],ensure_ascii=False))


### 4.6 Expand anchors with independent recycling-CNN consensus

This stage expands the Stage8 fit-only anchors only where both raw-train CNN sources agree above the frozen confidence thresholds; electronic additions also require the recycling model battery channel. It retrains the same three local feature branches on train fit IDs, and selects on held-out train groups using anchor agreement, flip stability, and fixed neighbor agreement.


In [ ]:
# Rebuild Stage9 consensus-seed expansion and local student from raw-train-derived evidence.
import copy, time
import torch.nn.functional as F
STAGE9_DIR=WORK_ROOT/'labels'/'stage9_consensus_anchor'
STAGE9_DIR.mkdir(parents=True,exist_ok=True)
_stage9_started=time.time()
_stage9_base_clean=(_stage8_support>=.65)&_stage8_agrees&(_stage8_coarse.max(1)>=.8)
_stage9_labels=_stage8_mapped.copy()
_stage9_strong=(GENERIC_PRIOR.argmax(1)==resnet_task_prior.argmax(1))&(GENERIC_PRIOR.max(1)>=.75)&(resnet_task_prior.max(1)>=.9)
_stage9_source_order=SEMANTIC_SOURCE_CLASSES['resnet_waste']
_stage9_battery=np.asarray(resnet_waste['probabilities'])[:,_stage9_source_order.index('battery')]
_stage9_strong&=(_stage9_labels!=_stage8_order.index('electronic'))|(_stage9_battery>=.8)
_stage9_labels[_stage9_strong]=resnet_task_prior.argmax(1)[_stage9_strong]
_stage9_clean=_stage9_base_clean|_stage9_strong
_stage9_seeds=np.flatnonzero(_stage9_clean[_stage8_fit])
_stage9_val_mask=_stage9_clean[_stage8_held]
_stage9_fit_counts=np.bincount(_stage9_labels[_stage8_fit[_stage9_seeds]],minlength=8)
_stage9_val_counts=np.bincount(_stage9_labels[_stage8_held][_stage9_val_mask],minlength=8)
assert (_stage9_fit_counts>=3).all(),_stage9_fit_counts
assert (_stage9_val_counts>0).all(),_stage9_val_counts
_stage9_y=torch.as_tensor(_stage9_labels[_stage8_fit[_stage9_seeds]],device=DEVICE,dtype=torch.long)
_stage9_weights=torch.as_tensor((_stage9_seeds.size/(8*_stage9_fit_counts)).astype(np.float32),device=DEVICE)

def _stage9_evaluate(head,branch):
    x=_stage8_block(branch);xf=_stage8_block(branch,True);head.eval()
    with torch.inference_mode():
        fit_pred=head(torch.as_tensor(x[_stage8_fit],device=DEVICE)).argmax(1).cpu().numpy()
        val_pred=head(torch.as_tensor(x[_stage8_held],device=DEVICE)).argmax(1).cpu().numpy()
        flip_pred=head(torch.as_tensor(xf[_stage8_flip_val_bank],device=DEVICE)).argmax(1).cpu().numpy()
    macro=float(np.mean([(val_pred[_stage9_val_mask&(_stage9_labels[_stage8_held]==c)]==c).mean() for c in range(8)]))
    flip=float((val_pred[_stage8_flip_val_local]==flip_pred).mean())
    neighbor=float((val_pred[:,None]==fit_pred[_stage8_nv]).mean())
    counts=np.bincount(val_pred,minlength=8)
    return {'score':.5*macro+.25*flip+.25*neighbor,'macro_anchor_agreement':macro,'flip_agreement':flip,
            'neighbor_agreement':neighbor,'occupied_classes':int((counts>0).sum()),'minimum_class_count':int(counts.min())}

_stage9_history=[];_stage9_best=None
_stage8_checkpoint=torch.load(STAGE8_DIR/'stage8_semantic_head.pt',map_location='cpu',weights_only=True)
_stage8_prior_head=_stage8_make_head(_stage8_checkpoint['head'],_stage8_block(_stage8_checkpoint['branch']).shape[1]).to(DEVICE)
_stage8_prior_head.load_state_dict(_stage8_checkpoint['state_dict'])
_stage8_metric=_stage9_evaluate(_stage8_prior_head,_stage8_checkpoint['branch'])
_stage9_history.append({'candidate':'stage8_control','branch':_stage8_checkpoint['branch'],'head':_stage8_checkpoint['head'],'epoch':_stage8_checkpoint['selection']['epoch'],**_stage8_metric})
if _stage8_metric['occupied_classes']==8 and _stage8_metric['minimum_class_count']>=5:
    _stage9_best={'candidate':'stage8_control','branch':_stage8_checkpoint['branch'],'head':_stage8_checkpoint['head'],
        'epoch':_stage8_checkpoint['selection']['epoch'],**_stage8_metric,'state':copy.deepcopy(_stage8_prior_head.state_dict())}

for _branch in ['waste','dino','hybrid']:
    x=_stage8_block(_branch);xf=_stage8_block(_branch,True)
    xt=torch.as_tensor(x[_stage8_fit],device=DEVICE,dtype=torch.float32)
    xflip=torch.as_tensor(xf[_stage8_flip_fit_bank],device=DEVICE,dtype=torch.float32)
    flip_local=torch.as_tensor(_stage8_flip_fit_local,device=DEVICE,dtype=torch.long)
    for _kind in ['linear','mlp']:
        torch.manual_seed(20261106)
        head=_stage8_make_head(_kind,x.shape[1]).to(DEVICE)
        optimizer=torch.optim.Adam(head.parameters(),lr=.003,weight_decay=1e-4)
        for epoch in range(1,181):
            head.train();logits=head(xt);prob=logits.softmax(-1)
            ce=F.cross_entropy(logits[_stage9_seeds],_stage9_y,weight=_stage9_weights)
            graph=-(prob[:,None,:]*prob[_stage8_nn]).sum(-1).clamp_min(1e-8).log().mean()
            alt=head(xflip).softmax(-1);orig=prob[flip_local];mean=(orig+alt)*.5
            js=.5*((orig*(orig.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum(1).mean()+
                   (alt*(alt.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum(1).mean())
            loss=ce+.03*graph+.2*js
            if not torch.isfinite(loss):raise RuntimeError('Non-finite Stage9 loss.')
            optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
            if epoch%10==0:
                metric=_stage9_evaluate(head,_branch)
                row={'branch':_branch,'head':_kind,'epoch':epoch,**metric};_stage9_history.append(row)
                if metric['occupied_classes']==8 and metric['minimum_class_count']>=5 and (_stage9_best is None or metric['score']>_stage9_best['score']+1e-7):
                    _stage9_best={**row,'state':copy.deepcopy(head.state_dict())}
        print('Stage9 consensus head',_branch,_kind,'complete',flush=True)
if _stage9_best is None:
    pd.DataFrame(_stage9_history).to_csv(STAGE9_DIR/'development_metrics.csv',index=False)
    raise RuntimeError('Stage9 has no model passing held-out occupancy checks; assignment gate remains closed.')
_stage9_state=_stage9_best.pop('state')
_stage9_final=_stage8_make_head(_stage9_best['head'],_stage8_block(_stage9_best['branch']).shape[1]).to(DEVICE)
_stage9_final.load_state_dict(_stage9_state);_stage9_final.eval()
with torch.inference_mode():
    _stage9_prob=_stage9_final(torch.as_tensor(_stage8_block(_stage9_best['branch']),device=DEVICE,dtype=torch.float32)).softmax(-1).float().cpu().numpy()
_stage9_report={'method':'raw_train_fit_only_stage8_anchors_plus_independent_recycling_CNN_consensus','train_rows':_stage8_n,
    'fit_rows':len(_stage8_fit),'heldout_rows':len(_stage8_held),'base_fit_anchors':int(_stage9_base_clean[_stage8_fit].sum()),
    'expanded_fit_anchors':int(_stage9_clean[_stage8_fit].sum()),'fit_anchor_counts':dict(zip(_stage8_order,_stage9_fit_counts.tolist())),
    'heldout_anchor_counts':dict(zip(_stage8_order,_stage9_val_counts.tolist())),'selected':_stage9_best,'class_order':_stage8_order,
    'test_images_processed':0,'test_used_for_fit':False,'archive_assignments_read':False,'active_assignment_parity_verified':False,
    'seconds':time.time()-_stage9_started,'source_fingerprint':sem_content_fingerprint}
torch.save({'state_dict':{k:v.detach().cpu() for k,v in _stage9_final.state_dict().items()},'branch':_stage9_best['branch'],'head':_stage9_best['head'],
    'class_order':_stage8_order,'fit_ids':_stage8_fit_ids.tolist(),'heldout_ids':_stage8_held_ids.tolist(),
    'fit_group_ids':coarse_fit_group_ids,'heldout_group_ids':coarse_heldout_group_ids,'selection':_stage9_best},STAGE9_DIR/'stage9_semantic_head.pt')
np.savez_compressed(STAGE9_DIR/'stage9_semantic_probabilities.npz',image_ids=sem_ids,source_sha256=sem_source_hash,
    probabilities=_stage9_prob,flip_ids=sem_flip_ids,flip_source_sha256=sem_flip_hash,class_order=np.asarray(_stage8_order))
pd.DataFrame(_stage9_history).to_csv(STAGE9_DIR/'development_metrics.csv',index=False)
(STAGE9_DIR/'stage9_semantic_report.json').write_text(json.dumps(_stage9_report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Stage9 raw-only semantic candidate:',json.dumps(_stage9_report['selected'],ensure_ascii=False))


### 4.7 Add the ontology CNN expert and rebuild Stage12

The pinned ConvNeXt ImageNet-22k ontology evidence expands Stage9 consensus anchors only under the historical confidence, coverage, and source-support rules. Stage12 student heads are fitted on Stage4 fit groups and selected against held-out train anchors, view stability, and fit-only nearest-neighbor agreement.


In [ ]:
from sklearn.preprocessing import normalize as sklearn_normalize
# Rebuild Stage12 ontology-expert anchors and student from raw-train-derived features.
import copy, time
import torch.nn.functional as F
STAGE12_DIR=WORK_ROOT/'labels'/'stage12_ontology_refinement'
STAGE12_DIR.mkdir(parents=True,exist_ok=True)
_stage12_started=time.time()
_stage12_labels=_stage9_labels.copy();_stage12_clean=_stage9_clean.copy()
_stage12_expert=np.asarray(convnext_ontology['conditional'],dtype=np.float32)
_stage12_coverage=np.asarray(convnext_ontology['coverage'],dtype=np.float32)
_stage12_class=_stage12_expert.argmax(1);_stage12_conf=_stage12_expert.max(1)
_stage12_supported=((_stage12_class==GENERIC_PRIOR.argmax(1))&(GENERIC_PRIOR.max(1)>=.65))|((_stage12_class==resnet_task_prior.argmax(1))&(resnet_task_prior.max(1)>=.8))
_stage12_extra=(_stage12_coverage>=.2)&(_stage12_conf>=.8)&_stage12_supported
_stage12_very=(_stage12_coverage>=.5)&(_stage12_conf>=.95)&((_stage12_class==_stage8_mapped)|_stage12_supported)
_stage12_very&=(~_stage9_strong)|(_stage12_class==_stage9_labels)
_stage12_extra|=_stage12_very
_stage12_labels[_stage12_extra]=_stage12_class[_stage12_extra]
_stage12_clean|=_stage12_extra
_stage12_fit_seed=np.flatnonzero(_stage12_clean[_stage8_fit])
_stage12_val_mask=_stage12_clean[_stage8_held]
_stage12_fit_counts=np.bincount(_stage12_labels[_stage8_fit[_stage12_fit_seed]],minlength=8)
_stage12_val_counts=np.bincount(_stage12_labels[_stage8_held][_stage12_val_mask],minlength=8)
assert (_stage12_fit_counts>=3).all(),_stage12_fit_counts
assert (_stage12_val_counts>0).all(),_stage12_val_counts

# All feature blocks are normalized per sample before fusion; flip rows align by raw image ID.
_stage12_conv=sklearn_normalize(np.asarray(convnext_ontology['features'],dtype=np.float32)).astype(np.float32)
_stage12_conv_flip=sklearn_normalize(np.asarray(convnext_ontology['flip_features'],dtype=np.float32)).astype(np.float32)
_stage12_patch=sklearn_normalize(np.asarray(dino2_patch_features,dtype=np.float32)).astype(np.float32)
_stage12_patch_flip=sklearn_normalize(np.asarray(dino2_patch_flip_features,dtype=np.float32)).astype(np.float32)
_stage12_patch_weights=np.repeat(np.asarray([np.sqrt(.5),.5,.5],dtype=np.float32),384)
_stage12_patch_weighted=_stage12_patch*_stage12_patch_weights
_stage12_patch_flip_weighted=_stage12_patch_flip*_stage12_patch_weights
_stage12_all=(np.concatenate([_stage12_patch_weighted,_stage8_waste],axis=1)/np.float32(np.sqrt(2))).astype(np.float32)
_stage12_all_flip=(np.concatenate([_stage12_patch_flip_weighted,_stage8_waste_flip],axis=1)/np.float32(np.sqrt(2))).astype(np.float32)
_stage12_features={
 'ontology':_stage12_conv,
 'ontology_dino':(np.concatenate([_stage12_conv,_stage8_dino],axis=1)/np.float32(np.sqrt(2))).astype(np.float32),
 'ontology_all':(np.concatenate([_stage12_conv,_stage12_all],axis=1)/np.float32(np.sqrt(2))).astype(np.float32),
 'waste':_stage8_waste,'dino':_stage8_dino,'hybrid':_stage8_block('hybrid')}
_stage12_flip_features={
 'ontology':_stage12_conv_flip,
 'ontology_dino':(np.concatenate([_stage12_conv_flip,_stage8_dino_flip],axis=1)/np.float32(np.sqrt(2))).astype(np.float32),
 'ontology_all':(np.concatenate([_stage12_conv_flip,_stage12_all_flip],axis=1)/np.float32(np.sqrt(2))).astype(np.float32),
 'waste':_stage8_waste_flip,'dino':_stage8_dino_flip,'hybrid':_stage8_block('hybrid',True)}
_stage12_feature_branch='ontology_dino'
_stage12_geometry=_stage12_features[_stage12_feature_branch]
with threadpool_limits(limits=4):
    _stage12_sim=_stage12_geometry[_stage8_fit]@_stage12_geometry[_stage8_fit].T
    np.fill_diagonal(_stage12_sim,-np.inf)
    _stage12_nn=np.argsort(-_stage12_sim,axis=1,kind='stable')[:,:10]
    _stage12_nv=np.argsort(-(_stage12_geometry[_stage8_held]@_stage12_geometry[_stage8_fit].T),axis=1,kind='stable')[:,:10]

def _stage12_evaluate(head,branch):
    x=_stage12_features[branch];xf=_stage12_flip_features[branch];head.eval()
    with torch.inference_mode():
        fit_pred=head(torch.as_tensor(x[_stage8_fit],device=DEVICE)).argmax(1).cpu().numpy()
        val_pred=head(torch.as_tensor(x[_stage8_held],device=DEVICE)).argmax(1).cpu().numpy()
        flip_pred=head(torch.as_tensor(xf[_stage8_flip_val_bank],device=DEVICE)).argmax(1).cpu().numpy()
    macro=float(np.mean([(val_pred[_stage12_val_mask&(_stage12_labels[_stage8_held]==c)]==c).mean() for c in range(8)]))
    flip=float((val_pred[_stage8_flip_val_local]==flip_pred).mean())
    neighbor=float((val_pred[:,None]==fit_pred[_stage12_nv]).mean())
    counts=np.bincount(val_pred,minlength=8)
    return {'score':.5*macro+.25*flip+.25*neighbor,'macro_anchor_agreement':macro,'flip_agreement':flip,
            'neighbor_agreement':neighbor,'occupied_classes':int((counts>0).sum()),'minimum_class_count':int(counts.min())}

_stage12_history=[];_stage12_best=None
_stage9_checkpoint=torch.load(STAGE9_DIR/'stage9_semantic_head.pt',map_location='cpu',weights_only=True)
_stage9_control=_stage8_make_head(_stage9_checkpoint['head'],_stage12_features[_stage9_checkpoint['branch']].shape[1]).to(DEVICE)
_stage9_control.load_state_dict(_stage9_checkpoint['state_dict'])
_stage9_control_metric=_stage12_evaluate(_stage9_control,_stage9_checkpoint['branch'])
_stage12_history.append({'candidate':'stage9_control','branch':_stage9_checkpoint['branch'],'head':_stage9_checkpoint['head'],
                         'epoch':_stage9_checkpoint['selection']['epoch'],**_stage9_control_metric})
if _stage9_control_metric['occupied_classes']==8 and _stage9_control_metric['minimum_class_count']>=5:
    _stage12_best={'candidate':'stage9_control','branch':_stage9_checkpoint['branch'],'head':_stage9_checkpoint['head'],
        'epoch':_stage9_checkpoint['selection']['epoch'],**_stage9_control_metric,'state':copy.deepcopy(_stage9_control.state_dict())}
_stage12_y=torch.as_tensor(_stage12_labels[_stage8_fit[_stage12_fit_seed]],device=DEVICE,dtype=torch.long)
_stage12_weights=torch.as_tensor((_stage12_fit_seed.size/(8*_stage12_fit_counts)).astype(np.float32),device=DEVICE)
for _branch in ['ontology','ontology_dino','ontology_all']:
    x=_stage12_features[_branch];xf=_stage12_flip_features[_branch]
    xt=torch.as_tensor(x[_stage8_fit],device=DEVICE,dtype=torch.float32)
    xflip=torch.as_tensor(xf[_stage8_flip_fit_bank],device=DEVICE,dtype=torch.float32)
    flip_local=torch.as_tensor(_stage8_flip_fit_local,device=DEVICE,dtype=torch.long)
    for _kind in ['linear','mlp']:
        torch.manual_seed(20261106)
        head=_stage8_make_head(_kind,x.shape[1]).to(DEVICE)
        optimizer=torch.optim.Adam(head.parameters(),lr=.003,weight_decay=1e-4)
        for epoch in range(1,181):
            head.train();logits=head(xt);prob=logits.softmax(-1)
            ce=F.cross_entropy(logits[_stage12_fit_seed],_stage12_y,weight=_stage12_weights)
            graph=-(prob[:,None,:]*prob[_stage12_nn]).sum(-1).clamp_min(1e-8).log().mean()
            alt=head(xflip).softmax(-1);orig=prob[flip_local];mean=(orig+alt)*.5
            js=.5*((orig*(orig.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum(1).mean()+
                   (alt*(alt.clamp_min(1e-8).log()-mean.clamp_min(1e-8).log())).sum(1).mean())
            loss=ce+.03*graph+.2*js
            if not torch.isfinite(loss):raise RuntimeError('Non-finite Stage12 loss.')
            optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
            if epoch%10==0:
                metric=_stage12_evaluate(head,_branch);row={'branch':_branch,'head':_kind,'epoch':epoch,**metric};_stage12_history.append(row)
                if metric['occupied_classes']==8 and metric['minimum_class_count']>=5 and (_stage12_best is None or metric['score']>_stage12_best['score']+1e-7):
                    _stage12_best={**row,'state':copy.deepcopy(head.state_dict())}
        print('Stage12 ontology head',_branch,_kind,'complete',flush=True)
if _stage12_best is None:
    pd.DataFrame(_stage12_history).to_csv(STAGE12_DIR/'development_metrics.csv',index=False)
    raise RuntimeError('Stage12 has no model passing held-out occupancy checks; assignment gate remains closed.')
_stage12_state=_stage12_best.pop('state')
_stage12_final=_stage8_make_head(_stage12_best['head'],_stage12_features[_stage12_best['branch']].shape[1]).to(DEVICE)
_stage12_final.load_state_dict(_stage12_state);_stage12_final.eval()
with torch.inference_mode():
    _stage12_prob=_stage12_final(torch.as_tensor(_stage12_features[_stage12_best['branch']],device=DEVICE,dtype=torch.float32)).softmax(-1).float().cpu().numpy()
_stage12_report={'method':'raw_train_fit_only_stage9_consensus_plus_convnext_ontology_expert','train_rows':_stage8_n,
    'fit_rows':len(_stage8_fit),'heldout_rows':len(_stage8_held),'expert_added_fit_anchors':int((_stage12_extra[_stage8_fit]&~_stage9_clean[_stage8_fit]).sum()),
    'fit_anchor_counts':dict(zip(_stage8_order,_stage12_fit_counts.tolist())),'heldout_anchor_counts':dict(zip(_stage8_order,_stage12_val_counts.tolist())),
    'selected':_stage12_best,'class_order':_stage8_order,'test_images_processed':0,'test_used_for_fit':False,
    'archive_assignments_read':False,'active_assignment_parity_verified':False,'seconds':time.time()-_stage12_started,
    'source_fingerprint':sem_content_fingerprint}
torch.save({'state_dict':{k:v.detach().cpu() for k,v in _stage12_final.state_dict().items()},'branch':_stage12_best['branch'],'head':_stage12_best['head'],
    'class_order':_stage8_order,'fit_ids':_stage8_fit_ids.tolist(),'heldout_ids':_stage8_held_ids.tolist(),
    'fit_group_ids':coarse_fit_group_ids,'heldout_group_ids':coarse_heldout_group_ids,'selection':_stage12_best},STAGE12_DIR/'stage12_semantic_head.pt')
np.savez_compressed(STAGE12_DIR/'stage12_semantic_probabilities.npz',image_ids=sem_ids,source_sha256=sem_source_hash,
    probabilities=_stage12_prob,flip_ids=sem_flip_ids,flip_source_sha256=sem_flip_hash,class_order=np.asarray(_stage8_order))
pd.DataFrame(_stage12_history).to_csv(STAGE12_DIR/'development_metrics.csv',index=False)
(STAGE12_DIR/'stage12_semantic_report.json').write_text(json.dumps(_stage12_report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Stage12 raw-only semantic candidate:',json.dumps(_stage12_report['selected'],ensure_ascii=False))


### 4.8 Rebuild independent-source Stage16 and fixed Stage17 routing

Stage16 membentuk anchor konsensus train-only dari empat keluarga fitur/model, lalu memasang head kandidat pada grup fit Stage4. Stage17 merutekan grup memakai keragaman konsensus sumber: minimal 20 anchor dan porsi kelas dominan di bawah 60% memilih Stage16; grup lain memakai Stage12. Aturan struktural ini ditetapkan sebelum evaluasi.

In [ ]:
from sklearn.preprocessing import normalize as sklearn_normalize
# Rebuild Stage16 independent-source consensus and Stage17 hierarchical routing.
import copy,time
import torch.nn.functional as F
from threadpoolctl import threadpool_limits
STAGE16_DIR=WORK_ROOT/'labels'/'stage16_source_consensus';STAGE17_DIR=WORK_ROOT/'labels'/'stage17_hierarchical_routing'
STAGE16_DIR.mkdir(parents=True,exist_ok=True);STAGE17_DIR.mkdir(parents=True,exist_ok=True)
_stage16_started=time.time()
def _stage16_map(prob,source_classes):
    mapping={'battery':'electronic','biological':'foodwaste','brown-glass':'glass','green-glass':'glass','white-glass':'glass',
             'glass':'glass','clothes':'textile','shoes':'textile','cardboard':'cardboard','paper':'paper','metal':'metal','plastic':'plastic'}
    ev=np.zeros((_stage8_n,8),dtype=np.float32)
    for j,name in enumerate(source_classes):
        target=mapping.get(name)
        if target is not None:ev[:,_stage8_order.index(target)]+=prob[:,j]
    cov=ev.sum(1);return ev/np.maximum(cov[:,None],1e-8),cov
_stage16_eff,_stage16_eff_cov=_stage16_map(second_waste['probabilities'],SEMANTIC_SOURCE_CLASSES['efficientnet_waste'])
_stage16_sources=np.stack([GENERIC_PRIOR,resnet_task_prior,_stage16_eff,_stage12_expert]).astype(np.float32)
_stage16_cov=np.stack([np.ones(_stage8_n,dtype=np.float32),resnet_task_coverage,_stage16_eff_cov,_stage12_coverage]).astype(np.float32)
_stage16_threshold=np.asarray([0,.6,.6,.2],dtype=np.float32)[:,None]
_stage16_reliable=(_stage16_sources.max(2)>=.8)&(_stage16_cov>=_stage16_threshold)
_stage16_votes=np.zeros((_stage8_n,8),dtype=np.uint8)
for j in range(4):
    rows=np.flatnonzero(_stage16_reliable[j]);_stage16_votes[rows,_stage16_sources[j,rows].argmax(1)]+=1
_stage16_rank=np.sort(_stage16_votes,axis=1)
_stage16_clean=(_stage16_rank[:,-1]>=2)&(_stage16_rank[:,-1]>_stage16_rank[:,-2])
_stage16_labels=_stage16_votes.argmax(1);_stage16_seed=np.flatnonzero(_stage16_clean[_stage8_fit]);_stage16_core=_stage16_clean[_stage8_held]
_stage16_counts=np.bincount(_stage16_labels[_stage8_fit[_stage16_seed]],minlength=8)
_stage16_val_counts=np.bincount(_stage16_labels[_stage8_held][_stage16_core],minlength=8)
assert (_stage16_counts>=3).all(),_stage16_counts
assert (_stage16_val_counts>0).all(),_stage16_val_counts
_stage16_eff_feat=sklearn_normalize(np.asarray(second_waste['features'],dtype=np.float32)).astype(np.float32)
_stage16_eff_flip=sklearn_normalize(np.asarray(second_waste['flip_features'],dtype=np.float32)).astype(np.float32)
_stage16_features={**_stage12_features,'dino':_stage8_dino,'ontology_all':_stage12_features['ontology_all'],
 'second_dino':(np.concatenate([_stage16_eff_feat,_stage8_dino],axis=1)/np.float32(np.sqrt(2))).astype(np.float32),
 'second_all':(np.concatenate([_stage16_eff_feat,_stage12_features['ontology_all']],axis=1)/np.float32(np.sqrt(2))).astype(np.float32)}
_stage16_flips={**_stage12_flip_features,'dino':_stage8_dino_flip,'ontology_all':_stage12_flip_features['ontology_all'],
 'second_dino':(np.concatenate([_stage16_eff_flip,_stage8_dino_flip],axis=1)/np.float32(np.sqrt(2))).astype(np.float32),
 'second_all':(np.concatenate([_stage16_eff_flip,_stage12_flip_features['ontology_all']],axis=1)/np.float32(np.sqrt(2))).astype(np.float32)}
with threadpool_limits(limits=4):
    _stage16_geom=_stage16_features['ontology_all']
    def _stage16_topk(query_rows,bank_rows,exclude_diagonal=False,k=10,chunk_size=64):
        bank=_stage16_geom[bank_rows];out=np.empty((len(query_rows),min(k,len(bank_rows))),dtype=np.int32)
        for start in range(0,len(query_rows),chunk_size):
            stop=min(start+chunk_size,len(query_rows))
            sim=_stage16_geom[query_rows[start:stop]]@bank.T
            if exclude_diagonal:sim[np.arange(stop-start),np.arange(start,stop)]=-np.inf
            out[start:stop]=np.argsort(-sim,axis=1,kind='stable')[:,:out.shape[1]]
        return out
    _stage16_nn=_stage16_topk(_stage8_fit,_stage8_fit,exclude_diagonal=True)
    _stage16_nv=_stage16_topk(_stage8_held,_stage8_fit)
def _stage16_eval(head,branch):
    x=_stage16_features[branch];xf=_stage16_flips[branch];head.eval()
    def _predict_rows(features,rows,batch_size=16):
        out=np.empty(len(rows),dtype=np.int64)
        with torch.inference_mode():
            for start in range(0,len(rows),batch_size):
                stop=min(start+batch_size,len(rows))
                batch=torch.as_tensor(features[rows[start:stop]],device=DEVICE,dtype=torch.float32)
                out[start:stop]=head(batch).argmax(1).cpu().numpy()
                del batch
        return out
    pred=_predict_rows(x,np.arange(len(x),dtype=np.int64))
    flipped=_predict_rows(xf,_stage8_flip_val_bank)
    val=pred[_stage8_held];macro=float(np.mean([(val[_stage16_core&(_stage16_labels[_stage8_held]==c)]==c).mean() for c in range(8)]))
    flip=float((val[_stage8_flip_val_local]==flipped).mean());near=float((val[:,None]==pred[_stage8_fit][_stage16_nv]).mean());counts=np.bincount(val,minlength=8)
    return {'score':.75*macro+.25*flip,'macro_anchor_agreement':macro,'flip_agreement':flip,'neighbor_agreement':near,
            'occupied_classes':int((counts>0).sum()),'minimum_class_count':int(counts.min())}
_stage16_hist=[];_stage16_best=None
_stage12_ckpt=torch.load(STAGE12_DIR/'stage12_semantic_head.pt',map_location='cpu',weights_only=True)
_control=_stage8_make_head(_stage12_ckpt['head'],_stage16_features[_stage12_ckpt['branch']].shape[1]).to(DEVICE)
_control.load_state_dict(_stage12_ckpt['state_dict']);_cm=_stage16_eval(_control,_stage12_ckpt['branch'])
_stage16_hist.append({'candidate':'stage12_control','branch':_stage12_ckpt['branch'],'head':_stage12_ckpt['head'],'epoch':_stage12_ckpt['selection']['epoch'],**_cm})
if _cm['occupied_classes']==8 and _cm['minimum_class_count']>=5:
    _stage16_best={'candidate':'stage12_control','branch':_stage12_ckpt['branch'],'head':_stage12_ckpt['head'],'epoch':_stage12_ckpt['selection']['epoch'],**_cm,'state':copy.deepcopy(_control.state_dict())}
_stage16_y=torch.as_tensor(_stage16_labels[_stage8_fit[_stage16_seed]],device=DEVICE,dtype=torch.long)
_stage16_w=torch.as_tensor((_stage16_seed.size/(8*_stage16_counts)).astype(np.float32),device=DEVICE)
for branch in ['dino','ontology_all','second_dino','second_all']:
    x=_stage16_features[branch];xf=_stage16_flips[branch];xt=torch.as_tensor(x[_stage8_fit],device=DEVICE,dtype=torch.float32)
    xflip=torch.as_tensor(xf[_stage8_flip_fit_bank],device=DEVICE,dtype=torch.float32);flocal=torch.as_tensor(_stage8_flip_fit_local,device=DEVICE,dtype=torch.long)
    for kind in ['linear','mlp']:
        torch.manual_seed(20261106);head=_stage8_make_head(kind,x.shape[1]).to(DEVICE);opt=torch.optim.Adam(head.parameters(),lr=.003,weight_decay=1e-4)
        for epoch in range(1,181):
            head.train();logits=head(xt);q=logits.softmax(-1);ce=F.cross_entropy(logits[_stage16_seed],_stage16_y,weight=_stage16_w)
            graph=-(q[:,None,:]*q[_stage16_nn]).sum(-1).clamp_min(1e-8).log().mean()
            alt=head(xflip).softmax(-1);orig=q[flocal];m=(orig+alt)*.5
            js=.5*((orig*(orig.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum(1).mean()+(alt*(alt.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum(1).mean())
            loss=ce+.005*graph+.2*js
            if not torch.isfinite(loss):raise RuntimeError('Non-finite Stage16 loss.')
            opt.zero_grad(set_to_none=True);loss.backward();opt.step()
            if epoch%10==0:
                metric=_stage16_eval(head,branch);row={'branch':branch,'head':kind,'epoch':epoch,**metric};_stage16_hist.append(row)
                if metric['occupied_classes']==8 and metric['minimum_class_count']>=5 and (_stage16_best is None or metric['score']>_stage16_best['score']+1e-7):_stage16_best={**row,'state':copy.deepcopy(head.state_dict())}
        print('Stage16',branch,kind,'complete',flush=True)
if _stage16_best is None:
    pd.DataFrame(_stage16_hist).to_csv(STAGE16_DIR/'development_metrics.csv',index=False)
    raise RuntimeError('Stage16 no candidate passed held-out occupancy guard; assignment gate remains closed.')
_stage16_state=_stage16_best.pop('state');_stage16_final=_stage8_make_head(_stage16_best['head'],_stage16_features[_stage16_best['branch']].shape[1]).to(DEVICE)
_stage16_final.load_state_dict(_stage16_state);_stage16_final.eval()
with torch.inference_mode():_stage16_prob=_stage16_final(torch.as_tensor(_stage16_features[_stage16_best['branch']],device=DEVICE,dtype=torch.float32)).softmax(-1).cpu().numpy()
_stage16_ckpt={'state_dict':{k:v.detach().cpu() for k,v in _stage16_final.state_dict().items()},'branch':_stage16_best['branch'],'head':_stage16_best['head'],
 'class_order':_stage8_order,'fit_ids':_stage8_fit_ids.tolist(),'heldout_ids':_stage8_held_ids.tolist(),'selection':_stage16_best}
torch.save(_stage16_ckpt,STAGE16_DIR/'stage16_semantic_head.pt')
np.savez_compressed(STAGE16_DIR/'stage16_semantic_probabilities.npz',image_ids=sem_ids,source_sha256=sem_source_hash,probabilities=_stage16_prob,
 flip_ids=sem_flip_ids,flip_source_sha256=sem_flip_hash,class_order=np.asarray(_stage8_order))
pd.DataFrame(_stage16_hist).to_csv(STAGE16_DIR/'development_metrics.csv',index=False)
_stage16_report={'method':'raw_train_independent_four_family_consensus_student','sources':['generic','resnet_waste','efficientnet_waste','convnext_ontology'],
 'source_coverage_thresholds':_stage16_threshold.ravel().tolist(),'fit_anchor_counts':dict(zip(_stage8_order,_stage16_counts.tolist())),
 'heldout_anchor_counts':dict(zip(_stage8_order,_stage16_val_counts.tolist())),'selected':_stage16_best,'test_images_processed':0,
 'test_used_for_fit':False,'coarse_cluster_targets_used':False,'archive_assignments_read':False,'active_assignment_parity_verified':False,
 'source_fingerprint':sem_content_fingerprint,'seconds':time.time()-_stage16_started}
(STAGE16_DIR/'stage16_semantic_report.json').write_text(json.dumps(_stage16_report,indent=2,ensure_ascii=False),encoding='utf-8')

# Fixed Stage17 group-diversity route; its threshold is not searched against reference assignments.
_stage17_records=[];_stage17_groups=[];_stage17_coarse=_stage8_coarse.argmax(1)
for g in range(8):
    mask=(_stage17_coarse[_stage8_fit]==g)&_stage16_clean[_stage8_fit];counts=np.bincount(_stage16_labels[_stage8_fit][mask],minlength=8)
    n=int(counts.sum());purity=float(counts.max()/max(n,1));use=n>=20 and purity<.6
    if use:_stage17_groups.append(g)
    _stage17_records.append({'anonymous_group':g,'fit_consensus_anchors':n,'dominant_semantic_share':purity,'refinement_selected':use,**dict(zip(_stage8_order,counts.tolist()))})
pd.DataFrame(_stage17_records).to_csv(STAGE17_DIR/'fit_only_group_diversity.csv',index=False)
_stage17_route=np.isin(_stage17_coarse,_stage17_groups)
_stage17_flip_groups=coarse_head(torch.as_tensor(dino2_flip_features,device=DEVICE)).argmax(1).cpu().numpy()
_stage17_route_flip=np.isin(_stage17_flip_groups,_stage17_groups)
_stage17_prob=np.where(_stage17_route[:,None],_stage16_prob,_stage12_prob)
_stage17_stable_flip=_stage12_flip_head if '_stage12_flip_head' in globals() else None
_stage17_h12=_stage8_make_head(_stage12_ckpt['head'],_stage12_flip_features[_stage12_ckpt['branch']].shape[1]).to(DEVICE)
_stage17_h12.load_state_dict(_stage12_ckpt['state_dict']);_stage17_h12.eval()
_stage17_h16=_stage8_make_head(_stage16_ckpt['head'],_stage16_flips[_stage16_ckpt['branch']].shape[1]).to(DEVICE)
_stage17_h16.load_state_dict(_stage16_ckpt['state_dict']);_stage17_h16.eval()
with torch.inference_mode():
    p12f=_stage17_h12(torch.as_tensor(_stage12_flip_features[_stage12_ckpt['branch']],device=DEVICE,dtype=torch.float32)).softmax(-1).cpu().numpy()
    p16f=_stage17_h16(torch.as_tensor(_stage16_flips[_stage16_ckpt['branch']],device=DEVICE,dtype=torch.float32)).softmax(-1).cpu().numpy()
_stage17_prob_flip=np.where(_stage17_route_flip[:,None],p16f,p12f)
_stage17_pred=_stage17_prob.argmax(1);_stage17_sorted=np.sort(_stage17_prob,axis=1)
_stage17_report={'method':'fixed_hierarchical_material_routing','stable_stage':'Stage12','refinement_stage':'Stage16','coarse_stage':'Stage4',
 'refinement_groups':_stage17_groups,'minimum_fit_consensus_anchors':20,'dominant_share_threshold':.6,
 'routed_train_rows':int(_stage17_route.sum()),'train_rows':_stage8_n,'test_images_processed':0,'archive_assignments_read':False,
 'active_assignment_parity_verified':False,'source_fingerprint':sem_content_fingerprint}
np.savez_compressed(STAGE17_DIR/'stage17_semantic_probabilities.npz',image_ids=sem_ids,source_sha256=sem_source_hash,
 probabilities=_stage17_prob,flip_ids=sem_flip_ids,flip_source_sha256=sem_flip_hash,flip_probabilities=_stage17_prob_flip,
 routed=_stage17_route,class_order=np.asarray(_stage8_order))
pd.DataFrame({'image_id':sem_ids,'coarse_group':_stage17_coarse,'routed_to_stage16':_stage17_route,
 'predicted_class':np.asarray(_stage8_order)[_stage17_pred],'confidence':_stage17_prob.max(1),
 'margin':_stage17_sorted[:,-1]-_stage17_sorted[:,-2],'source_sha256':sem_source_hash}).to_csv(STAGE17_DIR/'stage17_train_candidate_assignments.csv',index=False)
(STAGE17_DIR/'stage17_routing_report.json').write_text(json.dumps(_stage17_report,indent=2,ensure_ascii=False),encoding='utf-8')

# Separate post-hoc seed overlay for audit only. It never feeds Stage16/17 fitting,
# model selection, or the automatic assignment probabilities.
_stage17_seed_audit_labels=_stage17_pred.copy()
_stage17_seed_audit_mask=np.zeros(_stage8_n,dtype=bool)
_stage17_seed_pos={sid:i for i,sid in enumerate(sem_ids)}
for _seed_id,_seed_class in SEED_LABELS.items():
    _seed_pos=_stage17_seed_pos.get(str(_seed_id))
    if _seed_pos is None: raise ValueError(f'Participant seed missing from raw train: {_seed_id}')
    _stage17_seed_audit_labels[_seed_pos]=_stage8_order.index(_seed_class)
    _stage17_seed_audit_mask[_seed_pos]=True
assert int(_stage17_seed_audit_mask.sum())==37
_stage17_seed_audit=pd.DataFrame({'image_id':sem_ids,'source_sha256':sem_source_hash,
    'stage17_class':np.asarray(_stage8_order)[_stage17_pred],
    'stage17_plus_seed_audit_class':np.asarray(_stage8_order)[_stage17_seed_audit_labels],
    'seed_overlay_applied_for_audit_only':_stage17_seed_audit_mask})
_stage17_seed_audit.to_csv(STAGE17_DIR/'stage17_plus_37_seed_audit_assignments.csv',index=False)
_stage17_seed_audit_report={'scope':'audit overlay only; not used for fitting, model selection, or automatic predictions',
    'raw_train_rows':int(_stage8_n),'seed_ids_applied':int(_stage17_seed_audit_mask.sum()),
    'seed_overlay_changed_stage17_predictions':int(np.sum(_stage17_seed_audit_mask & (_stage17_seed_audit_labels!=_stage17_pred))),
    'stage17_class_counts':dict(zip(_stage8_order,np.bincount(_stage17_pred,minlength=8).tolist())),
    'stage17_plus_seed_audit_class_counts':dict(zip(_stage8_order,np.bincount(_stage17_seed_audit_labels,minlength=8).tolist())),
    'active_manifest_loaded':False,'active_manifest_agreement_computed':False,
    'test_images_processed':0,'source_fingerprint':sem_content_fingerprint}
(STAGE17_DIR/'stage17_plus_37_seed_audit_report.json').write_text(json.dumps(_stage17_seed_audit_report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Stage17 train-only candidate ready:',json.dumps({k:v for k,v in _stage17_report.items() if k not in ['source_fingerprint']},ensure_ascii=False))


### 4.9 Cross-fit denoising lima keluarga fitur

Tahap ini membangun assignment lanjutan dari prediksi Stage17 dan fitur gambar train. Lima keluarga model memprediksi fold yang ditahan berdasarkan grup sumber; perubahan otomatis diterima jika sedikitnya empat keluarga setuju. Lima keluarga menghasilkan vote untuk seluruh foto train; keputusan per ID dan dukungan vote disimpan dalam manifest. Seed peserta dicatat terpisah dan baru dipakai sebagai label eksplisit saat membentuk pool supervised.

In [ ]:
# Iterative five-family pseudo-label refinement; all targets originate in notebook raw-train stages.
import json, time
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

STAGE18_DIR=WORK_ROOT/'labels'/'stage18_iterative_crossfit'
STAGE18_DIR.mkdir(parents=True,exist_ok=True)
_s18_started=time.time();_s18_classes=list(_stage8_order);_s18_n=len(sem_ids)
assert _s18_n==1920 and _stage17_prob.shape==(_s18_n,8)
_s18_initial=_stage17_prob.argmax(1).astype(np.int32)
_s18_initial_conf=_stage17_prob.max(1).astype(np.float32)
# Independent source votes are regenerated above from raw-train image evidence.
_s18_reliable=(_stage16_sources.max(2)>=.8)&(_stage16_cov>=np.asarray([0,.6,.6,.2],dtype=np.float32)[:,None])
_s18_votes=np.zeros((_s18_n,8),dtype=np.uint8)
for _source_i in range(_s18_reliable.shape[0]):
    _rows=np.flatnonzero(_s18_reliable[_source_i])
    _s18_votes[_rows,_stage16_sources[_source_i,_rows].argmax(1)]+=1
_s18_ranked=np.sort(_s18_votes,axis=1)
_s18_auto_anchor=(_s18_ranked[:,-1]>=2)&(_s18_ranked[:,-1]>_s18_ranked[:,-2])
_s18_auto_class=_s18_votes.argmax(1).astype(np.int32)
_s18_core=_s18_auto_anchor&(_s18_auto_class==_s18_initial)&(_s18_initial_conf>=.90)
_s18_fixed=(_s18_initial_conf>=.95)&_s18_auto_anchor&(_s18_ranked[:,-1]>=3)&(_s18_auto_class==_s18_initial)
_s18_initial_counts=np.bincount(_s18_initial,minlength=8)
_s18_core_counts=np.bincount(_s18_initial[_s18_core],minlength=8)
assert np.all(_s18_initial_counts>0), f'Stage17 predictions miss classes: {np.flatnonzero(_s18_initial_counts==0).tolist()}'
assert _s18_core.any(), 'No high-confidence source anchors available; skip refinement.'
_s18_missing_core=np.flatnonzero(_s18_core_counts==0)
if len(_s18_missing_core):
    print('Stage18 note: core-anchor agreement unavailable for classes:',[ _s18_classes[c] for c in _s18_missing_core ],flush=True)
_s18_weight=(.25+.75*np.clip(_s18_initial_conf,0,1)).astype(np.float32)

# Align all image-only descriptors to numeric train IDs before cross-fitting.
_s18_d2pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(train_keys)}
_s18_sem_pos={sid:i for i,sid in enumerate(sem_ids)}
_s18_d2=np.stack([dino2_train_features[_s18_d2pos[sid]] for sid in sem_ids]).astype(np.float32)
_s18_d2flip_pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(flip_keys)}
_s18_flip_numeric=np.asarray([norm_id(k.split(':',1)[1]) for k in flip_keys],dtype=str)
assert len(set(_s18_flip_numeric))==len(_s18_flip_numeric)
_s18_flip_rows=np.asarray([_s18_sem_pos[k] for k in _s18_flip_numeric],dtype=np.int64)
_s18_d2flip=np.stack([dino2_flip_features[_s18_d2flip_pos[sid]] for sid in _s18_flip_numeric]).astype(np.float32)
_s18_patch=np.stack([dino2_patch_features[_s18_d2pos[sid]] for sid in sem_ids]).astype(np.float32)
_s18_patch_flip=np.stack([dino2_patch_flip_features[_s18_d2flip_pos[sid]] for sid in _s18_flip_numeric]).astype(np.float32)
_s18_patch_w=np.tile(np.asarray([np.sqrt(.5),.5,.5],dtype=np.float32),384)
_s18_d3pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(train_keys)}
_s18_d3flip_pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(flip_keys)}
_s18_v3=np.stack([dino3_train_features[_s18_d3pos[sid]] for sid in sem_ids]).astype(np.float32)
_s18_v3flip=np.stack([dino3_flip_features[_s18_d3flip_pos[sid]] for sid in _s18_flip_numeric]).astype(np.float32)
_s18_unit=lambda a:(a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12)).astype(np.float32)
_s18_fg=lambda a:_s18_unit(np.concatenate([a[:,:384]*np.sqrt(.5),a[:,768:1152]*.5,a[:,1152:]*.5],axis=1))
_s18_banks={
 'dino2':(_s18_unit(_s18_d2),_s18_unit(_s18_d2flip)),
 'dino2_local':(_s18_unit(_s18_patch*_s18_patch_w),_s18_unit(_s18_patch_flip*_s18_patch_w)),
 'dino3_global':(_s18_unit(_s18_v3[:,:768]),_s18_unit(_s18_v3flip[:,:768])),
 'dino3_foreground':(_s18_fg(_s18_v3),_s18_fg(_s18_v3flip)),
 'dino3_task8':(_s18_unit(np.concatenate([_s18_unit(_s18_v3[:,:768]),_s18_unit(task8_teacher_features)],1)),
                 _s18_unit(np.concatenate([_s18_unit(_s18_v3flip[:,:768]),_s18_unit(task8_teacher_flip_features)],1))) }
assert all(x.shape[0]==_s18_n and xf.shape[0]==len(sem_flip_ids) for x,xf in _s18_banks.values())
_s18_sem_flip_numeric=np.asarray([norm_id(str(k).split(':',1)[1] if ':' in str(k) else k) for k in sem_flip_ids],dtype=str)
assert np.array_equal(_s18_sem_flip_numeric,_s18_flip_numeric)
_s18_group_pos={sid:i for i,sid in enumerate(ordered_train_ids)}
_s18_groups=np.asarray([str(coarse_groups[_s18_group_pos[sid]]) for sid in sem_ids])
_s18_fold_ids=np.full(_s18_n,-1,dtype=np.int16)
_s18_folds=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=20261108).split(np.zeros(_s18_n),_s18_initial,groups=_s18_groups))
for _fold_i,(_tr,_va) in enumerate(_s18_folds):
    assert set(_s18_groups[_tr]).isdisjoint(_s18_groups[_va])
    _s18_fold_ids[_va]=_fold_i
assert (_s18_fold_ids>=0).all()

# Fixed train-only champion recipes from the established five-family procedure.
_s18_recipes=[
 {'family':'dino2','kind':'svm','kernel':'linear','C':1.0},
 {'family':'dino2_local','kind':'knn','neighbors':7},
 {'family':'dino3_global','kind':'knn','neighbors':21},
 {'family':'dino3_foreground','kind':'svm','kernel':'rbf','gamma':4.0,'C':1.0},
 {'family':'dino3_task8','kind':'svm','kernel':'linear','C':1.0},]

def _s18_make_kernel(sim,recipe):
    if recipe['kind']=='knn' or recipe['kernel']=='linear': return sim
    return np.exp(-2*recipe['gamma']*np.maximum(1-sim,0)).astype(np.float32)

def _s18_knn_scores(sim,fit,labels,weights,k):
    nn=np.argsort(-sim,axis=1,kind='stable')[:,:k]
    values=np.take_along_axis(sim,nn,axis=1)
    w=np.exp((values-values.max(1,keepdims=True))/.07)*weights[fit][nn]
    return np.stack([(w*(labels[fit][nn]==c)).sum(1) for c in range(8)],axis=1)

def _s18_oof_family(recipe,labels,weights,need_flip=False):
    name=recipe['family'];x,xf=_s18_banks[name];sim=x@x.T;fsim=xf@x.T
    kernel=_s18_make_kernel(sim,recipe);fkernel=_s18_make_kernel(fsim,recipe)
    pred=np.full(_s18_n,-1,dtype=np.int32);flip_pred=np.full(len(sem_flip_ids),-1,dtype=np.int32)
    for tr,va in _s18_folds:
        if recipe['kind']=='svm':
            model=SVC(kernel='precomputed',C=recipe['C'],class_weight='balanced',decision_function_shape='ovr',break_ties=True)
            model.fit(kernel[np.ix_(tr,tr)],labels[tr],sample_weight=weights[tr])
            pred[va]=model.predict(kernel[np.ix_(va,tr)])
            if need_flip:
                local=np.flatnonzero(np.isin(_s18_flip_rows,va))
                if len(local):flip_pred[local]=model.predict(fkernel[np.ix_(local,tr)])
        else:
            pred[va]=_s18_knn_scores(kernel[np.ix_(va,tr)],tr,labels,weights,recipe['neighbors']).argmax(1)
            if need_flip:
                local=np.flatnonzero(np.isin(_s18_flip_rows,va))
                if len(local):flip_pred[local]=_s18_knn_scores(fkernel[np.ix_(local,tr)],tr,labels,weights,recipe['neighbors']).argmax(1)
    assert (pred>=0).all()
    if need_flip: assert (flip_pred>=0).all()
    return pred,flip_pred

_s18_labels=_s18_initial.copy();_s18_rounds=[];_s18_first_metrics=[];_s18_last_predictions=None
with threadpool_limits(limits=4):
    for _round in range(1,5):
        _round_predictions=[]
        for _family_i,_recipe in enumerate(_s18_recipes):
            _pred,_flip_pred=_s18_oof_family(_recipe,_s18_labels,_s18_weight,need_flip=(_round==1))
            _round_predictions.append(_pred)
            if _round==1:
                _core_scores=[]
                for _c in range(8):
                    _m=_s18_core&(_s18_labels==_c)
                    if _m.any(): _core_scores.append(float((_pred[_m]==_c).mean()))
                _s18_first_metrics.append({'family':_recipe['family'],'core_macro_agreement':float(np.mean(_core_scores)),
                    'core_classes_covered':len(_core_scores),
                    'hflip_oof_agreement':float(np.mean(_pred[_s18_flip_rows]==_flip_pred)),
                    'recipe':_recipe})
        _family_predictions=np.stack(_round_predictions,axis=0)
        _tally=np.stack([(_family_predictions==c).sum(0) for c in range(8)],axis=1)
        _majority=_tally.argmax(1);_support=_tally.max(1)
        _change=(_majority!=_s18_labels)&(_support>=4)&~_s18_fixed
        _updated=_s18_labels.copy();_updated[_change]=_majority[_change]
        assert np.array_equal(_updated[_s18_fixed],_s18_initial[_s18_fixed])
        _counts=np.bincount(_updated,minlength=8)
        _s18_rounds.append({'round':_round,'changed_from_previous':int(_change.sum()),
            'changed_from_stage17':int((_updated!=_s18_initial).sum()),'fixed_automatic_anchors':int(_s18_fixed.sum()),
            'class_counts':dict(zip(_s18_classes,_counts.tolist()))})
        _s18_labels=_updated;_s18_last_predictions=_family_predictions
        print('Train-only refinement round:',json.dumps(_s18_rounds[-1]),flush=True)
        if int(_change.sum())<.01*_s18_n: break

# Apply the 37 team-reviewed train seeds after the automatic stage; keep this overlay explicit.
_s18_audit_seed_overlay=_s18_labels.copy();_s18_seed_mask=np.zeros(_s18_n,dtype=bool)
_s18_seed_pos={sid:i for i,sid in enumerate(sem_ids)}
for _sid,_category in SEED_LABELS.items():
    _pos=_s18_seed_pos.get(str(_sid))
    if _pos is None: continue
    _s18_audit_seed_overlay[_pos]=_s18_classes.index(_category);_s18_seed_mask[_pos]=True
_s18_support=np.zeros(_s18_n,dtype=np.uint8)
if _s18_last_predictions is not None:
    _last_tally=np.stack([(_s18_last_predictions==c).sum(0) for c in range(8)],axis=1)
    _s18_support=_last_tally.max(1).astype(np.uint8)
_s18_vote_class=_last_tally.argmax(1).astype(np.int32) if _s18_last_predictions is not None else _s18_labels.copy()
_s18_pseudo_accept=(_s18_support>=4)&(_s18_vote_class!=_s18_initial)&(~_s18_fixed)
_s18_pseudo_table=pd.DataFrame({
    'image_id':sem_ids,'source_sha256':sem_source_hash,'source_group':_s18_groups,'outer_fold':_s18_fold_ids,
    'stage17_class':np.asarray(_s18_classes)[_s18_initial],
    'crossfit_vote_class':np.asarray(_s18_classes)[_s18_vote_class],
    'crossfit_family_support':_s18_support,
    'pseudo_label_candidate':np.where(_s18_pseudo_accept,np.asarray(_s18_classes)[_s18_vote_class],''),
    'pseudo_label_accepted_train_only':_s18_pseudo_accept,
    'final_automatic_class':np.asarray(_s18_classes)[_s18_labels],
    'fixed_automatic_anchor':_s18_fixed,
    'seed_label_overlay_audit_only':_s18_seed_mask
})
_s18_pseudo_table.to_csv(STAGE18_DIR/'stage18_train_only_pseudo_label_decisions.csv',index=False)
_s18_table=pd.DataFrame({'image_id':sem_ids,'source_sha256':sem_source_hash,
    'stage17_class':np.asarray(_s18_classes)[_s18_initial],
    'automatic_refined_class':np.asarray(_s18_classes)[_s18_labels],
    'audit_seed_overlay_class':np.asarray(_s18_classes)[_s18_audit_seed_overlay],
    'automatic_confidence':_s18_initial_conf,'independent_source_votes':_s18_ranked[:,-1],
    'fixed_automatic_anchor':_s18_fixed,'four_of_five_support':_s18_support,
    'seed_overlay_applied_for_audit_only':_s18_seed_mask,'source_group':_s18_groups,'fold':_s18_fold_ids})
_s18_table.to_csv(STAGE18_DIR/'train_assignment_candidate.csv',index=False)
_s18_report={'method':'stage17_five_family_iterative_crossfit_denoising_v1','rows':_s18_n,
 'test_rows_processed':0,'assignment_npz_inputs_read':False,'active_label_files_read':False,
 'automatic_source_anchors':int(_s18_auto_anchor.sum()),'automatic_core_rows':int(_s18_core.sum()),
  'initial_class_counts':dict(zip(_s18_classes,_s18_initial_counts.tolist())),
  'core_anchor_class_counts':dict(zip(_s18_classes,_s18_core_counts.tolist())),
 'fixed_automatic_anchors':int(_s18_fixed.sum()),'seed_ids_in_audit_overlay':int(_s18_seed_mask.sum()),
 'train_only_pseudo_label_candidates':int(_s18_pseudo_accept.sum()),
 'seed_overlay_changed_automatic_labels':int(np.sum(_s18_seed_mask & (_s18_audit_seed_overlay!=_s18_labels))),
 'pseudo_label_output':'stage18_train_only_pseudo_label_decisions.csv',
 'family_oof_metrics_first_round':_s18_first_metrics,'rounds':_s18_rounds,
 'recipes':_s18_recipes,'stop_fraction':.01,'required_family_agreement':4,
 'class_counts_before_seed_overlay':dict(zip(_s18_classes,np.bincount(_s18_labels,minlength=8).tolist())),
 'class_counts_after_seed_audit_overlay':dict(zip(_s18_classes,np.bincount(_s18_audit_seed_overlay,minlength=8).tolist())),
 'source_fingerprint':sem_content_fingerprint,'seconds':time.time()-_s18_started,
 'semantic_accuracy':None,'active_assignment_parity_verified':False}
np.savez_compressed(STAGE18_DIR/'train_assignment_candidate.npz',image_ids=sem_ids,source_sha256=sem_source_hash,
    stage17_labels=_s18_initial,automatic_labels=_s18_labels,audit_seed_overlay_labels=_s18_audit_seed_overlay,
    source_group=_s18_groups,fold=_s18_fold_ids,fixed_anchor=_s18_fixed,team_seed_applied=_s18_seed_mask,
    family_support=_s18_support,class_order=np.asarray(_s18_classes))
(STAGE18_DIR/'run_report.json').write_text(json.dumps(_s18_report,indent=2,ensure_ascii=False),encoding='utf-8')
pd.DataFrame(_s18_first_metrics).to_json(STAGE18_DIR/'first_round_family_metrics.json',orient='records',indent=2)
print('Stage18 raw-train assignment candidate ready:',json.dumps({k:v for k,v in _s18_report.items() if k not in ['family_oof_metrics_first_round','recipes','source_fingerprint']},ensure_ascii=False),flush=True)


## 5. Membentuk pool train dan assignment algoritmis

Stage18 menyimpan vote cross-fit per ID. Pool supervised memakai seluruh foto utuh train yang eligible (sekitar 1.800). ID 0018, 0604, dan 0376 dipertahankan memakai assignment algoritmis notebook, tanpa kelas manual. 32 ID residual ditahan dari fit, bersama kebijakan hold lama. Pada CPU notebook menjalankan DINOv2 + RBF SVM; pada GPU jalur hybrid dan fine-tuning tetap tersedia. Dukungan cross-fit tetap dicatat sebagai diagnostik, bukan syarat masuk fit. Daftar ID dan jumlah aktual ditulis dalam laporan run.

Flag kesiapan memeriksa bahwa pool train, delapan kelas, crop, manifest test, dan urutan sample tersedia. Training, evaluasi, inferensi, dan ekspor submission berjalan terhadap target algoritmis yang dibangun pipeline.

In [ ]:
# Materialisasi target train-only untuk satu alur Kaggle: unanimous cross-fit + seed tim.
# Daftar acuan/parity tidak dibaca. Contoh berkeyakinan di bawah 5/5 tetap di luar fit.
from pathlib import Path
from shutil import copyfile

assert set(_s18_classes)==set(CLASSES), (_s18_classes,CLASSES)  # Stage18 stores class strings; internal order may differ.
assert len(_s18_table)==len(raw_train_paths)==1920
assert int(candidates.is_crop.sum())==7
_s18_auto_map=dict(zip(_s18_table.image_id.astype(str),_s18_table.automatic_refined_class.astype(str)))
_s18_support_map=dict(zip(_s18_table.image_id.astype(str),_s18_table.four_of_five_support.astype(int)))
_seed_map={norm_id(sid):cat for sid,cat in SEED_LABELS.items()}
_group_map={norm_id(sid):str(group) for sid,group in zip(ordered_train_ids,coarse_groups)}
_raw_hash_map={norm_id(r['image_id']):str(r['file_sha256']) for r in manifest.query("split == 'train'").to_dict('records')}

_train_candidates=candidates.query("split == 'train'").copy().reset_index(drop=True)
_train_candidates['parent_id_norm']=_train_candidates.parent_id.map(norm_id)
_train_candidates['crossfit_family_support']=_train_candidates.parent_id_norm.map(_s18_support_map)
_train_candidates['automatic_candidate_class']=_train_candidates.parent_id_norm.map(_s18_auto_map)
_train_candidates['seed_label']=_train_candidates.parent_id_norm.map(_seed_map)
_train_candidates['force_include']=_train_candidates.parent_id_norm.isin(RESIDUAL_FORCE_INCLUDE)
_train_candidates['residual_hold']=_train_candidates.parent_id_norm.isin(RESIDUAL_HOLD)
_train_candidates['assigned_class']=_train_candidates.seed_label.fillna(_train_candidates.automatic_candidate_class)
_train_candidates['label_source']=np.select(
    [_train_candidates.seed_label.notna(),_train_candidates.force_include],
    ['participant_reviewed_seed','participant_requested_keep_existing_algorithmic_assignment'],
    default='stage18_unanimous_crossfit')
_train_candidates['source_sha256']=_train_candidates.parent_id_norm.map(_raw_hash_map)
_train_candidates['source_group']=_train_candidates.parent_id_norm.map(_group_map)
assert _train_candidates.assigned_class.notna().all()
assert _train_candidates.source_sha256.notna().all() and _train_candidates.source_group.notna().all()
assert _train_candidates.automatic_candidate_class.isin(CLASSES).all()

# Train on every eligible train view, including algorithmic assignments with lower cross-fit support.
# The explicit 32-image hold list and the prior hold policy are excluded by training_eligible.
_eligible_candidates=_train_candidates[_train_candidates.training_eligible.astype(bool)].copy()
if DEVICE.type=='cpu':
    _eligible_candidates=_eligible_candidates[~_eligible_candidates.is_crop.astype(bool)].copy()
_eligible_candidates['used_for_supervised_fit']=True
train_frame=_eligible_candidates.copy().reset_index(drop=True)
low_consensus_train_frame=_eligible_candidates.iloc[0:0].copy()
_train_candidates['used_for_supervised_fit']=False
_train_candidates.loc[_eligible_candidates.index,'used_for_supervised_fit']=_eligible_candidates['used_for_supervised_fit'].to_numpy()
assert train_frame.sample_id.is_unique and len(train_frame)>1600
assert set(train_frame.assigned_class)==set(CLASSES),train_frame.assigned_class.value_counts().to_dict()
assert train_frame.groupby('assigned_class').size().min()>=20
assert RESIDUAL_FORCE_INCLUDE <= set(train_frame.parent_id_norm.astype(str))
assert not RESIDUAL_HOLD & set(train_frame.parent_id_norm.astype(str))
if DEVICE.type=='cpu': assert len(train_frame)>=1800 and not train_frame.is_crop.any()
assert not train_frame.residual_hold.any()

# Materialize notebook-generated processed folders from the staging images.
for category in CLASSES:(PROCESSED_ROOT/'train'/category).mkdir(parents=True,exist_ok=True)
(PROCESSED_ROOT/'test').mkdir(parents=True,exist_ok=True)
train_output=[]
for row in train_frame.itertuples():
    dest=PROCESSED_ROOT/'train'/row.assigned_class/f'{row.sample_id}.png'
    copyfile(row.staging_path,dest);train_output.append(str(dest))
train_frame['output_path']=train_output
_test_candidates=candidates.query("split == 'test'").copy()
_test_candidates['image_id_norm']=_test_candidates.parent_id.map(norm_id)
assert set(_test_candidates.image_id_norm)==set(sample.image_id_norm)
_test_lookup={r.image_id_norm:r for r in _test_candidates.itertuples()}
test_rows=[]
for sid in sample.image_id_norm.astype(str):
    row=_test_lookup[sid];dest=PROCESSED_ROOT/'test'/f'{sid}.png';copyfile(row.staging_path,dest)
    test_rows.append({'sample_id':sid,'parent_id':row.parent_id,'source_path':row.source_path,'bbox':row.bbox,
                      'is_crop':False,'region_name':'','output_path':str(dest),'source_sha256':sha256_file(row.source_path)})
test_frame=pd.DataFrame(test_rows)

train_ids=train_frame.sample_id.astype(str).tolist();train_paths=train_frame.output_path.astype(str).tolist()
test_ids=sample.image_id_norm.astype(str).tolist();test_paths=test_frame.output_path.astype(str).tolist()
pseudo=np.asarray([CLASSES.index(x) for x in train_frame.assigned_class],dtype=np.int64)
groups=train_frame.source_group.astype(str).to_numpy()
train_frame['parent_group']=groups
eligible_idx=np.arange(len(train_frame),dtype=np.int64)
assert len(train_paths)==len(train_ids)==len(pseudo)==len(groups)
assert len(test_paths)==len(test_ids)==len(sample)==480
assert len(set(test_ids))==480 and test_ids==sample.image_id_norm.astype(str).tolist()
assert set(groups).issubset(set(map(str,coarse_groups)))

train_frame[['sample_id','parent_id','assigned_class','label_source','crossfit_family_support','source_group','source_sha256','is_crop']].to_csv(WORK_ROOT/'labels/train_target_assignments.csv',index=False)
_train_candidates[['sample_id','parent_id','assigned_class','label_source','automatic_candidate_class','crossfit_family_support','source_group','source_sha256','training_eligible','force_include','residual_hold','used_for_supervised_fit','is_crop']].to_csv(WORK_ROOT/'labels/all_train_pseudo_label_decisions.csv',index=False)
low_consensus_train_frame[['sample_id','parent_id','assigned_class','automatic_candidate_class','crossfit_family_support','source_group','source_sha256','is_crop']].to_csv(WORK_ROOT/'labels/low_consensus_train_prediction_queue.csv',index=False)
test_frame[['sample_id','parent_id','source_path','output_path','source_sha256']].to_csv(WORK_ROOT/'processed/test_manifest.csv',index=False)
counts=train_frame.assigned_class.value_counts().reindex(CLASSES,fill_value=0)
counts.rename('train_samples').to_csv(WORK_ROOT/'labels/train_distribution.csv')
STATE_SHA256=hashlib.sha256(json.dumps(train_frame[['sample_id','assigned_class','source_sha256','source_group']].astype(str).values.tolist(),ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
proof={'mode':'raw_dataset_to_submission_train_only_pseudolabel_v1','raw_train_images':len(raw_train_paths),'raw_test_images':len(raw_test_paths),
       'eligible_train_candidates':len(_eligible_candidates),'supervised_fit_rows':len(train_frame),'low_consensus_train_rows':len(low_consensus_train_frame),
       'human_seed_rows':int(train_frame.label_source.eq('participant_reviewed_seed').sum()),'fit_policy':'all eligible train views; cross-fit support retained as diagnostic, not an inclusion threshold',
       'residual_force_include_ids':sorted(RESIDUAL_FORCE_INCLUDE),'residual_force_include_count':len(RESIDUAL_FORCE_INCLUDE),
       'residual_hold_ids':sorted(RESIDUAL_HOLD),'residual_hold_count':len(RESIDUAL_HOLD),
       'previous_hold_policy_ids':sorted(set(POLICY['hold'])),'total_unique_held_ids':len(TRAIN_HOLD),
       'crops_enabled':True,'crop_rows_in_fit':int(train_frame.is_crop.sum()),'class_order':CLASSES,'class_counts':counts.to_dict(),
       'test_rows_in_fit':0,'external_assignment_files_read':False,'reference_parity_claimed':False,'state_sha256':STATE_SHA256}
(WORK_ROOT/'processed/processing_report.json').write_text(json.dumps(proof,indent=2,ensure_ascii=False),encoding='utf-8')
PIPELINE_READY=True
print('Train-only target pool:',json.dumps(proof,ensure_ascii=False),flush=True)
print('Test prediction rows:',len(test_frame),'| final CSV path:',WORK_ROOT/'DSC2026_Submission.csv')

## 6. Distribusi foto utuh dan audit kesamaan

Hasil berikut menampilkan himpunan foto utuh yang diproses. Crop sengaja ditunda; histogram berasal dari manifest processed, bukan kategori sample submission.

In [ ]:
if PIPELINE_READY:
    counts=train_frame.assigned_class.value_counts().reindex(CLASSES,fill_value=0)
    counts.rename('train_samples').to_csv(WORK_ROOT/'labels/train_distribution.csv')
    display(counts.to_frame('train_samples'))
    fig,ax=plt.subplots(figsize=(10,4)); counts.plot.bar(ax=ax,rot=25,title='Distribusi train hasil algoritma'); ax.set_ylabel('sampel'); plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/processed_train_distribution.png',dpi=150); plt.show(); plt.close(fig)
    fig,axs=plt.subplots(2,4,figsize=(14,7))
    for ax,category in zip(axs.flat,CLASSES):
        row=train_frame[train_frame.assigned_class.eq(category)].iloc[0]
        ax.imshow(rgb_source(row.output_path)); ax.set_title(f'{category} | {row.sample_id}'); ax.axis('off')
    plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/processed_class_preview.png',dpi=150); plt.show(); plt.close(fig)
    crops=train_frame[train_frame.is_crop]
    if len(crops):
        fig,axs=plt.subplots(len(crops),2,figsize=(8,3*len(crops)),squeeze=False)
        for axes,row in zip(axs,crops.itertuples()):
            axes[0].imshow(rgb_source(row.source_path)); axes[0].set_title(f'Train source {row.parent_id}')
            axes[1].imshow(rgb_source(row.output_path)); axes[1].set_title(f'{row.sample_id}: {row.assigned_class}')
            for ax in axes: ax.axis('off')
        plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/crop_preprocessing_preview.png',dpi=130); plt.show(); plt.close(fig)
    else:
        print('Crop belum dibuat: fase ini hanya memproses foto utuh.')
    print('Train eligible full-frame:',len(train_frame))
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 6. Kemiripan foto train-test (diagnostik)

Backbone EfficientNetV2-S menghitung representasi deskriptif untuk pencarian foto train terdekat. Fitur test tidak dipakai untuk fitting normalisasi, PCA, target, sampler, fold, atau pemilihan model. Hasil tetangga hanya diagnostik; training memakai pool train yang dibentuk notebook.

In [ ]:
if PIPELINE_READY:
    from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
    from torchvision import models,transforms
    from torchvision.transforms import InterpolationMode
    MEAN=(.485,.456,.406); STD=(.229,.224,.225); MODEL_IMAGE_SIZE=224
    class ImageTensor(Dataset):
        def __init__(self,paths,augment=False):
            self.paths=list(paths); ops=[transforms.Resize((MODEL_IMAGE_SIZE,MODEL_IMAGE_SIZE),interpolation=InterpolationMode.BICUBIC)]
            if augment:
                ops += [transforms.RandomHorizontalFlip(.5),transforms.RandomAffine(8,translate=(.03,.03),scale=(.93,1.07),interpolation=InterpolationMode.BILINEAR,fill=(127,127,127)),transforms.RandomApply([transforms.ColorJitter(.12,.12,.12,.025)],p=.4)]
            self.tf=transforms.Compose(ops+[transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
        def __len__(self): return len(self.paths)
        def __getitem__(self,i):
            with Image.open(self.paths[i]) as im: return self.tf(im.convert('RGB'))
    try: embedding_net=models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1).to(DEVICE).eval()
    except Exception as error: raise RuntimeError('Bobot pretrained gagal diunduh. Aktifkan Internet Kaggle; bobot acak tidak digunakan.') from error
    def extract_embeddings(paths):
        loader=DataLoader(ImageTensor(paths),batch_size=24,shuffle=False,num_workers=2,pin_memory=True)
        chunks=[]
        with torch.inference_mode(),torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
            for xb in loader:
                xb=xb.to(DEVICE,non_blocking=True); z=embedding_net.avgpool(embedding_net.features(xb)).flatten(1)
                chunks.append(z.float().cpu().numpy())
        result=np.concatenate(chunks).astype('float32'); result/=np.maximum(np.linalg.norm(result,axis=1,keepdims=True),1e-12)
        return result
    train_emb=extract_embeddings(train_paths); test_emb=extract_embeddings(test_paths)
    np.savez_compressed(WORK_ROOT/'eda/processed_embeddings.npz',train=train_emb,test=test_emb,train_ids=np.array(train_ids),test_ids=np.array(test_ids))
    similarity=test_emb@train_emb.T
    nearest=np.argsort(-similarity,axis=1)[:,:3]
    from scipy.fft import dctn
    from skimage.metrics import structural_similarity
    def pixel_descriptor(path):
        with Image.open(path) as im:
            gray=im.convert('L')
            small=np.asarray(gray.resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
            low=dctn(small,type=2,norm='ortho')[:8,:8]; bits=(low>np.median(low)).reshape(-1)
            pix=np.asarray(gray.resize((64,64),Image.Resampling.LANCZOS),dtype=np.float32)/255.
        return bits,pix
    train_desc=[pixel_descriptor(p) for p in train_paths]; test_desc=[pixel_descriptor(p) for p in test_paths]
    train_phash=np.stack([p[0] for p in train_desc]); test_phash=np.stack([p[0] for p in test_desc])
    train_pixels=np.stack([p[1] for p in train_desc]); test_pixels=np.stack([p[1] for p in test_desc])
    neighbor_rows=[]
    for q,nn in enumerate(nearest):
        for rank,j in enumerate(nn,1):
            neighbor_rows.append({'test_id':test_ids[q],'rank':rank,'train_sample_id':train_ids[j],'train_class':train_frame.iloc[j].assigned_class,'cosine':float(similarity[q,j]),'phash_hamming':int(np.count_nonzero(test_phash[q]!=train_phash[j])),'grayscale_mae_64':float(np.abs(test_pixels[q]-train_pixels[j]).mean()),'ssim_gray_64':float(structural_similarity(test_pixels[q],train_pixels[j],data_range=1.)),'train_group':groups[j],'train_native_width':int(train_frame.iloc[j].bbox[2]-train_frame.iloc[j].bbox[0]),'train_native_height':int(train_frame.iloc[j].bbox[3]-train_frame.iloc[j].bbox[1]),'test_native_width':int(test_frame.iloc[q].bbox[2]-test_frame.iloc[q].bbox[0]),'test_native_height':int(test_frame.iloc[q].bbox[3]-test_frame.iloc[q].bbox[1])})
    pd.DataFrame(neighbor_rows).to_csv(WORK_ROOT/'eda/processed_train_test_neighbors.csv',index=False)
    phash_nearest=[]
    for q in range(len(test_paths)):
        distance=np.count_nonzero(train_phash!=test_phash[q],axis=1); j=int(distance.argmin())
        phash_nearest.append({'test_id':test_ids[q],'train_sample_id':train_ids[j],'phash_hamming':int(distance[j]),'embedding_cosine':float(similarity[q,j]),'candidate_near_duplicate':bool(distance[j]<=4)})
    pd.DataFrame(phash_nearest).to_csv(WORK_ROOT/'eda/processed_phash_nearest.csv',index=False)
    query_positions=np.argsort(-similarity.max(axis=1))[:8]
    fig,axs=plt.subplots(len(query_positions),2,figsize=(8,3*len(query_positions)),squeeze=False)
    for axes,q in zip(axs,query_positions):
        j=int(nearest[q,0]); axes[0].imshow(rgb_source(test_paths[q])); axes[0].set_title(f'Test {test_ids[q]} (no target)')
        axes[1].imshow(rgb_source(train_paths[j])); axes[1].set_title(f'Train {train_ids[j]}: {train_frame.iloc[j].assigned_class} | cos {similarity[q,j]:.3f}')
        for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.savefig(WORK_ROOT/'figures/processed_train_test_neighbors.png',dpi=130); plt.show(); plt.close(fig)
    del embedding_net; gc.collect(); torch.cuda.empty_cache()
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 7. Source-group cross-validation dan GPU training

Training memakai pool target train-only yang dibentuk di sel sebelumnya. Fold dikelompokkan berdasarkan source group. Setiap model menjalani 8 epoch/fold, dengan checkpoint terbaik berdasarkan macro-F1 proxy. Refit final memakai 3 epoch head + 18 epoch fine-tuning.

Empat backbone memakai AMP, class-balanced sampler, MixUp, label smoothing, dan cosine learning rate; kedua GPU dipakai bila runtime mendukung DataParallel. Metrik OOF mengukur agreement dengan assignment algoritmis, bukan akurasi label manusia independen. GPU Kaggle dan Internet dibutuhkan untuk jalur training/pretrained.

In [ ]:
if PIPELINE_READY:
    if not globals().get('PIPELINE_READY', False):
        raise RuntimeError('Classifier training ditunda: train-only target pool belum dibentuk dan divalidasi.')
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,log_loss
    from torch.nn import functional as F
    MODEL_NAMES=['resnet50','efficientnet_v2_s','convnext_small','swin_v2_t'] if DEVICE.type=='cuda' else []
    HEAD_EPOCHS=3 if DEVICE.type=='cuda' else 0; FINETUNE_EPOCHS=18 if DEVICE.type=='cuda' else 0
    FOLD_EPOCHS=8 if DEVICE.type=='cuda' else 0; N_SPLITS=3; BATCH_SIZE=24 if DEVICE.type=='cuda' else 8
    WORKERS=2 if DEVICE.type=='cuda' else 0
    USE_MULTI_GPU=DEVICE.type=='cuda' and torch.cuda.device_count()>1; RUN_START=time.time()
    TRAIN_CONFIG={'device':DEVICE.type,'classical_cpu_fallback':DEVICE.type=='cpu','models':MODEL_NAMES,'head_epochs':HEAD_EPOCHS,'finetune_epochs':FINETUNE_EPOCHS,'fold_epochs':FOLD_EPOCHS,'folds':N_SPLITS,'batch_size':BATCH_SIZE,'model_size':MODEL_IMAGE_SIZE,'seed':SEED,'state_sha256':STATE_SHA256,'torch':str(torch.__version__),'torchvision':str(torchvision.__version__)}
    CONFIG_HASH=hashlib.sha256(json.dumps(TRAIN_CONFIG,sort_keys=True).encode()).hexdigest()
    (WORK_ROOT/'logs/train_config.json').write_text(json.dumps(TRAIN_CONFIG,indent=2),encoding='utf-8')
    def core_model(model): return model.module if isinstance(model,torch.nn.DataParallel) else model
    def get_head(model,name):
        m=core_model(model)
        if name=='resnet50': return m.fc
        if name=='efficientnet_v2_s': return m.classifier[1]
        if name=='convnext_small': return m.classifier[2]
        if name=='swin_v2_t': return m.head
        raise ValueError(f'Unknown model: {name}')
    def build_model(name,pretrained=True):
        if name=='resnet50':
            m=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None); m.fc=torch.nn.Linear(m.fc.in_features,len(CLASSES))
        elif name=='efficientnet_v2_s':
            m=models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None); m.classifier[1]=torch.nn.Linear(m.classifier[1].in_features,len(CLASSES))
        elif name=='convnext_small':
            m=models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None); m.classifier[2]=torch.nn.Linear(m.classifier[2].in_features,len(CLASSES))
        elif name=='swin_v2_t':
            m=models.swin_v2_t(weights=models.Swin_V2_T_Weights.IMAGENET1K_V1 if pretrained else None); m.head=torch.nn.Linear(m.head.in_features,len(CLASSES))
        else: raise ValueError(name)
        return m
    class LabeledDataset(Dataset):
        def __init__(self,indices,augment=False):
            self.indices=np.asarray(indices); self.base=ImageTensor([train_paths[j] for j in self.indices],augment)
        def __len__(self): return len(self.indices)
        def __getitem__(self,k):
            j=int(self.indices[k]); return self.base[k],int(pseudo[j]),train_ids[j]
    def worker_seed(worker_id):
        value=torch.initial_seed()%2**32; np.random.seed(value); random.seed(value)
    def make_train_loader(indices,seed):
        counts=np.bincount(pseudo[indices],minlength=len(CLASSES)); weights=1/np.maximum(counts,1)
        generator=torch.Generator().manual_seed(seed)
        sampler=WeightedRandomSampler(torch.tensor(weights[pseudo[indices]],dtype=torch.double),num_samples=len(indices),replacement=True,generator=generator)
        return DataLoader(LabeledDataset(indices,True),batch_size=BATCH_SIZE,sampler=sampler,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda',worker_init_fn=worker_seed,generator=generator)
    def predict_loader(model,loader):
        model.eval(); chunks=[]; labels=[]; ids=[]
        with torch.inference_mode(),torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
            for batch in loader:
                xb=batch[0] if isinstance(batch,(list,tuple)) else batch
                chunks.append(model(xb.to(DEVICE,non_blocking=True)).float().softmax(1).cpu().numpy())
                if isinstance(batch,(list,tuple)):
                    labels.extend(batch[1].numpy().tolist()); ids.extend(list(batch[2]))
        return np.concatenate(chunks),labels,ids
    def train_epoch(model,loader,opt,scaler,head_only,name):
        model.train()
        if head_only: model.eval(); get_head(model,name).train()
        loss_sum=0.; n=0
        for xb,yb,_ in loader:
            xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16):
                if np.random.random()<.3 and len(xb)>1:
                    lam=float(np.random.beta(.15,.15)); perm=torch.randperm(len(xb),device=DEVICE)
                    logits=model(lam*xb+(1-lam)*xb[perm])
                    loss=lam*F.cross_entropy(logits,yb,label_smoothing=.025)+(1-lam)*F.cross_entropy(logits,yb[perm],label_smoothing=.025)
                else: loss=F.cross_entropy(model(xb),yb,label_smoothing=.025)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),2.0); scaler.step(opt); scaler.update()
            loss_sum+=float(loss.detach().cpu())*len(yb); n+=len(yb)
        return loss_sum/max(n,1)
    def load_checkpoint(path): return torch.load(path,map_location='cpu',weights_only=False)
    def atomic_save(obj,path):
        tmp=path.with_suffix('.tmp'); torch.save(obj,tmp); os.replace(tmp,path)
    def fit_one(name,tr_indices,va_indices,epochs,tag):
        seed=SEED+MODEL_NAMES.index(name)*100+sum(tag.encode()); seed_everything(seed)
        best_path=WORK_ROOT/f'checkpoints/{tag}_{name}_best.pt'
        final_path=WORK_ROOT/f'checkpoints/{tag}_{name}_final.pt'
        resume_path=WORK_ROOT/f'checkpoints/{tag}_{name}_resume.pt'
        done_path=WORK_ROOT/f'checkpoints/{tag}_{name}_done.json'
        chosen=best_path if va_indices is not None else final_path
        if done_path.exists() and chosen.exists():
            info=json.loads(done_path.read_text()); assert info['config_hash']==CONFIG_HASH,'Existing completed model has a different configuration.'
            ckpt=load_checkpoint(chosen); model=build_model(name,False); model.load_state_dict(ckpt['state']); model=model.to(DEVICE)
            if USE_MULTI_GPU and DEVICE.type=='cuda' and torch.cuda.device_count()>1: model=torch.nn.DataParallel(model)
            print('Reusing completed model:',tag,name); return model,pd.read_csv(WORK_ROOT/f'logs/{tag}_{name}_history.csv').to_dict('records'),chosen
        resume=load_checkpoint(resume_path) if resume_path.exists() else None
        if resume: assert resume['config_hash']==CONFIG_HASH,'Resume data/config mismatch.'
        model=build_model(name,pretrained=resume is None)
        if resume: model.load_state_dict(resume['state'])
        model=model.to(DEVICE)
        if USE_MULTI_GPU and DEVICE.type=='cuda' and torch.cuda.device_count()>1: model=torch.nn.DataParallel(model)
        scaler=torch.amp.GradScaler(DEVICE.type,enabled=DEVICE.type=='cuda')
        history=resume['history'] if resume else []; completed=int(resume['completed']) if resume else 0; best=float(resume['best']) if resume else -1.
        if resume:
            random.setstate(resume['rng_python']); np.random.set_state(resume['rng_numpy']); torch.set_rng_state(resume['rng_torch'])
            if DEVICE.type=='cuda': torch.cuda.set_rng_state_all(resume['rng_cuda'])
            scaler.load_state_dict(resume['scaler'])
        val_loader=DataLoader(LabeledDataset(va_indices),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda') if va_indices is not None else None
        stages=[('head',min(HEAD_EPOCHS,epochs)),('finetune',max(0,epochs-HEAD_EPOCHS))]
        start=0
        for stage,count in stages:
            if count==0: continue
            stop=start+count
            if completed>=stop: start=stop; continue
            for param in model.parameters(): param.requires_grad=stage!='head'
            for param in get_head(model,name).parameters(): param.requires_grad=True
            head_params=list(get_head(model,name).parameters()); head_ids={id(p) for p in head_params}
            if stage=='head': param_groups=[{'params':head_params,'lr':1e-3}]
            else: param_groups=[{'params':[p for p in model.parameters() if id(p) not in head_ids],'lr':2e-5},{'params':head_params,'lr':1e-4}]
            opt=torch.optim.AdamW(param_groups,weight_decay=1e-4); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=count,eta_min=1e-6)
            if resume and resume['stage']==stage and completed>start:
                opt.load_state_dict(resume['optimizer']); scheduler.load_state_dict(resume['scheduler'])
                # Optimizer tensors are moved explicitly for CPU-map-location resume.
                for values in opt.state.values():
                    for key,value in values.items():
                        if torch.is_tensor(value): values[key]=value.to(DEVICE)
            for epoch in range(max(completed,start),stop):
                epoch_start=time.time(); loader=make_train_loader(tr_indices,seed+epoch)
                loss=train_epoch(model,loader,opt,scaler,stage=='head',name)
                row={'model':name,'tag':tag,'stage':stage,'epoch':epoch+1,'train_loss':loss,'lr_backbone_or_head':float(opt.param_groups[0]['lr']),'seconds':round(time.time()-epoch_start,2)}
                if val_loader is not None:
                    pr,yy,_=predict_loader(model,val_loader); prediction=pr.argmax(1)
                    row.update(val_proxy_agreement=float(accuracy_score(yy,prediction)),val_proxy_macro_f1=float(f1_score(yy,prediction,labels=list(range(len(CLASSES))),average='macro',zero_division=0)),val_proxy_logloss=float(log_loss(yy,pr,labels=list(range(len(CLASSES))))))
                    if row['val_proxy_macro_f1']>best:
                        best=row['val_proxy_macro_f1']; atomic_save({'state':core_model(model).state_dict(),'name':name,'epoch':epoch+1,'config_hash':CONFIG_HASH,'classes':CLASSES},best_path)
                scheduler.step(); history.append(row); completed=epoch+1
                pd.DataFrame(history).to_csv(WORK_ROOT/f'logs/{tag}_{name}_history.csv',index=False)
                atomic_save({'state':core_model(model).state_dict(),'name':name,'stage':stage,'completed':completed,'best':best,'history':history,'optimizer':opt.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [],'config_hash':CONFIG_HASH},resume_path)
                print(row,flush=True)
            start=stop
        if va_indices is None: atomic_save({'state':core_model(model).state_dict(),'name':name,'epoch':completed,'config_hash':CONFIG_HASH,'classes':CLASSES},final_path)
        model.load_state_dict({('module.'+k if isinstance(model,torch.nn.DataParallel) else k):v for k,v in load_checkpoint(chosen)['state'].items()})
        done_path.write_text(json.dumps({'config_hash':CONFIG_HASH,'epochs':completed,'checkpoint':str(chosen)}),encoding='utf-8')
        if resume_path.exists(): resume_path.unlink()
        return model,history,chosen

    splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
    folds=list(splitter.split(np.zeros(len(train_frame)),pseudo,groups)); fold_ids=np.full(len(train_frame),-1)
    for fold,(tr,va) in enumerate(folds):
        assert set(groups[tr]).isdisjoint(set(groups[va]))
        assert set(pseudo[tr])==set(range(len(CLASSES)))
        fold_ids[va]=fold
    assert (fold_ids>=0).all()
    fold_table=train_frame[['sample_id','parent_id','parent_group','assigned_class','is_crop']].copy(); fold_table['fold']=fold_ids; fold_table.to_csv(WORK_ROOT/'validation/folds.csv',index=False)

    def write_submission(probabilities,path):
        assert probabilities.shape==(len(test_ids),len(CLASSES)) and np.isfinite(probabilities).all()
        assert np.allclose(probabilities.sum(1),1,atol=1e-4)
        pred=pd.DataFrame({'image_id_norm':test_ids,'waste_category':[CLASSES[j] for j in probabilities.argmax(1)]})
        out=sample[['image_id','image_id_norm']].merge(pred,on='image_id_norm',how='left',validate='one_to_one',sort=False)
        assert out.image_id.tolist()==sample.image_id.tolist()
        assert len(out)==480 and out.waste_category.notna().all() and set(out.waste_category)<=set(CLASSES)
        out=out[['image_id','waste_category']]; out.to_csv(path,index=False)
        return out
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 8. DINOv3 dan fitur warna/bentuk

Jalur yang sudah digunakan pada eksperimen sebelumnya: DINOv3 Base image-only, checkpoint/revisi/hash dibekukan, descriptor CLS + patch valid + foreground mean + foreground std. Anggota train, label, group dan ROI tetap persis processed hasil Run All. View native ROI dipakai sebelum resize model agar resep fitur lama terjaga. Test hanya ditransformasi dengan encoder beku. HSV/Lab/spatial color, HOG dan LBP menambah informasi warna/bentuk bagi tree models.

In [ ]:
if PIPELINE_READY:
    if DEVICE.type=='cpu':
        from sklearn.preprocessing import normalize
        DINO2_REPOSITORY=DINO2_REPO; DINO2_REVISION=DINO2_REV
        _d2pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(train_keys.astype(str))}
        _raw_by_norm={norm_id(p.stem):p for p in raw_train_paths}
        _raw_hash_by_norm={sid:file_sha256(path) for sid,path in _raw_by_norm.items()}
        with np.load(ASSIGN_FEATURE_DIR/'dinov2_small_train_features.npz',allow_pickle=False) as _z:
            assert np.array_equal(_z['keys'],train_keys)
            assert np.array_equal(_z['source_sha256'],train_source_sha256)
            assert str(_z['revision'].item())==DINO2_REV
            _d2_all=_z['features'].copy()
        dino_train=normalize(np.stack([_d2_all[_d2pos[norm_id(sid)]] for sid in train_frame.parent_id_norm.astype(str)]))
        _test_source_paths=[Path(p) for p in test_frame.source_path.astype(str)]
        _test_sha=np.asarray([file_sha256(path) for path in _test_source_paths],dtype=str)
        _test_cache=WORK_ROOT/'features'/'dinov2_small_test_features.npz'
        _test_identity=hashlib.sha256((DINO2_REV+'|'+ '|'.join(test_frame.sample_id.astype(str))+'|'+ '|'.join(_test_sha)).encode()).hexdigest()
        _test_cache.parent.mkdir(parents=True,exist_ok=True)
        if _test_cache.exists():
            with np.load(_test_cache,allow_pickle=False) as _z:
                assert str(_z['identity'].item())==_test_identity
                dino_test=_z['features'].copy()
        else:
            dino_test,_,_=_extract_global_dino2(_test_source_paths,np.asarray(['test:'+str(x) for x in test_frame.sample_id],dtype=str),batch_size=8)
            np.savez_compressed(_test_cache,features=dino_test,identity=np.asarray(_test_identity),source_sha256=_test_sha)
        hand_train=np.zeros((len(train_frame),1),dtype=np.float32);hand_test=np.zeros((len(test_frame),1),dtype=np.float32)
        assert dino_train.shape==(len(train_frame),384) and dino_test.shape==(len(test_frame),384)
        DINO_PROVENANCE={'repository':DINO2_REPOSITORY,'revision':DINO2_REVISION,'feature_source':'raw train DINOv2 cache generated in this notebook','train_rows':len(train_frame),'test_rows':len(test_frame),'test_used_for_fit_or_selection':False,'device':'cpu','test_cache_identity':_test_identity}
        (WORK_ROOT/'logs/dino_feature_provenance.json').write_text(json.dumps(DINO_PROVENANCE,indent=2),encoding='utf-8')
        print('CPU DINOv2 embeddings:',dino_train.shape,'| test:',dino_test.shape)
    else:
            import importlib.metadata,subprocess,joblib
            DEPENDENCY_PINS={'timm':'1.0.27','xgboost':'3.1.2'}
            for package,required in DEPENDENCY_PINS.items():
                try: current=importlib.metadata.version(package)
                except importlib.metadata.PackageNotFoundError: current=None
                if current!=required:
                    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',f'{package}=={required}'])
            import timm,xgboost,cv2
            from huggingface_hub import hf_hub_download
            from sklearn.preprocessing import normalize
            from skimage.feature import hog,local_binary_pattern
            D3_REPOSITORY='timm/vit_base_patch16_dinov3.lvd1689m'
            D3_REVISION='c6a5fb7d12bbd3cf3b0079253141c3332aaed7da'
            D3_WEIGHT_SHA256='1f9ed8a2378d65e24bb710ba522ac9fa7be4e036d7aefb4384ce022833926332'
            D3_RECIPE='dinov3base_256_letterbox_valid_foreground4x768_v1'
            d3_weight=hf_hub_download(D3_REPOSITORY,'model.safetensors',revision=D3_REVISION)
            assert file_sha256(d3_weight)==D3_WEIGHT_SHA256,'DINOv3 checkpoint SHA256 mismatch.'
            feature_rows=pd.concat([train_frame,test_frame],ignore_index=True)
            def native_view(row):
                im=rgb_source(row['source_path'])
                return im.crop(tuple(row['bbox'])) if row['is_crop'] else im
            def d3_prepare(row,mean,std):
                im=native_view(row); width,height=im.size; scale=256/max(width,height)
                nw,nh=max(1,round(width*scale)),max(1,round(height*scale)); dx,dy=(256-nw)//2,(256-nh)//2
                canvas=np.empty((256,256,3),np.uint8);canvas[:]=[124,116,104]
                resized=np.asarray(im.resize((nw,nh),Image.Resampling.BICUBIC));canvas[dy:dy+nh,dx:dx+nw]=resized
                valid=np.zeros((256,256),np.float32);valid[dy:dy+nh,dx:dx+nw]=1.
                border=np.concatenate([resized[0],resized[-1],resized[:,0],resized[:,-1]],axis=0).astype(np.float32)
                center=np.median(border,axis=0);dispersion=np.linalg.norm(border-center,axis=1);foreground=valid.copy()
                if float(np.quantile(dispersion,.75))<30:
                    distance=np.linalg.norm(resized.astype(np.float32)-center,axis=2)
                    mask=(distance>max(25.,float(np.quantile(dispersion,.95))+10.)).astype(np.uint8)
                    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
                    if .03<=float(mask.mean())<=.90:
                        foreground[:]=0;foreground[dy:dy+nh,dx:dx+nw]=mask.astype(np.float32);foreground+=.05*valid
                valid_patch=cv2.resize(valid,(16,16),interpolation=cv2.INTER_AREA).reshape(-1)
                foreground_patch=cv2.resize(foreground,(16,16),interpolation=cv2.INTER_AREA).reshape(-1)
                tensor=(canvas.astype(np.float32)/255.-mean[None,None,:])/std[None,None,:]
                return torch.from_numpy(tensor.transpose(2,0,1).copy()),valid_patch,foreground_patch
            def d3_descriptor_batch(encoder,prepared):
                images=torch.stack([a[0] for a in prepared]).to(DEVICE)
                tokens=encoder.forward_features(images);patches=tokens[:,encoder.num_prefix_tokens:]
                assert patches.shape[1:]==(256,768)
                valid=torch.as_tensor(np.stack([a[1] for a in prepared]),device=DEVICE);valid/=valid.sum(1,keepdim=True).clamp_min(1e-8)
                foreground=torch.as_tensor(np.stack([a[2] for a in prepared]),device=DEVICE);foreground/=foreground.sum(1,keepdim=True).clamp_min(1e-8)
                global_mean=(patches*valid[:,:,None]).sum(1);object_mean=(patches*foreground[:,:,None]).sum(1)
                object_std=(((patches-object_mean[:,None,:]).square()*foreground[:,:,None]).sum(1).clamp_min(0)).sqrt()
                return torch.cat([F.normalize(a,dim=1) for a in [tokens[:,0],global_mean,object_mean,object_std]],dim=1).cpu().numpy().astype(np.float32)
            def hand_descriptor(row):
                im=native_view(row);rgb=np.asarray(im.resize((128,128),Image.Resampling.LANCZOS),dtype=np.uint8)
                hsv=cv2.cvtColor(rgb,cv2.COLOR_RGB2HSV);lab=cv2.cvtColor(rgb,cv2.COLOR_RGB2LAB)
                blocks=[]
                def hist(values,bins,range_):
                    h=np.histogram(values,bins=bins,range=range_)[0].astype(np.float32);return h/max(h.sum(),1.)
                for channel,bins,upper in [(0,16,180),(1,8,256),(2,8,256)]:blocks.append(hist(hsv[:,:,channel],bins,(0,upper)))
                joint=np.histogram2d(hsv[:,:,0].ravel(),hsv[:,:,1].ravel(),bins=(8,4),range=((0,180),(0,256)))[0].reshape(-1).astype(np.float32);blocks.append(joint/max(joint.sum(),1.))
                for channel in range(3):blocks.append(hist(lab[:,:,channel],8,(0,256)))
                for yy in [0,64]:
                    for xx in [0,64]:
                        patch=hsv[yy:yy+64,xx:xx+64]
                        for channel,bins,upper in [(0,8,180),(1,4,256),(2,4,256)]:blocks.append(hist(patch[:,:,channel],bins,(0,upper)))
                arr=rgb.astype(np.float32)/255.;blocks.extend([arr.mean((0,1)),arr.std((0,1))])
                h,s,v=hsv[:,:,0],hsv[:,:,1],hsv[:,:,2]
                fractions=[((h>=4)&(h<=30)&(s>=45)&(v>=25)&(v<=225)).mean(),(s<30).mean(),(s>120).mean(),(v<40).mean(),(v>220).mean(),s.std()/255.,v.std()/255.]
                blocks.append(np.array(fractions,np.float32))
                gray=np.asarray(im.convert('L').resize((64,64),Image.Resampling.LANCZOS),dtype=np.uint8)
                blocks.append(hog(gray.astype(np.float32)/255.,orientations=9,pixels_per_cell=(16,16),cells_per_block=(2,2),block_norm='L2-Hys').astype(np.float32))
                blocks.append(hist(local_binary_pattern(gray,8,1,method='uniform'),10,(0,10)))
                return np.concatenate(blocks).astype(np.float32)
        
            cache=WORK_ROOT/'processed/dinov3_hybrid_features.npz'
            feature_identity=hashlib.sha256((STATE_SHA256+D3_WEIGHT_SHA256+D3_RECIPE+'native_roi_hand_v1').encode()).hexdigest()
            if cache.exists():
                with np.load(cache,allow_pickle=False) as zz:
                    assert str(zz['identity'].item())==feature_identity
                    d3_features=zz['dino'].copy();hand_features=zz['hand'].copy()
            else:
                encoder=timm.create_model('vit_base_patch16_dinov3.lvd1689m',pretrained=True,num_classes=0,pretrained_cfg_overlay={'file':str(d3_weight)}).eval().to(DEVICE)
                if hasattr(encoder,'rope') and hasattr(encoder.rope,'periods'):encoder.rope.periods=encoder.rope.periods.to(torch.bfloat16).to(torch.float32)
                cfg=timm.data.resolve_model_data_config(encoder);mean=np.asarray(cfg['mean'],np.float32);std=np.asarray(cfg['std'],np.float32)
                assert encoder.num_features==768
                records=feature_rows.to_dict('records');chunks=[];batch=8 if DEVICE.type=='cuda' else 2
                # FP32 preserves the established feature recipe; no local-label adaptation of this encoder.
                with torch.inference_mode():
                    for start in range(0,len(records),batch):
                        chunks.append(d3_descriptor_batch(encoder,[d3_prepare(row,mean,std) for row in records[start:start+batch]]))
                        if start%256==0:print('DINOv3 descriptors:',min(start+batch,len(records)),'/',len(records),flush=True)
                d3_features=np.concatenate(chunks);hand_features=np.stack([hand_descriptor(row) for row in records])
                np.savez_compressed(cache,dino=d3_features,hand=hand_features,identity=np.array(feature_identity))
                del encoder;gc.collect();torch.cuda.empty_cache()
            assert d3_features.shape==(len(feature_rows),3072) and np.isfinite(d3_features).all() and np.isfinite(hand_features).all()
            dino_train=normalize(d3_features[:len(train_frame)]);dino_test=normalize(d3_features[len(train_frame):])
            hand_train=hand_features[:len(train_frame)];hand_test=hand_features[len(train_frame):]
            DINO_PROVENANCE={'repository':D3_REPOSITORY,'revision':D3_REVISION,'weight_sha256':D3_WEIGHT_SHA256,'recipe':D3_RECIPE,'feature_identity':feature_identity,'train_rows':len(train_frame),'test_rows':len(test_frame),'tree_features':'train-fold PCA of DINO + HSV/Lab/spatial color + HOG/LBP','native_view':'same processed sample membership/ROI/label, native resolution before model resize','timm':timm.__version__,'xgboost':xgboost.__version__}
            (WORK_ROOT/'logs/dino_feature_provenance.json').write_text(json.dumps(DINO_PROVENANCE,indent=2),encoding='utf-8')
            print('DINO:',dino_train.shape,'handcrafted:',hand_train.shape)
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 9. SVM, group-kNN, Extra Trees, dan XGBoost

RBF-SVM dan kNN berbobot kelompok sumber menjadi baseline. Extra Trees dan XGBoost memakai PCA fitur DINO yang di-fit pada partisi train ditambah fitur warna/tekstur. Pilihan k, bobot tetangga, dan campuran tree dipilih lewat inner grouped validation; outer validation tidak ikut fitting atau pemilihan tersebut. Seluruh model mempertahankan assignment processed. Baseline, tree individual dan campurannya diekspor terpisah. CSV utama awal tersedia sebelum training CNN agar hasil klasik tidak hilang bila sesi berhenti kemudian.

In [ ]:
if PIPELINE_READY:
    if DEVICE.type=='cpu':
        import joblib
        from scipy.special import softmax
        from sklearn.metrics import accuracy_score,f1_score,confusion_matrix
        from sklearn.multiclass import OneVsRestClassifier
        from sklearn.svm import SVC
        from threadpoolctl import threadpool_limits
        from huggingface_hub import hf_hub_download
        from timm.data import resolve_model_data_config
        CLASSIC_SELECTED_NAME='dino2_dino3_blend'
        _d3pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(train_keys)}
        dino3_cpu_train=np.stack([dino3_train_features[_d3pos[sid]] for sid in train_ids]).astype(np.float32)
        assert dino3_cpu_train.shape==(len(train_frame),1536)
        def _cpu_dino_svm(x,y,seed):
            model=OneVsRestClassifier(SVC(C=4.,kernel='rbf',gamma='scale'),n_jobs=1)
            with threadpool_limits(limits=4): model.fit(x,y)
            return model
        def _metric_cpu(prob):
            pred=prob.argmax(1)
            return {'proxy_agreement':float(accuracy_score(pseudo,pred)),
                    'proxy_macro_f1':float(f1_score(pseudo,pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0))}
        _oof2=np.full((len(train_frame),len(CLASSES)),np.nan,dtype=np.float32)
        _oof3=np.full_like(_oof2,np.nan);classic_choices=[]
        for _fold,(_tr,_va) in enumerate(folds):
            assert set(groups[_tr]).isdisjoint(set(groups[_va]))
            _m2=_cpu_dino_svm(dino_train[_tr],pseudo[_tr],SEED+_fold)
            _m3=_cpu_dino_svm(dino3_cpu_train[_tr],pseudo[_tr],SEED+100+_fold)
            _oof2[_va]=softmax(_m2.decision_function(dino_train[_va]),axis=1).astype(np.float32)
            _oof3[_va]=softmax(_m3.decision_function(dino3_cpu_train[_va]),axis=1).astype(np.float32)
            classic_choices.append({'fold':_fold,'fit_rows':len(_tr),'validation_rows':len(_va)})
        assert np.isfinite(_oof2).all() and np.isfinite(_oof3).all()
        _blend_grid=[]
        for _w3 in (0.,.25,.5,.75,1.):
            _candidate=(1.-_w3)*_oof2+_w3*_oof3
            _blend_grid.append((_metric_cpu(_candidate)['proxy_agreement'],_metric_cpu(_candidate)['proxy_macro_f1'],-_w3,_w3,_candidate))
        _,_,_,_blend_weight3,_oof=max(_blend_grid,key=lambda z:(z[0],z[1],z[2]))
        _blend_weight2=1.-_blend_weight3
        classic_oof_probs={'dino2_rbf_C4':_oof2,'dino3_rbf_C4':_oof3,'dino2_dino3_blend':_oof}
        _metrics=[{'model':name,**_metric_cpu(prob)} for name,prob in classic_oof_probs.items()]
        classic_summaries=_metrics
        _pred=_oof.argmax(1)
        pd.DataFrame(_metrics).to_csv(WORK_ROOT/'validation/proxy_summary.csv',index=False)
        pd.DataFrame(_oof,columns=CLASSES).assign(sample_id=train_ids,fold=fold_ids,target=train_frame.assigned_class,predicted=[CLASSES[j] for j in _pred]).to_csv(WORK_ROOT/'validation/oof_dino2_dino3_blend.csv',index=False)
        pd.DataFrame(confusion_matrix(pseudo,_pred,labels=list(range(len(CLASSES)))),index=CLASSES,columns=CLASSES).to_csv(WORK_ROOT/'validation/confusion_dino2_dino3_blend.csv')
        _test_cache=WORK_ROOT/'features'/'dinov3_small_test_features.npz'
        _test_raw_paths=[Path(p) for p in test_frame.source_path.astype(str)]
        _test_raw_sha=np.asarray([_sha256_file(p) for p in _test_raw_paths],dtype=str)
        _test_identity=hashlib.sha256(json.dumps({'ids':test_ids,'source_sha256':_test_raw_sha.tolist(),'revision':DINO3S_REV,'recipe':'dinov3_small_valid_foreground_descriptors_v1'},sort_keys=True).encode()).hexdigest()
        if _test_cache.exists():
            with np.load(_test_cache,allow_pickle=False) as _z:
                assert str(_z['identity'].item())==_test_identity and np.array_equal(_z['source_sha256'],_test_raw_sha)
                dino3_cpu_test=_z['features'].astype(np.float32)
        else:
            _weight=hf_hub_download(DINO3S_REPO,'model.safetensors',revision=DINO3S_REV)
            _encoder=timm.create_model('vit_small_patch16_dinov3.lvd1689m',pretrained=True,num_classes=0,pretrained_cfg_overlay={'file':str(_weight)}).eval().to(DEVICE)
            _cfg=resolve_model_data_config(_encoder);_prefix=_encoder.num_prefix_tokens;_mean=_cfg['mean'];_std=_cfg['std'];_parts=[]
            with torch.inference_mode():
                for _start in range(0,len(_test_raw_paths),8):
                    _imgs=[];_valids=[];_fgs=[]
                    for _path in _test_raw_paths[_start:_start+8]:
                        _canvas,_valid,_fg=_dino3_small_input(_path)
                        _imgs.append(normalize(to_tensor(_canvas),_mean,_std));_valids.append(_valid);_fgs.append(_fg)
                    _x=torch.stack(_imgs).to(DEVICE,non_blocking=True)
                    with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda',dtype=torch.float16): _tokens=_encoder.forward_features(_x)
                    _patches=_tokens[:,_prefix:].float();_v=torch.from_numpy(np.stack(_valids)).to(DEVICE);_f=torch.from_numpy(np.stack(_fgs)).to(DEVICE)
                    _v=_v/_v.sum(1,keepdim=True).clamp_min(1e-8);_f=_f/_f.sum(1,keepdim=True).clamp_min(1e-8)
                    _mv=(_patches*_v[:,:,None]).sum(1);_mf=(_patches*_f[:,:,None]).sum(1)
                    _sf=((_patches-_mf[:,None,:]).square()*_f[:,:,None]).sum(1).clamp_min(0).sqrt()
                    _block=torch.cat([torch.nn.functional.normalize(v,dim=1) for v in [_tokens[:,0].float(),_mv,_mf,_sf]],1)
                    _parts.append(_block.cpu().numpy().astype(np.float32))
                    print('DINOv3 test features:',min(_start+8,len(_test_raw_paths)),'/',len(_test_raw_paths),flush=True)
            dino3_cpu_test=np.concatenate(_parts).astype(np.float32)
            assert dino3_cpu_test.shape==(len(test_ids),1536)
            np.savez_compressed(_test_cache,features=dino3_cpu_test,source_sha256=_test_raw_sha,identity=np.asarray(_test_identity))
            del _encoder,_parts;gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        _final2=_cpu_dino_svm(dino_train,pseudo,SEED+5000)
        _final3=_cpu_dino_svm(dino3_cpu_train,pseudo,SEED+6000)
        _prob2=softmax(_final2.decision_function(dino_test),axis=1).astype(np.float32)
        _prob3=softmax(_final3.decision_function(dino3_cpu_test),axis=1).astype(np.float32)
        _test_prob=_blend_weight2*_prob2+_blend_weight3*_prob3
        classic_final_probabilities={'dino2_rbf_C4':_prob2,'dino3_rbf_C4':_prob3,'dino2_dino3_blend':_test_prob}
        joblib.dump({'dino2':_final2,'dino3':_final3,'dino3_train':dino3_cpu_train,'y':pseudo,'groups':groups,'classes':CLASSES,'train_ids':train_ids,'blend_weights':{'dino2':_blend_weight2,'dino3':_blend_weight3}},WORK_ROOT/'checkpoints/final_dino2_dino3_blend.joblib',compress=3)
        pd.DataFrame(_test_prob,columns=CLASSES).assign(image_id=test_ids).to_csv(WORK_ROOT/'predictions/test_probabilities_dino2_dino3_blend.csv',index=False)
        _selected_metric=_metric_cpu(_oof)
        classic_report={'selected_family':CLASSIC_SELECTED_NAME,'device':'cpu','fit_rows':len(train_frame),'full_frame_rows':int((~train_frame.is_crop.astype(bool)).sum()),'outer_folds':len(folds),'outer_choices':classic_choices,'recipe':{'dino2':'DINOv2 small 384d + OVR RBF SVM C=4','dino3':'DINOv3 small 1536d valid/foreground descriptor + OVR RBF SVM C=4','probability_blend':{'dino2':_blend_weight2,'dino3':_blend_weight3}},'proxy_metrics':_selected_metric,'test_rows_in_fit':0}
        (WORK_ROOT/'logs/classical_selection.json').write_text(json.dumps(classic_report,indent=2),encoding='utf-8')
        print('CPU DINOv2/DINOv3 blend fit:',len(train_frame),'| grouped OOF agreement:',round(_selected_metric['proxy_agreement'],4),'| macro-F1:',round(_selected_metric['proxy_macro_f1'],4))
    else:
            from collections import Counter
            from scipy.special import softmax
            from sklearn.ensemble import ExtraTreesClassifier
            from sklearn.decomposition import PCA
            from sklearn.multiclass import OneVsRestClassifier
            from sklearn.svm import SVC
            from sklearn.utils.class_weight import compute_sample_weight
            from threadpoolctl import threadpool_limits
            K_GRID=(1,3,5,7,11);NEIGHBOR_BLEND_GRID=(0.,.1,.2,.3,.4);TREE_COUNT=240 if DEVICE.type=='cpu' else 600;XGB_ROUNDS=180 if DEVICE.type=='cpu' else 360;INNER_SPLITS=3
            TREE_RECIPES=[{'base':1.,'et':0.,'xgb':0.},{'base':.9,'et':.1,'xgb':0.},{'base':.9,'et':0.,'xgb':.1},{'base':.8,'et':.1,'xgb':.1},{'base':.8,'et':.2,'xgb':0.},{'base':.8,'et':0.,'xgb':.2},{'base':.6,'et':.2,'xgb':.2}]
            def scores(y,prob):
                pred=prob.argmax(1)
                return {'agreement':float(accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0))}
            def not_degraded(candidate,baseline):
                return candidate['agreement']>=baseline['agreement'] and candidate['macro_f1']>=baseline['macro_f1'] and (candidate['agreement']>baseline['agreement'] or candidate['macro_f1']>baseline['macro_f1'])
            def group_neighbors(qx,fx,fy,fit_groups,k):
                sim=qx@fx.T;k=min(k,len(fx));ix=np.argsort(-sim,axis=1,kind='stable')[:,:k]
                top=np.take_along_axis(sim,ix,axis=1);count=Counter(fit_groups.tolist())
                source_weight=np.array([1./count[g] for g in fit_groups]);w=np.exp((top-top.max(1,keepdims=True))/.05)*source_weight[ix]
                pp=np.column_stack([(w*(fy[ix]==c)).sum(1) for c in range(len(CLASSES))]);return pp/np.maximum(pp.sum(1,keepdims=True),1e-12)
            def fit_feature_models(x,h,y,g,qx,qh,seed):
                assert set(y)==set(range(len(CLASSES))),f'Classifier fit split is missing class IDs: {sorted(set(range(len(CLASSES)))-set(y))}'
                svm=OneVsRestClassifier(SVC(C=4.,kernel='rbf',gamma='scale'),n_jobs=1).fit(x,y)
                p_svm=softmax(svm.decision_function(qx),axis=1)
                # PCA is fitted inside the training partition, never on validation/test.
                pca=PCA(n_components=min(192,len(x)-1,x.shape[1]),svd_solver='randomized',random_state=seed)
                xt=np.column_stack([pca.fit_transform(x),h]).astype(np.float32);xq=np.column_stack([pca.transform(qx),qh]).astype(np.float32)
                counts=Counter(g.tolist());source_weights=np.array([1./counts[v] for v in g],np.float32)
                et=ExtraTreesClassifier(n_estimators=TREE_COUNT,min_samples_leaf=2,max_features=.6,class_weight='balanced',n_jobs=min(4,os.cpu_count() or 1),random_state=seed)
                et.fit(xt,y,sample_weight=source_weights)
                backend='cuda' if DEVICE.type=='cuda' else 'cpu'
                xgb=xgboost.XGBClassifier(n_estimators=XGB_ROUNDS,max_depth=4,learning_rate=.04,min_child_weight=2.,subsample=.85,colsample_bytree=.75,reg_lambda=8.,reg_alpha=.1,objective='multi:softprob',num_class=len(CLASSES),tree_method='hist',device=backend,max_bin=128,n_jobs=min(4,os.cpu_count() or 1),random_state=seed,eval_metric='mlogloss')
                weights=compute_sample_weight('balanced',y)*source_weights
                try:xgb.fit(xt,y,sample_weight=weights)
                except xgboost.core.XGBoostError as error:
                    if backend=='cuda' and ('cuda' in str(error).lower() or 'gpu' in str(error).lower()):
                        print('XGBoost CUDA fallback to CPU:',str(error)[:240],flush=True);xgb.set_params(device='cpu');xgb.fit(xt,y,sample_weight=weights)
                    else:raise
                assert np.array_equal(et.classes_,np.arange(len(CLASSES))) and np.array_equal(xgb.classes_,np.arange(len(CLASSES)))
                predictions={'svm':p_svm,'et':et.predict_proba(xq),'xgb':xgb.predict_proba(xq),'neighbors':{k:group_neighbors(qx,x,y,g,k) for k in K_GRID}}
                return predictions,{'svm':svm,'pca':pca,'extra_trees':et,'xgboost':xgb}
            def choose_knn_recipe(y,svm,neighbors):
                options=[]
                for k in K_GRID:
                    for weight in NEIGHBOR_BLEND_GRID:
                        metric=scores(y,(1-weight)*svm+weight*neighbors[k])
                        options.append((metric['agreement'],metric['macro_f1'],-weight,-k,k,weight))
                best=max(options);return {'k':int(best[4]),'weight':float(best[5])}
            def blend_trees(base,et,xgb,recipe):return recipe['base']*base+recipe['et']*et+recipe['xgb']*xgb
            def choose_tree_recipe(y,base,et,xgb):
                baseline=scores(y,base);chosen=TREE_RECIPES[0];best=baseline
                for recipe in TREE_RECIPES[1:]:
                    metric=scores(y,blend_trees(base,et,xgb,recipe))
                    if not_degraded(metric,baseline) and (metric['agreement'],metric['macro_f1'])>(best['agreement'],best['macro_f1']):chosen=recipe;best=metric
                return dict(chosen)
            def valid_inner_folds(y,g,seed):
                # Grouped stratification is approximate. Retry deterministic seeds solely for class coverage.
                for attempt in range(16):
                    used_seed=seed+attempt
                    cv=StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=used_seed)
                    candidate=list(cv.split(np.zeros(len(y)),y,g))
                    if all(set(y[tr])==set(range(len(CLASSES))) for tr,va in candidate):
                        for tr,va in candidate:assert set(g[tr]).isdisjoint(set(g[va]))
                        return candidate,used_seed
                raise ValueError('Cannot form source-group inner folds with all classes in each fit split.')
            def inner_classical_choice(indices,seed):
                y=pseudo[indices];g=groups[indices];n=len(indices)
                inner,inner_seed=valid_inner_folds(y,g,seed)
                sv=np.full((n,len(CLASSES)),np.nan);et=np.full_like(sv,np.nan);xg=np.full_like(sv,np.nan);neighbors={k:np.full_like(sv,np.nan) for k in K_GRID}
                with threadpool_limits(limits=4):
                    for fold,(tr,va) in enumerate(inner):
                        assert set(g[tr]).isdisjoint(set(g[va]))
                        result,bundle=fit_feature_models(dino_train[indices[tr]],hand_train[indices[tr]],y[tr],g[tr],dino_train[indices[va]],hand_train[indices[va]],seed+fold)
                        sv[va]=result['svm'];et[va]=result['et'];xg[va]=result['xgb']
                        for k in K_GRID:neighbors[k][va]=result['neighbors'][k]
                        del bundle;gc.collect()
                assert np.isfinite(sv).all() and np.isfinite(et).all() and np.isfinite(xg).all()
                neighbor_recipe=choose_knn_recipe(y,sv,neighbors)
                neighbor_recipe['inner_split_seed']=inner_seed
                base=(1-neighbor_recipe['weight'])*sv+neighbor_recipe['weight']*neighbors[neighbor_recipe['k']]
                tree_recipe=choose_tree_recipe(y,base,et,xg)
                return neighbor_recipe,tree_recipe
        
            classic_names=['dino_svm','dino_groupknn','extra_trees','xgboost','dino_tree_hybrid']
            classic_oof_probs={name:np.full((len(train_frame),len(CLASSES)),np.nan,np.float32) for name in classic_names};classic_choices=[]
            with threadpool_limits(limits=4):
                for fold,(tr,va) in enumerate(folds):
                    neighbor_recipe,tree_recipe=inner_classical_choice(tr,SEED+2000+fold)
                    result,bundle=fit_feature_models(dino_train[tr],hand_train[tr],pseudo[tr],groups[tr],dino_train[va],hand_train[va],SEED+3000+fold)
                    base=(1-neighbor_recipe['weight'])*result['svm']+neighbor_recipe['weight']*result['neighbors'][neighbor_recipe['k']]
                    classic_oof_probs['dino_svm'][va]=result['svm'];classic_oof_probs['dino_groupknn'][va]=base
                    classic_oof_probs['extra_trees'][va]=result['et'];classic_oof_probs['xgboost'][va]=result['xgb']
                    classic_oof_probs['dino_tree_hybrid'][va]=blend_trees(base,result['et'],result['xgb'],tree_recipe)
                    classic_choices.append({'fold':fold,'neighbor':neighbor_recipe,'tree':tree_recipe})
                    print('Classic nested outer fold',fold,classic_choices[-1],flush=True);del bundle;gc.collect()
            classic_summaries=[]
            for name,pp in classic_oof_probs.items():
                assert np.isfinite(pp).all() and np.allclose(pp.sum(1),1,atol=1e-4)
                metric=scores(pseudo,pp);classic_summaries.append({'model':name,'proxy_agreement':metric['agreement'],'proxy_macro_f1':metric['macro_f1']})
                pd.DataFrame(pp,columns=CLASSES).assign(sample_id=train_ids,fold=fold_ids,target=train_frame.assigned_class).to_csv(WORK_ROOT/f'validation/oof_{name}.csv',index=False)
            baseline_metric=scores(pseudo,classic_oof_probs['dino_groupknn']);hybrid_metric=scores(pseudo,classic_oof_probs['dino_tree_hybrid'])
            CLASSIC_SELECTED_NAME='dino_tree_hybrid' if not_degraded(hybrid_metric,baseline_metric) else 'dino_groupknn'
            full_neighbor,full_tree=inner_classical_choice(eligible_idx,SEED+4000)
            with threadpool_limits(limits=4):
                result,bundle=fit_feature_models(dino_train,hand_train,pseudo,groups,dino_test,hand_test,SEED+5000)
            base=(1-full_neighbor['weight'])*result['svm']+full_neighbor['weight']*result['neighbors'][full_neighbor['k']]
            classic_final_probabilities={'dino_svm':result['svm'],'dino_groupknn':base,'extra_trees':result['et'],'xgboost':result['xgb'],'dino_tree_hybrid':blend_trees(base,result['et'],result['xgb'],full_tree)}
            bundle['xgboost'].save_model(WORK_ROOT/'checkpoints/final_xgboost.json')
            joblib.dump({k:v for k,v in bundle.items() if k!='xgboost'},WORK_ROOT/'checkpoints/final_dino_tree_models.joblib',compress=3)
            np.savez_compressed(WORK_ROOT/'checkpoints/final_dino_knn_fit.npz',x=dino_train,y=pseudo,groups=groups,classes=np.asarray(CLASSES))
            for name,pp in classic_final_probabilities.items():
                write_submission(pp,WORK_ROOT/f'predictions/submission_{name}.csv')
                pd.DataFrame(pp,columns=CLASSES).assign(image_id=test_ids).to_csv(WORK_ROOT/f'predictions/test_probabilities_{name}.csv',index=False)
            write_submission(classic_final_probabilities[CLASSIC_SELECTED_NAME],WORK_ROOT/'DSC2026_Submission.csv')
            classic_report={'selected_family':CLASSIC_SELECTED_NAME,'outer_folds':N_SPLITS,'inner_folds':INNER_SPLITS,'outer_choices':classic_choices,'full_train_neighbor':full_neighbor,'full_train_tree':full_tree,'tree_count':TREE_COUNT,'xgb_rounds':XGB_ROUNDS,'xgb_actual_device':json.loads(bundle['xgboost'].get_booster().save_config()).get('learner',{}).get('generic_param',{}).get('device'),'pca':'192 components max, fitted independently within every fit split','selection_rule':'train-only inner validation; retain group-kNN baseline unless tree hybrid is nondegraded on paired outer agreement and macro-F1','outer_metrics':classic_summaries,'data_state_sha256':STATE_SHA256,'test_rows_in_fit':0,'probability_note':'SVM softmax margins are ranking scores, not calibrated confidence'}
            (WORK_ROOT/'logs/classical_selection.json').write_text(json.dumps(classic_report,indent=2),encoding='utf-8')
            display(pd.DataFrame(classic_summaries));print('Classical submission already saved; selected:',CLASSIC_SELECTED_NAME)
            del bundle;gc.collect();torch.cuda.empty_cache()
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 10. Evaluasi CNN pada source fold yang sama

Empat CNN dilatih seperti jadwal sebelumnya. Ensemble CNN global dibandingkan dengan baseline klasik menggunakan campuran tetap 80% klasik dan 20% CNN. Campuran hanya dipromosikan bila agreement dan macro-F1 grouped OOF tidak turun dan sedikitnya satu naik. Ini perbandingan internal untuk seleksi model, bukan estimasi untouched final accuracy; tidak ada label atau aturan per ID test.

In [ ]:
if PIPELINE_READY:
    if MODEL_NAMES:
        oof_probs=dict(classic_oof_probs); proxy_summary=list(classic_summaries)
        for name in MODEL_NAMES:
            oof=np.full((len(train_frame),len(CLASSES)),np.nan,dtype=np.float32)
            for fold,(tr,va) in enumerate(folds):
                model,history,checkpoint=fit_one(name,tr,va,FOLD_EPOCHS,f'fold{fold}')
                loader=DataLoader(LabeledDataset(va),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda')
                pp,yy,ids=predict_loader(model,loader); assert ids==[train_ids[j] for j in va]; oof[va]=pp
                del model; gc.collect(); torch.cuda.empty_cache()
            assert np.isfinite(oof).all() and np.allclose(oof.sum(1),1,atol=1e-4)
            oof_probs[name]=oof; pred=oof.argmax(1)
            row={'model':name,'proxy_agreement':float(accuracy_score(pseudo,pred)),'proxy_macro_f1':float(f1_score(pseudo,pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0))}; proxy_summary.append(row)
            pd.DataFrame(oof,columns=CLASSES).assign(sample_id=train_ids,fold=fold_ids,target=train_frame.assigned_class,predicted=[CLASSES[j] for j in pred]).to_csv(WORK_ROOT/f'validation/oof_{name}.csv',index=False)
            pd.DataFrame(confusion_matrix(pseudo,pred,labels=list(range(len(CLASSES)))),index=CLASSES,columns=CLASSES).to_csv(WORK_ROOT/f'validation/confusion_{name}.csv')
            pd.DataFrame(classification_report(pseudo,pred,labels=list(range(len(CLASSES))),target_names=CLASSES,output_dict=True,zero_division=0)).T.to_csv(WORK_ROOT/f'validation/report_{name}.csv')
            print(row)
        oof_ensemble=np.mean([oof_probs[name] for name in MODEL_NAMES],axis=0)
        proxy_summary.append({'model':'equal_probability_ensemble','proxy_agreement':float(accuracy_score(pseudo,oof_ensemble.argmax(1))),'proxy_macro_f1':float(f1_score(pseudo,oof_ensemble.argmax(1),labels=list(range(len(CLASSES))),average='macro',zero_division=0))})
        pd.DataFrame(proxy_summary).to_csv(WORK_ROOT/'validation/proxy_summary.csv',index=False); display(pd.DataFrame(proxy_summary))
        fixed_hybrid_oof=.8*classic_oof_probs[CLASSIC_SELECTED_NAME]+.2*oof_ensemble
        PROMOTE_NEURAL_BLEND=not_degraded(scores(pseudo,fixed_hybrid_oof),scores(pseudo,classic_oof_probs[CLASSIC_SELECTED_NAME]))
        hybrid_metric=scores(pseudo,fixed_hybrid_oof)
        proxy_summary.append({'model':'fixed_80classic_20neural','proxy_agreement':hybrid_metric['agreement'],'proxy_macro_f1':hybrid_metric['macro_f1']})
        pd.DataFrame(proxy_summary).to_csv(WORK_ROOT/'validation/proxy_summary.csv',index=False)
        print('Promote fixed global CNN blend:',PROMOTE_NEURAL_BLEND,'| baseline:',CLASSIC_SELECTED_NAME)
    else:
        PROMOTE_NEURAL_BLEND=False
        proxy_summary=list(classic_summaries)
        pd.DataFrame(proxy_summary).to_csv(WORK_ROOT/'validation/proxy_summary.csv',index=False)
        print('CPU mode: memakai model klasik train-only; fine-tuning CNN dilewati.')
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 11. Refit CNN, inferensi, dan final submission

Refit seluruh sampel full-frame eligible, satu backbone per giliran; bagian ini hanya tercapai setelah guard seed terlewati. Simpan CSV setiap model, ensemble neural, dan campuran klasik-neural. `DSC2026_Submission.csv` mengikuti kebijakan global yang ditetapkan dari train; model tambahan yang tidak lolos guard tidak menggantikan baseline klasik.

In [ ]:
if PIPELINE_READY:
    FINAL_EPOCHS=HEAD_EPOCHS+FINETUNE_EPOCHS
    model_probabilities={}; final_meta=[]
    test_loader=DataLoader(ImageTensor(test_paths),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda')
    def write_submission(probabilities,path):
        assert probabilities.shape==(len(test_ids),len(CLASSES)) and np.isfinite(probabilities).all()
        assert np.allclose(probabilities.sum(1),1,atol=1e-4)
        pred=pd.DataFrame({'image_id_norm':test_ids,'waste_category':[CLASSES[j] for j in probabilities.argmax(1)]})
        out=sample[['image_id','image_id_norm']].merge(pred,on='image_id_norm',how='left',validate='one_to_one',sort=False)
        assert out.image_id.tolist()==sample.image_id.tolist()
        assert len(out)==480 and out.waste_category.notna().all() and set(out.waste_category)<=set(CLASSES)
        out=out[['image_id','waste_category']]; out.to_csv(path,index=False)
        return out
    for name in MODEL_NAMES:
        model,history,checkpoint=fit_one(name,eligible_idx,None,FINAL_EPOCHS,'final')
        pp,_,_=predict_loader(model,test_loader)
        model_probabilities[name]=pp
        pd.DataFrame(pp,columns=CLASSES).assign(image_id=test_ids).to_csv(WORK_ROOT/f'predictions/test_probabilities_{name}.csv',index=False)
        write_submission(pp,WORK_ROOT/f'predictions/submission_{name}.csv')
        final_meta.append({'model':name,'checkpoint':str(checkpoint.relative_to(WORK_ROOT)),'checkpoint_sha256':file_sha256(checkpoint),'completed_epochs':len(history),'selection':'last epoch of fixed full-train refit; no validation set'})
        del model; gc.collect(); torch.cuda.empty_cache()
    if model_probabilities:
        cnn_ensemble=np.mean(list(model_probabilities.values()),axis=0)
        write_submission(cnn_ensemble,WORK_ROOT/'predictions/submission_neural_ensemble.csv')
        hybrid_ensemble=.8*classic_final_probabilities[CLASSIC_SELECTED_NAME]+.2*cnn_ensemble
        write_submission(hybrid_ensemble,WORK_ROOT/'predictions/submission_classic_neural_hybrid.csv')
        ensemble=hybrid_ensemble if PROMOTE_NEURAL_BLEND else classic_final_probabilities[CLASSIC_SELECTED_NAME]
        weights={'classic':.8,'cnn_mean':.2} if PROMOTE_NEURAL_BLEND else {'classic':1.,'cnn_mean':0.}
    else:
        ensemble=classic_final_probabilities[CLASSIC_SELECTED_NAME]
        weights={'classic':1.,'cnn_mean':0.}
    FINAL_ENSEMBLE_POLICY={'classic_family':CLASSIC_SELECTED_NAME,'cnn_blend_promoted':bool(PROMOTE_NEURAL_BLEND),'weights':weights,'device':DEVICE.type,'guard':'grouped train-only model selection; test used only for final inference'}
    submission=write_submission(ensemble,WORK_ROOT/'DSC2026_Submission.csv')
    pd.DataFrame(ensemble,columns=[f'prob_{c}' for c in CLASSES]).assign(image_id=test_ids).to_csv(WORK_ROOT/'predictions/test_probabilities_ensemble.csv',index=False)
    display(submission.head(10)); print('Final CSV:',WORK_ROOT/'DSC2026_Submission.csv')
    if DEVICE.type=='cpu':
        # Separate train-OOF patch candidate. The primary CSV keeps the selected global blend.
        _patch_pos={norm_id(k.split(':',1)[1]):i for i,k in enumerate(train_keys)}
        _patch_train=normalize(np.stack([dino2_patch_features[_patch_pos[sid]] for sid in train_ids]).astype(np.float32))
        _patch_oof=np.full((len(train_frame),len(CLASSES)),np.nan,dtype=np.float32)
        for _fold,(_tr,_va) in enumerate(folds):
            _patch_fold=_cpu_dino_svm(_patch_train[_tr],pseudo[_tr],SEED+7000+_fold)
            _patch_oof[_va]=softmax(_patch_fold.decision_function(_patch_train[_va]),axis=1).astype(np.float32)
        assert np.isfinite(_patch_oof).all()
        _base_oof=classic_oof_probs[CLASSIC_SELECTED_NAME]
        _patch_options=[]
        for _weight in (0.,.1,.2,.3,.4,.5,.6,.7,1.):
            _candidate=(1.-_weight)*_base_oof+_weight*_patch_oof
            _stats=_metric_cpu(_candidate)
            _patch_options.append((_stats['proxy_agreement'],_stats['proxy_macro_f1'],-_weight,_weight))
        _,_,_,_patch_weight=max(_patch_options)
        _patch_report={'blend_patch_weight':_patch_weight,'base':_metric_cpu(_base_oof),
                       'patch':_metric_cpu(_patch_oof),'combined':_metric_cpu((1.-_patch_weight)*_base_oof+_patch_weight*_patch_oof),
                       'selection':'grouped train OOF only','test_labels_used':False}
        (WORK_ROOT/'logs/patch_candidate_selection.json').write_text(json.dumps(_patch_report,indent=2),encoding='utf-8')
        if _patch_weight>0:
            _patch_source_paths=[Path(p) for p in test_frame.source_path.astype(str)]
            _,_patch_test,_=_extract_global_dino2(_patch_source_paths,np.asarray(['test:'+sid for sid in test_ids]),batch_size=8)
            _patch_test=normalize(_patch_test.astype(np.float32))
            _patch_final=_cpu_dino_svm(_patch_train,pseudo,SEED+8000)
            _patch_prob=softmax(_patch_final.decision_function(_patch_test),axis=1).astype(np.float32)
            _patch_candidate=(1.-_patch_weight)*classic_final_probabilities[CLASSIC_SELECTED_NAME]+_patch_weight*_patch_prob
            write_submission(_patch_candidate,WORK_ROOT/'predictions/submission_dino_patch_candidate.csv')
            pd.DataFrame(_patch_candidate,columns=CLASSES).assign(image_id=test_ids).to_csv(WORK_ROOT/'predictions/probabilities_dino_patch_candidate.csv',index=False)
            print('Train-OOF patch candidate:',WORK_ROOT/'predictions/submission_dino_patch_candidate.csv','| weight:',_patch_weight)
else:
    print("Dilewati: train-only target pool belum dibentuk.")


## 12. Ekspor log, kurva, hash, dan artefak

Manifest merekam label state, descriptor DINOv3, sumber bobot, seluruh evaluasi kandidat, recipe inner/outer, checkpoint, dan kebijakan submission. Semua keluaran disimpan di working dan paket ZIP.

In [ ]:
if PIPELINE_READY:
    for name in MODEL_NAMES:
        fig,axs=plt.subplots(1,2,figsize=(12,4))
        for fold in range(N_SPLITS):
            h=pd.read_csv(WORK_ROOT/f'logs/fold{fold}_{name}_history.csv')
            axs[0].plot(h.epoch,h.train_loss,label=f'fold {fold}')
            axs[1].plot(h.epoch,h.val_proxy_macro_f1,label=f'fold {fold}')
        axs[0].set_title(name+' train loss'); axs[1].set_title(name+' validation proxy macro-F1')
        for ax in axs: ax.set_xlabel('epoch'); ax.legend()
        plt.tight_layout(); plt.savefig(WORK_ROOT/f'figures/history_{name}.png',dpi=150); plt.show(); plt.close(fig)
    run_manifest={'run_utc':datetime.now(timezone.utc).isoformat(),'environment':ENVIRONMENT,'processing_report':proof,'training_config':TRAIN_CONFIG,'config_sha256':CONFIG_HASH,'proxy_validation':proxy_summary,'final_models':final_meta,'ensemble_policy':FINAL_ENSEMBLE_POLICY,'classical_selection':classic_report,'dino_feature_provenance':DINO_PROVENANCE,'submission_rows':len(submission),'submission_sha256':file_sha256(WORK_ROOT/'DSC2026_Submission.csv'),'submission_label_distribution':submission.waste_category.value_counts().to_dict(),'raw_test_used_for_fit':False,'assignment_fitting_repeated':False,'elapsed_training_seconds':round(time.time()-RUN_START,2)}
    pretrained={'resnet50':models.ResNet50_Weights.IMAGENET1K_V2,'efficientnet_v2_s':models.EfficientNet_V2_S_Weights.IMAGENET1K_V1,'convnext_small':models.ConvNeXt_Small_Weights.IMAGENET1K_V1,'swin_v2_t':models.Swin_V2_T_Weights.IMAGENET1K_V1}
    run_manifest['pretrained_weight_sources']={}
    for name,weight in pretrained.items():
        cached=Path(torch.hub.get_dir())/'checkpoints'/weight.url.rsplit('/',1)[-1]
        run_manifest['pretrained_weight_sources'][name]={'enum':str(weight),'url':weight.url,'sha256':file_sha256(cached) if cached.exists() else None}
    (WORK_ROOT/'logs/run_manifest.json').write_text(json.dumps(run_manifest,indent=2),encoding='utf-8')
    csv=pd.read_csv(WORK_ROOT/'DSC2026_Submission.csv',dtype={'image_id':str})
    assert list(csv.columns)==['image_id','waste_category'] and csv.image_id.tolist()==sample.image_id.tolist() and len(csv)==480
    zip_path=Path('/kaggle/working/GelarRasa_GPU_run_artifacts.zip')
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
        for p in WORK_ROOT.rglob('*'):
            if not p.is_file() or p.name.endswith('_resume.pt'): continue
            if 'checkpoints' in p.parts and not p.name.startswith('final_'): continue
            archive.write(p,p.relative_to(WORK_ROOT.parent))
    print('CSV:',WORK_ROOT/'DSC2026_Submission.csv')
    print('Proof:',WORK_ROOT/'processed/processing_report.json')
    print('ZIP:',zip_path)
else:
    print("Dilewati: train-only target pool belum dibentuk.")
